In [9]:
# ============================================================
# A2 — FRESH NOTEBOOK SETUP
# ============================================================

import os
import gc
import time
import math
import json
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from kaggle_secrets import UserSecretsClient
from tokenizers import Tokenizer
from datasets import load_dataset

# ------------------------------------------------------------
# Basic environment
# ------------------------------------------------------------

assert torch.cuda.is_available(), "❌ CUDA GPU not available"

device = torch.device("cuda")

# ------------------------------------------------------------
# Hugging Face token
# ------------------------------------------------------------

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

REPO = "Exploration-Lab/CS-781-Capstone"


TOKENIZER_PATH = "/kaggle/input/datasets/mathimohankowshik/tokenizer1/tokenizer.json"

WORK_DIR = "/kaggle/working/a2_production"
os.makedirs(WORK_DIR, exist_ok=True)

# ------------------------------------------------------------
# GPU information
# ------------------------------------------------------------

gpu_name = torch.cuda.get_device_name(0)
gpu_total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("=" * 60)
print("A2 — FRESH SESSION READY")
print("=" * 60)
print("GPU          :", gpu_name)
print("VRAM         :", f"{gpu_total:.2f} GB")
print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.version.cuda)
print("HF token     :", "✅ loaded")
print("Tokenizer    :", os.path.exists(TOKENIZER_PATH))
print("Working dir  :", WORK_DIR)
print("=" * 60)

assert os.path.exists(TOKENIZER_PATH), "❌ tokenizer.json not found"

print("✅ SETUP COMPLETE")

A2 — FRESH SESSION READY
GPU          : Tesla T4
VRAM         : 14.56 GB
PyTorch      : 2.11.0+cu128
CUDA         : 12.8
HF token     : ✅ loaded
Tokenizer    : True
Working dir  : /kaggle/working/a2_production
✅ SETUP COMPLETE


In [6]:
# ============================================================
# A2 — LOCATE EXISTING TOKENIZER
# ============================================================

import os

matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "tokenizer.json":
            matches.append(os.path.join(root, file))

print("=" * 60)
print("TOKENIZER SEARCH")
print("=" * 60)

if matches:
    for path in matches:
        print("✅ Found:", path)
else:
    print("❌ tokenizer.json not found under /kaggle/input")

print("=" * 60)

TOKENIZER SEARCH
✅ Found: /kaggle/input/datasets/mathimohankowshik/tokenizer1/tokenizer.json


In [10]:
# ============================================================
# A2 — TOKENIZER + MODEL CONFIG
# ============================================================

# Load the Phase-1 tokenizer
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

VOCAB_SIZE = tokenizer.get_vocab_size()

# Verified A2 architecture configuration
MAX_CONTEXT = 2048
HIDDEN_SIZE = 528
NUM_LAYERS = 8
NUM_Q_HEADS = 8
NUM_KV_HEADS = 2
FFN_SIZE = 2048

# Training configuration
TRAIN_BLOCK = 512
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2

TARGET_TOKENS = 200_000_000

print("=" * 60)
print("A2 — CONFIGURATION")
print("=" * 60)
print("Vocabulary       :", VOCAB_SIZE)
print("Context          :", MAX_CONTEXT)
print("Hidden size      :", HIDDEN_SIZE)
print("Layers           :", NUM_LAYERS)
print("Q heads          :", NUM_Q_HEADS)
print("KV heads         :", NUM_KV_HEADS)
print("FFN size         :", FFN_SIZE)
print("Train block      :", TRAIN_BLOCK)
print("Batch size       :", BATCH_SIZE)
print("Grad accumulation:", GRAD_ACCUM_STEPS)
print("Effective batch  :", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Target tokens    :", f"{TARGET_TOKENS:,}")
print("=" * 60)

assert VOCAB_SIZE == 50_000
assert HIDDEN_SIZE % NUM_Q_HEADS == 0
assert NUM_Q_HEADS % NUM_KV_HEADS == 0

print("✅ TOKENIZER + CONFIG READY")

A2 — CONFIGURATION
Vocabulary       : 50000
Context          : 2048
Hidden size      : 528
Layers           : 8
Q heads          : 8
KV heads         : 2
FFN size         : 2048
Train block      : 512
Batch size       : 8
Grad accumulation: 2
Effective batch  : 16
Target tokens    : 200,000,000
✅ TOKENIZER + CONFIG READY


In [11]:
# ============================================================
# A2 — VERIFIED GQA DECODER-ONLY MODEL
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class GQAAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads,
        dropout=0.0
    ):
        super().__init__()

        assert hidden_size % num_q_heads == 0
        assert num_q_heads % num_kv_heads == 0

        self.hidden_size = hidden_size
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = hidden_size // num_q_heads

        kv_dim = num_kv_heads * self.head_dim

        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=True)
        self.k_proj = nn.Linear(hidden_size, kv_dim, bias=True)
        self.v_proj = nn.Linear(hidden_size, kv_dim, bias=True)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=True)

        self.dropout = dropout

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # [B,T,C] -> [B,heads,T,head_dim]
        q = q.view(
            B, T,
            self.num_q_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        # Native PyTorch GQA
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
            enable_gqa=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(y)


class GQABlock(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads,
        ffn_size
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.attn = GQAAttention(
            hidden_size=hidden_size,
            num_q_heads=num_q_heads,
            num_kv_heads=num_kv_heads,
            dropout=0.0
        )

        self.ln2 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.fc1 = nn.Linear(
            hidden_size,
            ffn_size,
            bias=True
        )

        self.fc2 = nn.Linear(
            ffn_size,
            hidden_size,
            bias=True
        )

    def forward(self, x):

        # Pre-LN attention
        x = x + self.attn(self.ln1(x))

        # Pre-LN feed-forward
        y = self.ln2(x)
        y = self.fc1(y)
        y = F.gelu(y, approximate="tanh")
        y = self.fc2(y)

        x = x + y

        return x


class A2GQALM(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_context,
        hidden_size,
        num_layers,
        num_q_heads,
        num_kv_heads,
        ffn_size
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.max_context = max_context
        self.hidden_size = hidden_size

        # Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        # Learned positional embeddings
        self.position_embedding = nn.Embedding(
            max_context,
            hidden_size
        )

        self.blocks = nn.ModuleList([
            GQABlock(
                hidden_size=hidden_size,
                num_q_heads=num_q_heads,
                num_kv_heads=num_kv_heads,
                ffn_size=ffn_size
            )
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        # Tied output projection:
        # no separate lm_head parameters
        self.lm_bias = nn.Parameter(
            torch.zeros(vocab_size)
        )

        self.apply(self._init_weights)

        # Tie output weights to token embedding
        # logits = hidden @ embedding.T + bias

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, input_ids):

        B, T = input_ids.shape

        assert T <= self.max_context

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        for block in self.blocks:
            x = block(x)

        x = self.final_ln(x)

        # Tied embedding output
        logits = F.linear(
            x,
            self.token_embedding.weight,
            self.lm_bias
        )

        return logits


# ------------------------------------------------------------
# Build model
# ------------------------------------------------------------

model = A2GQALM(
    vocab_size=VOCAB_SIZE,
    max_context=MAX_CONTEXT,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_q_heads=NUM_Q_HEADS,
    num_kv_heads=NUM_KV_HEADS,
    ffn_size=FFN_SIZE
)

param_count = sum(
    p.numel()
    for p in model.parameters()
)

print("=" * 60)
print("A2 — MODEL CREATED")
print("=" * 60)
print(
    "Parameters:",
    f"{param_count:,}",
    f"({param_count/1e6:.2f}M)"
)
print("Q heads   :", NUM_Q_HEADS)
print("KV heads  :", NUM_KV_HEADS)
print("Head dim  :", HIDDEN_SIZE // NUM_Q_HEADS)
print("Tied embed:", True)
print("=" * 60)

assert param_count == 50_407_648

print("✅ EXACT A2 MODEL VERIFIED")

A2 — MODEL CREATED
Parameters: 50,457,648 (50.46M)
Q heads   : 8
KV heads  : 2
Head dim  : 66
Tied embed: True


AssertionError: 

In [12]:
# ============================================================
# A2 — PARAMETER BREAKDOWN
# ============================================================

print("=" * 60)
print("A2 CURRENT MODEL PARAMETER BREAKDOWN")
print("=" * 60)

total = 0

for name, param in model.named_parameters():
    n = param.numel()
    total += n
    print(f"{name:60s} {n:12,}")

print("=" * 60)
print(f"TOTAL PARAMETERS: {total:,}")
print(f"TOTAL MILLIONS  : {total/1e6:.4f}M")
print("=" * 60)

A2 CURRENT MODEL PARAMETER BREAKDOWN
lm_bias                                                            50,000
token_embedding.weight                                         26,400,000
position_embedding.weight                                       1,081,344
blocks.0.ln1.weight                                                   528
blocks.0.ln1.bias                                                     528
blocks.0.attn.q_proj.weight                                       278,784
blocks.0.attn.q_proj.bias                                             528
blocks.0.attn.k_proj.weight                                        69,696
blocks.0.attn.k_proj.bias                                             132
blocks.0.attn.v_proj.weight                                        69,696
blocks.0.attn.v_proj.bias                                             132
blocks.0.attn.out_proj.weight                                     278,784
blocks.0.attn.out_proj.bias                                           528
b

In [13]:
# ============================================================
# A2 — FINAL VERIFIED 50.41M GQA MODEL
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class GQAAttention(nn.Module):
    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads,
        dropout=0.0
    ):
        super().__init__()

        assert hidden_size % num_q_heads == 0
        assert num_q_heads % num_kv_heads == 0

        self.hidden_size = hidden_size
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = hidden_size // num_q_heads

        kv_dim = num_kv_heads * self.head_dim

        self.q_proj = nn.Linear(
            hidden_size,
            hidden_size,
            bias=True
        )

        self.k_proj = nn.Linear(
            hidden_size,
            kv_dim,
            bias=True
        )

        self.v_proj = nn.Linear(
            hidden_size,
            kv_dim,
            bias=True
        )

        self.out_proj = nn.Linear(
            hidden_size,
            hidden_size,
            bias=True
        )

        self.dropout = dropout

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # ----------------------------------------------------
        # Reshape into attention heads
        # ----------------------------------------------------

        q = q.view(
            B,
            T,
            self.num_q_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        # ----------------------------------------------------
        # Native Grouped Query Attention
        # ----------------------------------------------------

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=True,
            enable_gqa=True
        )

        # [B,H,T,D] -> [B,T,C]
        y = (
            y.transpose(1, 2)
             .contiguous()
             .view(B, T, C)
        )

        return self.out_proj(y)


class GQABlock(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads,
        ffn_size
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.attn = GQAAttention(
            hidden_size=hidden_size,
            num_q_heads=num_q_heads,
            num_kv_heads=num_kv_heads,
            dropout=0.0
        )

        self.ln2 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.fc1 = nn.Linear(
            hidden_size,
            ffn_size,
            bias=True
        )

        self.fc2 = nn.Linear(
            ffn_size,
            hidden_size,
            bias=True
        )

    def forward(self, x):

        # ----------------------------------------------------
        # Pre-LN attention
        # ----------------------------------------------------

        x = x + self.attn(
            self.ln1(x)
        )

        # ----------------------------------------------------
        # Feed-forward network
        # ----------------------------------------------------

        y = self.ln2(x)

        y = self.fc1(y)

        y = F.gelu(
            y,
            approximate="tanh"
        )

        y = self.fc2(y)

        x = x + y

        return x


class A2GQALM(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_context,
        hidden_size,
        num_layers,
        num_q_heads,
        num_kv_heads,
        ffn_size
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.max_context = max_context
        self.hidden_size = hidden_size

        # ----------------------------------------------------
        # Token embedding
        # ----------------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        # ----------------------------------------------------
        # Learned positional embedding
        # ----------------------------------------------------

        self.position_embedding = nn.Embedding(
            max_context,
            hidden_size
        )

        # ----------------------------------------------------
        # Transformer blocks
        # ----------------------------------------------------

        self.blocks = nn.ModuleList(
            [
                GQABlock(
                    hidden_size=hidden_size,
                    num_q_heads=num_q_heads,
                    num_kv_heads=num_kv_heads,
                    ffn_size=ffn_size
                )
                for _ in range(num_layers)
            ]
        )

        # ----------------------------------------------------
        # Final LayerNorm
        # ----------------------------------------------------

        self.final_ln = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        # ----------------------------------------------------
        # Initialization
        # ----------------------------------------------------

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

        elif isinstance(module, nn.LayerNorm):

            nn.init.ones_(
                module.weight
            )

            nn.init.zeros_(
                module.bias
            )

    def forward(self, input_ids):

        B, T = input_ids.shape

        assert T <= self.max_context

        # ----------------------------------------------------
        # Positions
        # ----------------------------------------------------

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        # ----------------------------------------------------
        # Input embeddings
        # ----------------------------------------------------

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        # ----------------------------------------------------
        # Transformer blocks
        # ----------------------------------------------------

        for block in self.blocks:
            x = block(x)

        # ----------------------------------------------------
        # Final normalization
        # ----------------------------------------------------

        x = self.final_ln(x)

        # ----------------------------------------------------
        # TIED OUTPUT HEAD
        #
        # No separate lm_bias.
        # Output weights = token_embedding.weight
        # ----------------------------------------------------

        logits = F.linear(
            x,
            self.token_embedding.weight,
            bias=None
        )

        return logits


# ============================================================
# BUILD A2
# ============================================================

model = A2GQALM(
    vocab_size=VOCAB_SIZE,
    max_context=MAX_CONTEXT,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_q_heads=NUM_Q_HEADS,
    num_kv_heads=NUM_KV_HEADS,
    ffn_size=FFN_SIZE
)


# ============================================================
# VERIFY PARAMETER COUNT
# ============================================================

param_count = sum(
    p.numel()
    for p in model.parameters()
)

print("=" * 60)
print("A2 — FINAL MODEL")
print("=" * 60)

print(
    "Parameters:",
    f"{param_count:,}",
    f"({param_count / 1e6:.2f}M)"
)

print("Vocabulary :", VOCAB_SIZE)
print("Context    :", MAX_CONTEXT)
print("Hidden     :", HIDDEN_SIZE)
print("Layers     :", NUM_LAYERS)
print("Q heads    :", NUM_Q_HEADS)
print("KV heads   :", NUM_KV_HEADS)
print("Head dim   :", HIDDEN_SIZE // NUM_Q_HEADS)
print("FFN        :", FFN_SIZE)
print("GQA        : Native PyTorch SDPA")
print("Embeddings : Tied")
print("=" * 60)

assert param_count == 50_407_648

print("✅ EXACT 50.41M A2 MODEL VERIFIED")

A2 — FINAL MODEL
Parameters: 50,407,648 (50.41M)
Vocabulary : 50000
Context    : 2048
Hidden     : 528
Layers     : 8
Q heads    : 8
KV heads   : 2
Head dim   : 66
FFN        : 2048
GQA        : Native PyTorch SDPA
Embeddings : Tied
✅ EXACT 50.41M A2 MODEL VERIFIED


In [14]:
# ============================================================
# A2 — GPU / OPTIMIZER SETUP
# ============================================================

import torch

# Performance settings
torch.set_float32_matmul_precision("high")

# ------------------------------------------------------------
# Move model to GPU in FP32
# ------------------------------------------------------------

model = model.to(device)
model.train()

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,
    fused=True
)

# ------------------------------------------------------------
# Mixed precision
# ------------------------------------------------------------

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)

# ------------------------------------------------------------
# Memory check
# ------------------------------------------------------------

torch.cuda.synchronize()

allocated = (
    torch.cuda.memory_allocated() / 1024**3
)

reserved = (
    torch.cuda.memory_reserved() / 1024**3
)

total = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print("=" * 60)
print("A2 — GPU SETUP COMPLETE")
print("=" * 60)
print("Device     :", device)
print("Model dtype:", next(model.parameters()).dtype)
print("Optimizer  : AdamW fused")
print("AMP        : FP16")
print("LR         : 3e-4")
print("Batch      :", BATCH_SIZE)
print("Grad accum :", GRAD_ACCUM_STEPS)
print("Effective  :", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("-" * 60)
print("VRAM total :", f"{total:.2f} GB")
print("Allocated  :", f"{allocated:.2f} GB")
print("Reserved   :", f"{reserved:.2f} GB")
print("Free approx:",
      f"{total-reserved:.2f} GB")
print("=" * 60)

assert next(model.parameters()).is_cuda

print("✅ MODEL + OPTIMIZER READY")

A2 — GPU SETUP COMPLETE
Device     : cuda
Model dtype: torch.float32
Optimizer  : AdamW fused
AMP        : FP16
LR         : 3e-4
Batch      : 8
Grad accum : 2
Effective  : 16
------------------------------------------------------------
VRAM total : 14.56 GB
Allocated  : 0.19 GB
Reserved   : 0.20 GB
Free approx: 14.36 GB
✅ MODEL + OPTIMIZER READY


In [12]:
# ============================================================
# A2 — OFFICIAL DATA + ONE-STEP GPU CHECK
# ============================================================

import gc
import time
import numpy as np

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]

TRAIN_BLOCK = 512

# ------------------------------------------------------------
# Open the four official Phase-1 streams
# ------------------------------------------------------------

streams = []

for path in TRAIN_FILES:
    ds = load_dataset(
        REPO,
        data_files=path,
        split="train",
        streaming=True,
        token=HF_TOKEN
    )
    streams.append(iter(ds))

print("=" * 60)
print("A2 — OFFICIAL DATA STREAMS")
print("=" * 60)
print("Sources opened:", len(streams))

# ------------------------------------------------------------
# Fast sequence generator
# ------------------------------------------------------------

def sequence_generator():

    carry = np.empty(0, dtype=np.int64)
    source_idx = 0

    while True:

        try:
            example = next(streams[source_idx])

        except StopIteration:

            ds = load_dataset(
                REPO,
                data_files=TRAIN_FILES[source_idx],
                split="train",
                streaming=True,
                token=HF_TOKEN
            )

            streams[source_idx] = iter(ds)

            example = next(streams[source_idx])

        source_idx = (
            source_idx + 1
        ) % len(streams)

        text = example.get("text")

        if not text:
            continue

        ids = tokenizer.encode(text).ids

        if not ids:
            continue

        ids = np.asarray(
            ids,
            dtype=np.int64
        )

        if carry.size == 0:
            carry = ids
        else:
            carry = np.concatenate(
                (carry, ids)
            )

        complete = (
            carry.size // TRAIN_BLOCK
        ) * TRAIN_BLOCK

        if complete == 0:
            continue

        usable = carry[:complete]

        carry = carry[complete:]

        for start in range(
            0,
            usable.size,
            TRAIN_BLOCK
        ):
            yield usable[
                start:start + TRAIN_BLOCK
            ]


seq_iter = sequence_generator()

# ------------------------------------------------------------
# Build one training batch
# ------------------------------------------------------------

print("Building one batch...")

t0 = time.perf_counter()

batch_np = np.stack([
    next(seq_iter)
    for _ in range(BATCH_SIZE)
])

batch = torch.from_numpy(
    batch_np
).long().to(
    device,
    non_blocking=True
)

# ------------------------------------------------------------
# One forward + backward pass
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):

    logits = model(batch)

    # Next-token prediction
    loss = F.cross_entropy(
        logits[:, :-1, :].reshape(
            -1,
            VOCAB_SIZE
        ),
        batch[:, 1:].reshape(-1)
    )

scaler.scale(loss).backward()

# ------------------------------------------------------------
# Check finite values
# ------------------------------------------------------------

loss_value = loss.detach().float().item()

finite_loss = math.isfinite(loss_value)

grad_finite = True

for p in model.parameters():

    if p.grad is not None:

        if not torch.isfinite(
            p.grad
        ).all():

            grad_finite = False
            break

# Do NOT update weights in this diagnostic step
optimizer.zero_grad(
    set_to_none=True
)

torch.cuda.synchronize()

elapsed = time.perf_counter() - t0

allocated = (
    torch.cuda.memory_allocated()
    / 1024**3
)

peak = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("=" * 60)
print("A2 — ONE-STEP RESULT")
print("=" * 60)
print("Batch shape :", tuple(batch.shape))
print("Loss        :", f"{loss_value:.4f}")
print("Loss finite :", finite_loss)
print("Grad finite :", grad_finite)
print("Step time   :", f"{elapsed:.2f} sec")
print("Current VRAM:", f"{allocated:.2f} GB")
print("Peak VRAM   :", f"{peak:.2f} GB")
print("=" * 60)

assert finite_loss
assert grad_finite

del logits, loss, batch, batch_np

gc.collect()
torch.cuda.empty_cache()

print("✅ ONE-STEP GPU CHECK PASSED")

A2 — OFFICIAL DATA STREAMS
Sources opened: 4
Building one batch...


RuntimeError: shape '[-1, 50000]' is invalid for input of size 539616

In [ ]:
# ============================================================
# A2 — FINAL MEMORY-SAFE PRODUCTION RUN
# ============================================================
#
# 50.41M parameter GQA decoder-only LM
# Official Phase-1 data
# 200M training tokens
#
# MEMORY PROTECTION:
#   Micro-batch          = 2
#   Gradient accumulation= 8
#   Effective batch      = 16
#   Sequence length      = 512
#   Activation checkpointing = ON
#   Chunked vocabulary CE    = ON
#   FP32 master weights + FP16 AMP
#   Fused AdamW
#   NO torch.compile
#
# CHECKPOINT:
#   Every 1000 optimizer steps
#   Keeps only latest 2 checkpoints
#   Automatically resumes after kernel restart
# ============================================================

import os
import gc
import math
import json
import time
import glob
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.checkpoint import checkpoint
from kaggle_secrets import UserSecretsClient
from tokenizers import Tokenizer
from datasets import load_dataset


# ============================================================
# 1. ENVIRONMENT
# ============================================================

assert torch.cuda.is_available(), "CUDA GPU not available"

device = torch.device("cuda")

torch.set_float32_matmul_precision("high")

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

REPO = "Exploration-Lab/CS-781-Capstone"

TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mathimohankowshik/tokenizer1/tokenizer.json"
)

WORK_DIR = "/kaggle/working/a2_production"
CHECKPOINT_DIR = os.path.join(
    WORK_DIR,
    "checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

assert os.path.exists(TOKENIZER_PATH)


# ============================================================
# 2. CONFIGURATION
# ============================================================

tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

VOCAB_SIZE = tokenizer.get_vocab_size()

MAX_CONTEXT = 2048

HIDDEN_SIZE = 528
NUM_LAYERS = 8
NUM_Q_HEADS = 8
NUM_KV_HEADS = 2
FFN_SIZE = 2048

TRAIN_BLOCK = 512

# MEMORY-SAFE MICRO-BATCH
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8

EFFECTIVE_BATCH = (
    BATCH_SIZE * GRAD_ACCUM_STEPS
)

TARGET_TOKENS = 200_000_000

# Vocabulary-loss chunks.
# Prevents creation of:
# [batch, sequence, 50000] all at once.
LM_CHUNK_SIZE = 512

# Training
LEARNING_RATE = 1.0e-4
MIN_LR = 1.0e-5
WARMUP_STEPS = 1000

MAX_GRAD_NORM = 1.0

# Checkpointing
CHECKPOINT_EVERY = 1000
KEEP_CHECKPOINTS = 2

USE_ACTIVATION_CHECKPOINTING = True

print("=" * 70)
print("A2 — FINAL MEMORY-SAFE CONFIGURATION")
print("=" * 70)

print("GPU                 :", torch.cuda.get_device_name(0))
print("VRAM                :",
      f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")

print("Vocabulary          :", VOCAB_SIZE)
print("Context             :", MAX_CONTEXT)
print("Hidden              :", HIDDEN_SIZE)
print("Layers              :", NUM_LAYERS)
print("Q heads             :", NUM_Q_HEADS)
print("KV heads            :", NUM_KV_HEADS)
print("FFN                 :", FFN_SIZE)

print("-" * 70)

print("Sequence length     :", TRAIN_BLOCK)
print("Micro-batch         :", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)
print("Effective batch     :", EFFECTIVE_BATCH)

print("Target tokens       :", f"{TARGET_TOKENS:,}")
print("Learning rate       :", LEARNING_RATE)
print("Warmup steps        :", WARMUP_STEPS)
print("Activation checkpoint:", USE_ACTIVATION_CHECKPOINTING)
print("Chunked CE size     :", LM_CHUNK_SIZE)

print("=" * 70)

assert VOCAB_SIZE == 50_000
assert HIDDEN_SIZE % NUM_Q_HEADS == 0
assert NUM_Q_HEADS % NUM_KV_HEADS == 0


# ============================================================
# 3. MODEL
# ============================================================

class GQAAttention(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads,
        dropout=0.0
    ):
        super().__init__()

        assert hidden_size % num_q_heads == 0
        assert num_q_heads % num_kv_heads == 0

        self.hidden_size = hidden_size
        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads

        self.head_dim = (
            hidden_size // num_q_heads
        )

        kv_dim = (
            num_kv_heads * self.head_dim
        )

        self.q_proj = nn.Linear(
            hidden_size,
            hidden_size,
            bias=True
        )

        self.k_proj = nn.Linear(
            hidden_size,
            kv_dim,
            bias=True
        )

        self.v_proj = nn.Linear(
            hidden_size,
            kv_dim,
            bias=True
        )

        self.out_proj = nn.Linear(
            hidden_size,
            hidden_size,
            bias=True
        )

        self.dropout = dropout

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.num_q_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=True,
            enable_gqa=True
        )

        y = (
            y.transpose(1, 2)
             .contiguous()
             .view(B, T, C)
        )

        return self.out_proj(y)


class GQABlock(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads,
        ffn_size
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.attn = GQAAttention(
            hidden_size,
            num_q_heads,
            num_kv_heads
        )

        self.ln2 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.fc1 = nn.Linear(
            hidden_size,
            ffn_size,
            bias=True
        )

        self.fc2 = nn.Linear(
            ffn_size,
            hidden_size,
            bias=True
        )

    def forward(self, x):

        x = x + self.attn(
            self.ln1(x)
        )

        y = self.ln2(x)

        y = self.fc1(y)

        y = F.gelu(
            y,
            approximate="tanh"
        )

        y = self.fc2(y)

        x = x + y

        return x


class A2GQALM(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_context,
        hidden_size,
        num_layers,
        num_q_heads,
        num_kv_heads,
        ffn_size
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.max_context = max_context
        self.hidden_size = hidden_size

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            max_context,
            hidden_size
        )

        self.blocks = nn.ModuleList(
            [
                GQABlock(
                    hidden_size,
                    num_q_heads,
                    num_kv_heads,
                    ffn_size
                )
                for _ in range(num_layers)
            ]
        )

        self.final_ln = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

        elif isinstance(module, nn.LayerNorm):

            nn.init.ones_(
                module.weight
            )

            nn.init.zeros_(
                module.bias
            )

    def forward(
        self,
        input_ids
    ):

        B, T = input_ids.shape

        assert T <= self.max_context

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        for block in self.blocks:

            if (
                self.training
                and USE_ACTIVATION_CHECKPOINTING
            ):
                x = checkpoint(
                    block,
                    x,
                    use_reentrant=False
                )
            else:
                x = block(x)

        x = self.final_ln(x)

        return x


# ============================================================
# 4. CREATE MODEL
# ============================================================

model = A2GQALM(
    vocab_size=VOCAB_SIZE,
    max_context=MAX_CONTEXT,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_q_heads=NUM_Q_HEADS,
    num_kv_heads=NUM_KV_HEADS,
    ffn_size=FFN_SIZE
)

param_count = sum(
    p.numel()
    for p in model.parameters()
)

print("=" * 70)
print("A2 MODEL")
print("=" * 70)
print(
    "Parameters:",
    f"{param_count:,}",
    f"({param_count/1e6:.2f}M)"
)
print("=" * 70)

assert param_count == 50_407_648

print("✅ EXACT 50.41M A2 MODEL")


# ============================================================
# 5. GPU + OPTIMIZER
# ============================================================

model = model.to(device)

model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,
    fused=True
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True
)


# ============================================================
# 6. OFFICIAL TRAINING STREAMS
# ============================================================

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]

streams = []

print("=" * 70)
print("OPENING OFFICIAL PHASE-1 DATA")
print("=" * 70)

for path in TRAIN_FILES:

    ds = load_dataset(
        REPO,
        data_files=path,
        split="train",
        streaming=True,
        token=HF_TOKEN
    )

    streams.append(iter(ds))

    print("✅", path)

print("=" * 70)


# ============================================================
# 7. MEMORY-EFFICIENT SEQUENCE GENERATOR
# ============================================================

def sequence_generator():

    carry = []
    position = 0
    source_idx = 0

    while True:

        try:
            example = next(
                streams[source_idx]
            )

        except StopIteration:

            ds = load_dataset(
                REPO,
                data_files=TRAIN_FILES[source_idx],
                split="train",
                streaming=True,
                token=HF_TOKEN
            )

            streams[source_idx] = iter(ds)

            example = next(
                streams[source_idx]
            )

        source_idx = (
            source_idx + 1
        ) % len(streams)

        text = example.get("text")

        if not text:
            continue

        ids = tokenizer.encode(text).ids

        if not ids:
            continue

        carry.extend(ids)

        while (
            len(carry) - position
            >= TRAIN_BLOCK
        ):

            block = np.asarray(
                carry[
                    position:
                    position + TRAIN_BLOCK
                ],
                dtype=np.int64
            )

            position += TRAIN_BLOCK

            yield block

        # Periodically compact the Python list
        # so old tokens are released.
        if position >= 65536:

            carry = carry[position:]
            position = 0


seq_iter = sequence_generator()


# ============================================================
# 8. CHUNKED LANGUAGE-MODEL LOSS
# ============================================================
#
# IMPORTANT:
#
# We NEVER create:
#
#   [batch, sequence, 50000]
#
# for the entire sequence.
#
# Instead only 512 tokens × 50000 vocabulary
# are materialized at once.
# ============================================================

def chunked_lm_loss(
    hidden,
    targets,
    chunk_size=LM_CHUNK_SIZE
):

    hidden = hidden[:, :-1, :]
    targets = targets[:, 1:]

    flat_hidden = hidden.reshape(
        -1,
        hidden.size(-1)
    )

    flat_targets = targets.reshape(
        -1
    )

    total_tokens = flat_targets.numel()

    total_loss = None

    for start in range(
        0,
        total_tokens,
        chunk_size
    ):

        end = min(
            start + chunk_size,
            total_tokens
        )

        logits = F.linear(
            flat_hidden[start:end],
            model.token_embedding.weight
        )

        chunk_loss = F.cross_entropy(
            logits,
            flat_targets[start:end],
            reduction="sum"
        )

        if total_loss is None:
            total_loss = chunk_loss
        else:
            total_loss = (
                total_loss + chunk_loss
            )

        del logits

    return total_loss / total_tokens


# ============================================================
# 9. LEARNING-RATE SCHEDULE
# ============================================================

TOTAL_OPTIMIZER_STEPS = math.ceil(
    TARGET_TOKENS
    / (
        BATCH_SIZE
        * TRAIN_BLOCK
        * GRAD_ACCUM_STEPS
    )
)


def get_lr(step):

    if step < WARMUP_STEPS:

        return (
            LEARNING_RATE
            * (step + 1)
            / WARMUP_STEPS
        )

    progress = (
        step - WARMUP_STEPS
    ) / max(
        1,
        TOTAL_OPTIMIZER_STEPS
        - WARMUP_STEPS
    )

    progress = min(
        max(progress, 0.0),
        1.0
    )

    cosine = (
        0.5
        * (1.0 + math.cos(
            math.pi * progress
        ))
    )

    return (
        MIN_LR
        + (
            LEARNING_RATE
            - MIN_LR
        ) * cosine
    )


def set_lr(lr):

    for group in optimizer.param_groups:
        group["lr"] = lr


# ============================================================
# 10. CHECKPOINT FUNCTIONS
# ============================================================

def checkpoint_files():

    files = glob.glob(
        os.path.join(
            CHECKPOINT_DIR,
            "step_*.pt"
        )
    )

    return sorted(files)


def save_checkpoint(
    step,
    tokens_seen,
    final=False
):

    if final:

        path = os.path.join(
            CHECKPOINT_DIR,
            "final.pt"
        )

    else:

        path = os.path.join(
            CHECKPOINT_DIR,
            f"step_{step:08d}.pt"
        )

    temp_path = path + ".tmp"

    checkpoint_data = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "step": step,
        "tokens_seen": tokens_seen,
    }

    torch.save(
        checkpoint_data,
        temp_path
    )

    os.replace(
        temp_path,
        path
    )

    # Keep only latest 2 normal checkpoints
    if not final:

        files = checkpoint_files()

        while len(files) > KEEP_CHECKPOINTS:

            old = files.pop(0)

            try:
                os.remove(old)
            except FileNotFoundError:
                pass

    print(
        f"\n💾 CHECKPOINT SAVED | "
        f"step {step:,} | "
        f"tokens {tokens_seen:,}"
    )

    return path


def load_latest_checkpoint():

    files = checkpoint_files()

    if not files:

        return 0, 0

    path = files[-1]

    print("=" * 70)
    print("RESUMING FROM CHECKPOINT")
    print("=" * 70)
    print("File:", path)

    data = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        data["model"]
    )

    optimizer.load_state_dict(
        data["optimizer"]
    )

    scaler.load_state_dict(
        data["scaler"]
    )

    step = int(
        data["step"]
    )

    tokens = int(
        data["tokens_seen"]
    )

    del data

    gc.collect()
    torch.cuda.empty_cache()

    print("Step   :", step)
    print("Tokens :", f"{tokens:,}")
    print("=" * 70)

    return step, tokens


# ============================================================
# 11. RESUME IF POSSIBLE
# ============================================================

global_step, tokens_seen = (
    load_latest_checkpoint()
)

if tokens_seen >= TARGET_TOKENS:

    print("=" * 70)
    print("✅ TRAINING ALREADY COMPLETE")
    print("=" * 70)

else:

    print("=" * 70)
    print("STARTING A2 PRODUCTION TRAINING")
    print("=" * 70)
    print(
        "Remaining tokens:",
        f"{TARGET_TOKENS - tokens_seen:,}"
    )
    print(
        "Total optimizer steps:",
        TOTAL_OPTIMIZER_STEPS
    )
    print("=" * 70)


# ============================================================
# 12. PRODUCTION TRAINING
# ============================================================

optimizer.zero_grad(
    set_to_none=True
)

run_start = time.perf_counter()
last_log_time = run_start
last_log_tokens = tokens_seen

running_loss = 0.0
running_microsteps = 0

while tokens_seen < TARGET_TOKENS:

    step_start = time.perf_counter()

    optimizer.zero_grad(
        set_to_none=True
    )

    step_loss = 0.0

    # --------------------------------------------------------
    # Gradient accumulation
    # --------------------------------------------------------

    for microstep in range(
        GRAD_ACCUM_STEPS
    ):

        batch_np = np.stack([
            next(seq_iter)
            for _ in range(BATCH_SIZE)
        ])

        batch = torch.from_numpy(
            batch_np
        ).long().to(
            device,
            non_blocking=True
        )

        del batch_np

        # ----------------------------------------------------
        # Forward + chunked CE
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            hidden = model(batch)

            loss = chunked_lm_loss(
                hidden,
                batch
            )

            scaled_loss = (
                loss
                / GRAD_ACCUM_STEPS
            )

        loss_value = (
            loss.detach()
            .float()
            .item()
        )

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        scaler.scale(
            scaled_loss
        ).backward()

        step_loss += loss_value

        del (
            hidden,
            loss,
            scaled_loss,
            batch
        )

    # --------------------------------------------------------
    # Gradient clipping + optimizer update
    # --------------------------------------------------------

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM,
        error_if_nonfinite=False
    )

    scaler.step(optimizer)

    scaler.update()

    global_step += 1

    # One optimizer step processes:
    # 2 × 512 × 8 = 8192 input tokens
    tokens_seen += (
        BATCH_SIZE
        * TRAIN_BLOCK
        * GRAD_ACCUM_STEPS
    )

    # --------------------------------------------------------
    # Learning rate
    # --------------------------------------------------------

    lr = get_lr(
        global_step
    )

    set_lr(lr)

    # --------------------------------------------------------
    # Running loss
    # --------------------------------------------------------

    average_step_loss = (
        step_loss
        / GRAD_ACCUM_STEPS
    )

    running_loss += average_step_loss
    running_microsteps += 1

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    if (
        global_step % 100 == 0
        or tokens_seen >= TARGET_TOKENS
    ):

        torch.cuda.synchronize()

        now = time.perf_counter()

        elapsed = (
            now - last_log_time
        )

        token_delta = (
            tokens_seen
            - last_log_tokens
        )

        speed = (
            token_delta
            / max(elapsed, 1e-9)
        )

        overall_progress = (
            min(
                tokens_seen
                / TARGET_TOKENS,
                1.0
            )
            * 100
        )

        allocated = (
            torch.cuda.memory_allocated()
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved()
            / 1024**3
        )

        peak = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        avg_loss = (
            running_loss
            / max(
                running_microsteps,
                1
            )
        )

        print(
            f"Step {global_step:6d}/"
            f"{TOTAL_OPTIMIZER_STEPS} | "
            f"Tokens {tokens_seen:12,d}/"
            f"{TARGET_TOKENS:,} | "
            f"{overall_progress:6.2f}% | "
            f"Loss {avg_loss:.4f} | "
            f"Speed {speed:,.0f} tok/s | "
            f"LR {lr:.2e} | "
            f"VRAM {allocated:.2f}/"
            f"{reserved:.2f} GB | "
            f"Peak {peak:.2f} GB"
        )

        last_log_time = now
        last_log_tokens = tokens_seen

        running_loss = 0.0
        running_microsteps = 0

        torch.cuda.reset_peak_memory_stats()

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    if (
        global_step % CHECKPOINT_EVERY == 0
    ):

        save_checkpoint(
            global_step,
            tokens_seen
        )

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# 13. FINAL SAVE
# ============================================================

torch.cuda.synchronize()

save_checkpoint(
    global_step,
    tokens_seen,
    final=True
)

# Also save a simple model-only file
final_model_path = os.path.join(
    WORK_DIR,
    "a2_final_model.pt"
)

torch.save(
    model.state_dict(),
    final_model_path
)


# ============================================================
# 14. FINAL VERIFICATION
# ============================================================

bad_tensors = 0

for name, tensor in model.state_dict().items():

    if not torch.isfinite(tensor).all():

        print(
            "❌ NON-FINITE:",
            name
        )

        bad_tensors += 1


total_time = (
    time.perf_counter()
    - run_start
)

print("\n")
print("=" * 70)
print("🚀 A2 TRAINING COMPLETE")
print("=" * 70)
print(
    "Final optimizer step:",
    global_step
)
print(
    "Final tokens:",
    f"{tokens_seen:,}"
)
print(
    "Progress:",
    f"{min(tokens_seen/TARGET_TOKENS,1.0)*100:.2f}%"
)
print(
    "Training time:",
    f"{total_time/3600:.2f} hours"
)
print(
    "Final model:",
    final_model_path
)
print(
    "Non-finite tensors:",
    bad_tensors
)

final_allocated = (
    torch.cuda.memory_allocated()
    / 1024**3
)

final_reserved = (
    torch.cuda.memory_reserved()
    / 1024**3
)

print(
    "Final VRAM allocated:",
    f"{final_allocated:.2f} GB"
)

print(
    "Final VRAM reserved:",
    f"{final_reserved:.2f} GB"
)

print("=" * 70)

assert bad_tensors == 0

print("✅ A2 PRODUCTION RUN FINISHED SUCCESSFULLY")

A2 — FINAL MEMORY-SAFE CONFIGURATION
GPU                 : Tesla T4
VRAM                : 14.56 GB
Vocabulary          : 50000
Context             : 2048
Hidden              : 528
Layers              : 8
Q heads             : 8
KV heads            : 2
FFN                 : 2048
----------------------------------------------------------------------
Sequence length     : 512
Micro-batch         : 2
Gradient accumulation: 8
Effective batch     : 16
Target tokens       : 200,000,000
Learning rate       : 0.0001
Warmup steps        : 1000
Activation checkpoint: True
Chunked CE size     : 512
A2 MODEL
Parameters: 50,407,648 (50.41M)
✅ EXACT 50.41M A2 MODEL
OPENING OFFICIAL PHASE-1 DATA
✅ Phase_1_Pretraining/Train/train_acts.json
✅ Phase_1_Pretraining/Train/train_high_courts.json
✅ Phase_1_Pretraining/Train/train_supreme_court.json
✅ Phase_1_Pretraining/Train/train_tribunals.json
STARTING A2 PRODUCTION TRAINING
Remaining tokens: 200,000,000
Total optimizer steps: 24415


In [33]:
# ============================================================
# A2 — FINAL FAST + MEMORY-SAFE TRAINING
# ============================================================
#
# Architecture:
#   50.41M parameter GQA decoder-only Transformer
#
# Training:
#   sequence       = 512
#   micro batch    = 2
#   accumulation   = 8
#   effective batch= 16
#
# Memory:
#   FP32 master weights
#   FP16 autocast
#   normal CE
#   NO torch.compile
#   NO activation checkpointing
#
# Checkpoint:
#   every 500 optimizer steps
#   automatically resumes
# ============================================================

import os
import gc
import math
import time
import glob
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from kaggle_secrets import UserSecretsClient
from tokenizers import Tokenizer
from datasets import load_dataset


# ============================================================
# 1. ENVIRONMENT
# ============================================================

assert torch.cuda.is_available(), "CUDA GPU not available"

device = torch.device("cuda")

torch.set_float32_matmul_precision("high")

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

REPO = "Exploration-Lab/CS-781-Capstone"

TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mathimohankowshik/tokenizer1/tokenizer.json"
)

WORK_DIR = "/kaggle/working/a2_final"

CKPT_DIR = os.path.join(
    WORK_DIR,
    "checkpoints"
)

os.makedirs(
    CKPT_DIR,
    exist_ok=True
)

assert os.path.exists(
    TOKENIZER_PATH
)

tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

VOCAB_SIZE = tokenizer.get_vocab_size()

MAX_CONTEXT = 2048

HIDDEN_SIZE = 528
NUM_LAYERS = 8

NUM_Q_HEADS = 8
NUM_KV_HEADS = 2

FFN_SIZE = 2048

TRAIN_BLOCK = 512

# ------------------------------------------------------------
# MEMORY-SAFE TRAINING
# ------------------------------------------------------------

BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8

EFFECTIVE_BATCH = (
    BATCH_SIZE * GRAD_ACCUM_STEPS
)

TARGET_TOKENS = 200_000_000

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

LEARNING_RATE = 1e-4
MIN_LR = 1e-5

WARMUP_STEPS = 1000

MAX_GRAD_NORM = 1.0

# ------------------------------------------------------------
# Checkpointing
# ------------------------------------------------------------

CHECKPOINT_EVERY = 500
KEEP_CHECKPOINTS = 2

print("=" * 70)
print("A2 — FINAL TRAINING CONFIG")
print("=" * 70)

print("GPU                  :", torch.cuda.get_device_name(0))
print(
    "VRAM                 :",
    f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB"
)

print("Vocabulary           :", VOCAB_SIZE)
print("Hidden               :", HIDDEN_SIZE)
print("Layers               :", NUM_LAYERS)
print("Q heads              :", NUM_Q_HEADS)
print("KV heads             :", NUM_KV_HEADS)
print("FFN                  :", FFN_SIZE)

print("-" * 70)

print("Sequence             :", TRAIN_BLOCK)
print("Micro batch          :", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)
print("Effective batch      :", EFFECTIVE_BATCH)

print("Target tokens        :", f"{TARGET_TOKENS:,}")
print("Learning rate        :", LEARNING_RATE)

print("=" * 70)


# ============================================================
# 3. GQA ATTENTION
# ============================================================

class GQAAttention(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads
    ):
        super().__init__()

        assert hidden_size % num_q_heads == 0
        assert num_q_heads % num_kv_heads == 0

        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads

        self.head_dim = (
            hidden_size // num_q_heads
        )

        kv_dim = (
            num_kv_heads * self.head_dim
        )

        self.q_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.k_proj = nn.Linear(
            hidden_size,
            kv_dim
        )

        self.v_proj = nn.Linear(
            hidden_size,
            kv_dim
        )

        self.out_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.num_q_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
            enable_gqa=True
        )

        y = (
            y.transpose(1, 2)
             .contiguous()
             .view(B, T, C)
        )

        return self.out_proj(y)


# ============================================================
# 4. TRANSFORMER BLOCK
# ============================================================

class GQABlock(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads,
        ffn_size
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.attn = GQAAttention(
            hidden_size,
            num_q_heads,
            num_kv_heads
        )

        self.ln2 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.fc1 = nn.Linear(
            hidden_size,
            ffn_size
        )

        self.fc2 = nn.Linear(
            ffn_size,
            hidden_size
        )

    def forward(self, x):

        x = x + self.attn(
            self.ln1(x)
        )

        y = self.ln2(x)

        y = self.fc1(y)

        y = F.gelu(
            y,
            approximate="tanh"
        )

        y = self.fc2(y)

        return x + y


# ============================================================
# 5. A2 MODEL
# ============================================================

class A2GQALM(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_context,
        hidden_size,
        num_layers,
        num_q_heads,
        num_kv_heads,
        ffn_size
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden_size
        )

        self.position_embedding = nn.Embedding(
            max_context,
            hidden_size
        )

        self.blocks = nn.ModuleList(
            [
                GQABlock(
                    hidden_size,
                    num_q_heads,
                    num_kv_heads,
                    ffn_size
                )
                for _ in range(num_layers)
            ]
        )

        self.final_ln = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(
            module,
            nn.Linear
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(
            module,
            nn.Embedding
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

        elif isinstance(
            module,
            nn.LayerNorm
        ):

            nn.init.ones_(
                module.weight
            )

            nn.init.zeros_(
                module.bias
            )

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(
                positions
            )
        )

        for block in self.blocks:

            x = block(x)

        x = self.final_ln(x)

        return x


# ============================================================
# 6. CREATE MODEL
# ============================================================

model = A2GQALM(
    vocab_size=VOCAB_SIZE,
    max_context=MAX_CONTEXT,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_q_heads=NUM_Q_HEADS,
    num_kv_heads=NUM_KV_HEADS,
    ffn_size=FFN_SIZE
)

param_count = sum(
    p.numel()
    for p in model.parameters()
)

print("\n" + "=" * 70)
print("MODEL VERIFICATION")
print("=" * 70)

print(
    "Parameters:",
    f"{param_count:,}",
    f"({param_count/1e6:.2f}M)"
)

assert param_count == 50_407_648

print(
    "✅ EXACT 50.41M A2 MODEL"
)


# ============================================================
# 7. GPU + OPTIMIZER
# ============================================================

model = model.to(device)

model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,
    fused=True
)

scaler = torch.amp.GradScaler(
    "cuda"
)


# ============================================================
# 8. OFFICIAL DATA
# ============================================================

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]

streams = []

print("\n" + "=" * 70)
print("OPENING OFFICIAL DATA")
print("=" * 70)

for path in TRAIN_FILES:

    ds = load_dataset(
        REPO,
        data_files=path,
        split="train",
        streaming=True,
        token=HF_TOKEN
    )

    streams.append(
        iter(ds)
    )

    print("✅", path)

print("=" * 70)


# ============================================================
# 9. LOW-MEMORY DATA GENERATOR
# ============================================================

def sequence_generator():

    carry = []
    source = 0

    while True:

        try:

            example = next(
                streams[source]
            )

        except StopIteration:

            ds = load_dataset(
                REPO,
                data_files=TRAIN_FILES[source],
                split="train",
                streaming=True,
                token=HF_TOKEN
            )

            streams[source] = iter(ds)

            example = next(
                streams[source]
            )

        source = (
            source + 1
        ) % len(streams)

        text = example.get(
            "text"
        )

        if not text:
            continue

        ids = tokenizer.encode(
            text
        ).ids

        if not ids:
            continue

        carry.extend(ids)

        while len(carry) >= TRAIN_BLOCK:

            block = np.asarray(
                carry[:TRAIN_BLOCK],
                dtype=np.int64
            )

            del carry[
                :TRAIN_BLOCK
            ]

            yield block


seq_iter = sequence_generator()


# ============================================================
# 10. CHECKPOINT UTILITIES
# ============================================================

def get_checkpoints():

    files = glob.glob(
        os.path.join(
            CKPT_DIR,
            "step_*.pt"
        )
    )

    return sorted(files)


def save_checkpoint(
    step,
    tokens
):

    path = os.path.join(
        CKPT_DIR,
        f"step_{step:08d}.pt"
    )

    temp = path + ".tmp"

    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "step": step,
            "tokens": tokens,
        },
        temp
    )

    os.replace(
        temp,
        path
    )

    files = get_checkpoints()

    while len(files) > KEEP_CHECKPOINTS:

        old = files.pop(0)

        try:
            os.remove(old)
        except:
            pass

    print(
        f"\n💾 CHECKPOINT "
        f"step={step:,} "
        f"tokens={tokens:,}"
    )


def load_checkpoint():

    files = get_checkpoints()

    if not files:

        return 0, 0

    path = files[-1]

    print("\n" + "=" * 70)
    print("RESUMING CHECKPOINT")
    print("=" * 70)
    print(path)

    data = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        data["model"]
    )

    optimizer.load_state_dict(
        data["optimizer"]
    )

    scaler.load_state_dict(
        data["scaler"]
    )

    step = int(
        data["step"]
    )

    tokens = int(
        data["tokens"]
    )

    del data

    gc.collect()
    torch.cuda.empty_cache()

    print(
        "Resumed step :",
        step
    )

    print(
        "Resumed tokens:",
        f"{tokens:,}"
    )

    return step, tokens


# ============================================================
# 11. LEARNING RATE
# ============================================================

def learning_rate(step):

    if step < WARMUP_STEPS:

        return (
            LEARNING_RATE
            * (step + 1)
            / WARMUP_STEPS
        )

    progress = (
        step - WARMUP_STEPS
    ) / max(
        1,
        math.ceil(
            TARGET_TOKENS
            / (
                BATCH_SIZE
                * TRAIN_BLOCK
                * GRAD_ACCUM_STEPS
            )
        )
        - WARMUP_STEPS
    )

    progress = min(
        max(progress, 0.0),
        1.0
    )

    cosine = (
        0.5
        * (
            1
            + math.cos(
                math.pi * progress
            )
        )
    )

    return (
        MIN_LR
        + (
            LEARNING_RATE
            - MIN_LR
        ) * cosine
    )


# ============================================================
# 12. RESUME
# ============================================================

global_step, tokens_seen = (
    load_checkpoint()
)

TOKENS_PER_STEP = (
    BATCH_SIZE
    * TRAIN_BLOCK
    * GRAD_ACCUM_STEPS
)

TOTAL_STEPS = math.ceil(
    TARGET_TOKENS
    / TOKENS_PER_STEP
)

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

print(
    "Tokens/optimizer step:",
    f"{TOKENS_PER_STEP:,}"
)

print(
    "Total optimizer steps:",
    TOTAL_STEPS
)

print(
    "Already completed:",
    f"{tokens_seen:,}"
)

print(
    "Remaining:",
    f"{max(0, TARGET_TOKENS-tokens_seen):,}"
)

print("=" * 70)


# ============================================================
# 13. TRAIN
# ============================================================

train_start = time.perf_counter()

last_log_time = train_start
last_log_tokens = tokens_seen

loss_sum = 0.0

optimizer.zero_grad(
    set_to_none=True
)

while tokens_seen < TARGET_TOKENS:

    step_loss = 0.0

    # --------------------------------------------------------
    # Accumulation
    # --------------------------------------------------------

    for micro in range(
        GRAD_ACCUM_STEPS
    ):

        batch_np = np.stack(
            [
                next(seq_iter)
                for _ in range(BATCH_SIZE)
            ]
        )

        batch = torch.from_numpy(
            batch_np
        ).long().to(
            device,
            non_blocking=True
        )

        del batch_np

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            hidden = model(
                batch
            )

            # Tied output projection
            logits = F.linear(
                hidden[:, :-1, :],
                model.token_embedding.weight
            )

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                batch[:, 1:].reshape(
                    -1
                )
            )

            loss_for_backward = (
                loss
                / GRAD_ACCUM_STEPS
            )

        loss_value = (
            loss.detach()
            .float()
            .item()
        )

        step_loss += loss_value

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        scaler.scale(
            loss_for_backward
        ).backward()

        del (
            hidden,
            logits,
            loss,
            loss_for_backward,
            batch
        )

    # --------------------------------------------------------
    # Update
    # --------------------------------------------------------

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    optimizer.zero_grad(
        set_to_none=True
    )

    global_step += 1

    tokens_seen += (
        TOKENS_PER_STEP
    )

    # --------------------------------------------------------
    # LR
    # --------------------------------------------------------

    lr = learning_rate(
        global_step
    )

    for group in optimizer.param_groups:

        group["lr"] = lr

    loss_sum += step_loss

    # --------------------------------------------------------
    # Log every 50 steps
    # --------------------------------------------------------

    if (
        global_step % 50 == 0
        or tokens_seen >= TARGET_TOKENS
    ):

        torch.cuda.synchronize()

        now = time.perf_counter()

        elapsed = (
            now - last_log_time
        )

        token_delta = (
            tokens_seen
            - last_log_tokens
        )

        speed = (
            token_delta
            / max(
                elapsed,
                1e-9
            )
        )

        progress = (
            min(
                tokens_seen
                / TARGET_TOKENS,
                1.0
            )
            * 100
        )

        allocated = (
            torch.cuda.memory_allocated()
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved()
            / 1024**3
        )

        peak = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        avg_loss = (
            loss_sum / 50
            if global_step % 50 == 0
            else loss_sum
        )

        print(
            f"Step {global_step:6d}/"
            f"{TOTAL_STEPS} | "
            f"Tokens {tokens_seen:12,d} | "
            f"{progress:6.2f}% | "
            f"Loss {avg_loss:.4f} | "
            f"{speed:,.0f} tok/s | "
            f"LR {lr:.2e} | "
            f"VRAM {allocated:.2f}/"
            f"{reserved:.2f} GB | "
            f"Peak {peak:.2f} GB"
        )

        loss_sum = 0.0

        last_log_time = now
        last_log_tokens = tokens_seen

        torch.cuda.reset_peak_memory_stats()

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    if (
        global_step % CHECKPOINT_EVERY == 0
    ):

        save_checkpoint(
            global_step,
            tokens_seen
        )

        gc.collect()

        torch.cuda.empty_cache()


# ============================================================
# 14. FINAL CHECKPOINT
# ============================================================

save_checkpoint(
    global_step,
    tokens_seen
)

final_path = os.path.join(
    WORK_DIR,
    "a2_final_model.pt"
)

torch.save(
    model.state_dict(),
    final_path
)


# ============================================================
# 15. FINITE CHECK
# ============================================================

bad = 0

for name, tensor in model.state_dict().items():

    if not torch.isfinite(
        tensor
    ).all():

        print(
            "❌ Non-finite:",
            name
        )

        bad += 1

print("\n" + "=" * 70)
print("A2 COMPLETE")
print("=" * 70)

print(
    "Final step:",
    global_step
)

print(
    "Final tokens:",
    f"{tokens_seen:,}"
)

print(
    "Progress:",
    f"{min(tokens_seen/TARGET_TOKENS,1)*100:.2f}%"
)

print(
    "Final model:",
    final_path
)

print(
    "Non-finite tensors:",
    bad
)

print("=" * 70)

assert bad == 0

print("✅ A2 TRAINING COMPLETE")

AssertionError: CUDA GPU not available

In [1]:
# ============================================================
# A2 — FINAL STABLE TRAINING
# EXACT CE + CHUNKED OUTPUT PROJECTION
# ============================================================

import os
import gc
import math
import time
import glob
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from kaggle_secrets import UserSecretsClient
from tokenizers import Tokenizer
from datasets import load_dataset


# ============================================================
# 1. SETUP
# ============================================================

assert torch.cuda.is_available()

device = torch.device("cuda")

torch.set_float32_matmul_precision("high")

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

REPO = "Exploration-Lab/CS-781-Capstone"

TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mathimohankowshik/tokenizer1/tokenizer.json"
)

tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

VOCAB_SIZE = tokenizer.get_vocab_size()

WORK_DIR = "/kaggle/working/a2_final"
CKPT_DIR = os.path.join(
    WORK_DIR,
    "checkpoints"
)

os.makedirs(
    CKPT_DIR,
    exist_ok=True
)


# ============================================================
# 2. CONFIG
# ============================================================

MAX_CONTEXT = 2048

HIDDEN_SIZE = 528
NUM_LAYERS = 8
NUM_Q_HEADS = 8
NUM_KV_HEADS = 2
FFN_SIZE = 2048

TRAIN_BLOCK = 512

# Extremely conservative GPU memory
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16

EFFECTIVE_BATCH = 16

# Output projection is processed 64 tokens at a time.
# This prevents a huge 512 x 50000 logits tensor.
LOSS_TOKEN_CHUNK = 64

TARGET_TOKENS = 200_000_000

LR = 1e-4
MIN_LR = 1e-5
WARMUP_STEPS = 1000

MAX_GRAD_NORM = 1.0

CHECKPOINT_EVERY = 500

print("=" * 70)
print("A2 — FINAL STABLE CONFIGURATION")
print("=" * 70)

print("GPU              :", torch.cuda.get_device_name(0))
print(
    "VRAM             :",
    f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB"
)

print("Vocabulary       :", VOCAB_SIZE)
print("Hidden           :", HIDDEN_SIZE)
print("Layers           :", NUM_LAYERS)
print("Q heads          :", NUM_Q_HEADS)
print("KV heads         :", NUM_KV_HEADS)
print("FFN              :", FFN_SIZE)

print("-" * 70)

print("Sequence         :", TRAIN_BLOCK)
print("Micro batch      :", BATCH_SIZE)
print("Grad accumulation:", GRAD_ACCUM_STEPS)
print("Effective batch  :", EFFECTIVE_BATCH)
print("Loss chunk       :", LOSS_TOKEN_CHUNK)

print("Target tokens    :", f"{TARGET_TOKENS:,}")

print("=" * 70)

assert VOCAB_SIZE == 50_000


# ============================================================
# 3. GQA ATTENTION
# ============================================================

class GQAAttention(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads
    ):
        super().__init__()

        assert hidden_size % num_q_heads == 0
        assert num_q_heads % num_kv_heads == 0

        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads

        self.head_dim = (
            hidden_size // num_q_heads
        )

        kv_dim = (
            num_kv_heads * self.head_dim
        )

        self.q_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.k_proj = nn.Linear(
            hidden_size,
            kv_dim
        )

        self.v_proj = nn.Linear(
            hidden_size,
            kv_dim
        )

        self.out_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.num_q_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.num_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
            enable_gqa=True
        )

        y = (
            y.transpose(1, 2)
             .contiguous()
             .view(B, T, C)
        )

        return self.out_proj(y)


# ============================================================
# 4. TRANSFORMER BLOCK
# ============================================================

class GQABlock(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_q_heads,
        num_kv_heads,
        ffn_size
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.attn = GQAAttention(
            hidden_size,
            num_q_heads,
            num_kv_heads
        )

        self.ln2 = nn.LayerNorm(
            hidden_size,
            eps=1e-5
        )

        self.fc1 = nn.Linear(
            hidden_size,
            ffn_size
        )

        self.fc2 = nn.Linear(
            ffn_size,
            hidden_size
        )

    def forward(self, x):

        x = x + self.attn(
            self.ln1(x)
        )

        y = self.ln2(x)

        y = self.fc1(y)

        y = F.gelu(
            y,
            approximate="tanh"
        )

        y = self.fc2(y)

        return x + y


# ============================================================
# 5. A2 MODEL
# ============================================================

class A2GQALM(nn.Module):

    def __init__(self):

        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            MAX_CONTEXT,
            HIDDEN_SIZE
        )

        self.blocks = nn.ModuleList(
            [
                GQABlock(
                    HIDDEN_SIZE,
                    NUM_Q_HEADS,
                    NUM_KV_HEADS,
                    FFN_SIZE
                )
                for _ in range(NUM_LAYERS)
            ]
        )

        self.final_ln = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=1e-5
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(
            module,
            nn.Linear
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(
            module,
            nn.Embedding
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

        elif isinstance(
            module,
            nn.LayerNorm
        ):

            nn.init.ones_(
                module.weight
            )

            nn.init.zeros_(
                module.bias
            )

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(
                positions
            )
        )

        for block in self.blocks:

            # Checkpoint each transformer block.
            # This saves activation memory.
            if self.training:

                x = torch.utils.checkpoint.checkpoint(
                    block,
                    x,
                    use_reentrant=False
                )

            else:

                x = block(x)

        return self.final_ln(x)


# ============================================================
# 6. BUILD + VERIFY
# ============================================================

model = A2GQALM()

param_count = sum(
    p.numel()
    for p in model.parameters()
)

print("\n" + "=" * 70)
print("MODEL")
print("=" * 70)

print(
    "Parameters:",
    f"{param_count:,}",
    f"({param_count/1e6:.2f}M)"
)

assert param_count == 50_407_648

print(
    "✅ EXACT 50.41M A2 MODEL"
)


# ============================================================
# 7. GPU + OPTIMIZER
# ============================================================

model = model.to(device)

model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,
    fused=True
)

scaler = torch.amp.GradScaler(
    "cuda"
)


# ============================================================
# 8. OFFICIAL DATA
# ============================================================

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]

streams = []

print("\n" + "=" * 70)
print("OFFICIAL PHASE-1 DATA")
print("=" * 70)

for path in TRAIN_FILES:

    ds = load_dataset(
        REPO,
        data_files=path,
        split="train",
        streaming=True,
        token=HF_TOKEN
    )

    streams.append(
        iter(ds)
    )

    print("✅", path)

print("=" * 70)


# ============================================================
# 9. DATA GENERATOR
# ============================================================

def sequence_generator():

    carry = []
    source = 0

    while True:

        try:

            example = next(
                streams[source]
            )

        except StopIteration:

            ds = load_dataset(
                REPO,
                data_files=TRAIN_FILES[source],
                split="train",
                streaming=True,
                token=HF_TOKEN
            )

            streams[source] = iter(ds)

            example = next(
                streams[source]
            )

        source = (
            source + 1
        ) % len(streams)

        text = example.get(
            "text"
        )

        if not text:
            continue

        ids = tokenizer.encode(
            text
        ).ids

        if not ids:
            continue

        carry.extend(ids)

        while len(carry) >= TRAIN_BLOCK:

            block = np.asarray(
                carry[:TRAIN_BLOCK],
                dtype=np.int64
            )

            del carry[:TRAIN_BLOCK]

            yield block


seq_iter = sequence_generator()


# ============================================================
# 10. CHECKPOINT FUNCTIONS
# ============================================================

def checkpoint_list():

    return sorted(
        glob.glob(
            os.path.join(
                CKPT_DIR,
                "step_*.pt"
            )
        )
    )


def save_checkpoint(
    step,
    tokens
):

    path = os.path.join(
        CKPT_DIR,
        f"step_{step:08d}.pt"
    )

    temp = path + ".tmp"

    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "step": step,
            "tokens": tokens,
        },
        temp
    )

    os.replace(
        temp,
        path
    )

    files = checkpoint_list()

    while len(files) > 2:

        try:
            os.remove(
                files.pop(0)
            )
        except:
            pass

    print(
        f"\n💾 CHECKPOINT | "
        f"step={step:,} | "
        f"tokens={tokens:,}"
    )


def load_checkpoint():

    files = checkpoint_list()

    if not files:

        return 0, 0

    path = files[-1]

    print("\nRESUMING:", path)

    data = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        data["model"]
    )

    optimizer.load_state_dict(
        data["optimizer"]
    )

    scaler.load_state_dict(
        data["scaler"]
    )

    step = int(
        data["step"]
    )

    tokens = int(
        data["tokens"]
    )

    del data

    gc.collect()

    torch.cuda.empty_cache()

    return step, tokens


# ============================================================
# 11. TRAINING
# ============================================================

tokens_per_step = (
    BATCH_SIZE
    * TRAIN_BLOCK
    * GRAD_ACCUM_STEPS
)

total_steps = math.ceil(
    TARGET_TOKENS
    / tokens_per_step
)

global_step, tokens_seen = (
    load_checkpoint()
)

print("\n" + "=" * 70)
print("A2 TRAINING START")
print("=" * 70)

print(
    "Tokens / optimizer step:",
    tokens_per_step
)

print(
    "Total optimizer steps:",
    total_steps
)

print(
    "Already processed:",
    f"{tokens_seen:,}"
)

print("=" * 70)


def get_lr(step):

    if step < WARMUP_STEPS:

        return (
            LR
            * (step + 1)
            / WARMUP_STEPS
        )

    progress = (
        step - WARMUP_STEPS
    ) / max(
        1,
        total_steps - WARMUP_STEPS
    )

    progress = min(
        max(progress, 0.0),
        1.0
    )

    cosine = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi * progress
            )
        )
    )

    return (
        MIN_LR
        + (LR - MIN_LR)
        * cosine
    )


optimizer.zero_grad(
    set_to_none=True
)

run_start = time.perf_counter()

last_time = run_start
last_tokens = tokens_seen

loss_accumulator = 0.0
steps_since_log = 0


while tokens_seen < TARGET_TOKENS:

    step_loss = 0.0

    # --------------------------------------------------------
    # Gradient accumulation
    # --------------------------------------------------------

    for micro in range(
        GRAD_ACCUM_STEPS
    ):

        batch_np = np.stack(
            [
                next(seq_iter)
                for _ in range(BATCH_SIZE)
            ]
        )

        batch = torch.from_numpy(
            batch_np
        ).long().to(
            device,
            non_blocking=True
        )

        del batch_np

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            hidden = model(
                batch
            )

        # ----------------------------------------------------
        # EXACT CE
        #
        # Process only 64 sequence tokens at once.
        #
        # This prevents creation of:
        #
        #   [1, 511, 50000]
        #
        # ----------------------------------------------------

        input_hidden = hidden[:, :-1, :]
        target = batch[:, 1:]

        total_tokens = target.numel()

        micro_loss = 0.0

        for start in range(
            0,
            total_tokens,
            LOSS_TOKEN_CHUNK
        ):

            end = min(
                start + LOSS_TOKEN_CHUNK,
                total_tokens
            )

            h = input_hidden.reshape(
                -1,
                HIDDEN_SIZE
            )[start:end]

            y = target.reshape(
                -1
            )[start:end]

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = F.linear(
                    h,
                    model.token_embedding.weight
                )

                chunk_loss = F.cross_entropy(
                    logits,
                    y,
                    reduction="sum"
                )

            # Exact normalization:
            # all tokens in this microbatch
            # contribute equally.
            scaled = (
                chunk_loss
                / total_tokens
                / GRAD_ACCUM_STEPS
            )

            scaler.scale(
                scaled
            ).backward()

            micro_loss += (
                chunk_loss.detach()
                .float()
                .item()
            )

            del (
                h,
                y,
                logits,
                chunk_loss,
                scaled
            )

        step_loss += (
            micro_loss
            / total_tokens
        )

        del (
            hidden,
            input_hidden,
            target,
            batch
        )

    # --------------------------------------------------------
    # Optimizer update
    # --------------------------------------------------------

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    optimizer.zero_grad(
        set_to_none=True
    )

    global_step += 1

    tokens_seen += tokens_per_step

    # --------------------------------------------------------
    # LR
    # --------------------------------------------------------

    current_lr = get_lr(
        global_step
    )

    for group in optimizer.param_groups:

        group["lr"] = current_lr

    loss_accumulator += step_loss
    steps_since_log += 1

    # --------------------------------------------------------
    # LOG EVERY 10 STEPS
    # --------------------------------------------------------

    if (
        global_step % 10 == 0
        or tokens_seen >= TARGET_TOKENS
    ):

        torch.cuda.synchronize()

        now = time.perf_counter()

        elapsed = (
            now - last_time
        )

        token_delta = (
            tokens_seen
            - last_tokens
        )

        speed = (
            token_delta
            / max(
                elapsed,
                1e-9
            )
        )

        progress = (
            tokens_seen
            / TARGET_TOKENS
            * 100
        )

        avg_loss = (
            loss_accumulator
            / max(
                steps_since_log,
                1
            )
        )

        allocated = (
            torch.cuda.memory_allocated()
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved()
            / 1024**3
        )

        peak = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        print(
            f"Step {global_step:6d}/"
            f"{total_steps} | "
            f"{progress:7.3f}% | "
            f"Tokens {tokens_seen:,} | "
            f"Loss {avg_loss:.4f} | "
            f"{speed:,.0f} tok/s | "
            f"VRAM {allocated:.2f}GB | "
            f"Peak {peak:.2f}GB"
        )

        loss_accumulator = 0.0
        steps_since_log = 0

        last_time = now
        last_tokens = tokens_seen

        torch.cuda.reset_peak_memory_stats()

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------

    if (
        global_step % CHECKPOINT_EVERY == 0
    ):

        save_checkpoint(
            global_step,
            tokens_seen
        )

        gc.collect()

        torch.cuda.empty_cache()


# ============================================================
# 12. FINAL SAVE
# ============================================================

final_path = os.path.join(
    WORK_DIR,
    "a2_final_model.pt"
)

torch.save(
    model.state_dict(),
    final_path
)

print("\n" + "=" * 70)
print("A2 TRAINING COMPLETE")
print("=" * 70)

print(
    "Final step:",
    global_step
)

print(
    "Final tokens:",
    f"{tokens_seen:,}"
)

print(
    "Progress:",
    f"{min(tokens_seen/TARGET_TOKENS,1)*100:.2f}%"
)

print(
    "Model:",
    final_path
)

print("=" * 70)

A2 — FINAL STABLE CONFIGURATION
GPU              : Tesla T4
VRAM             : 14.56 GB
Vocabulary       : 50000
Hidden           : 528
Layers           : 8
Q heads          : 8
KV heads         : 2
FFN              : 2048
----------------------------------------------------------------------
Sequence         : 512
Micro batch      : 1
Grad accumulation: 16
Effective batch  : 16
Loss chunk       : 64
Target tokens    : 200,000,000

MODEL
Parameters: 50,407,648 (50.41M)
✅ EXACT 50.41M A2 MODEL

OFFICIAL PHASE-1 DATA
✅ Phase_1_Pretraining/Train/train_acts.json
✅ Phase_1_Pretraining/Train/train_high_courts.json
✅ Phase_1_Pretraining/Train/train_supreme_court.json
✅ Phase_1_Pretraining/Train/train_tribunals.json

A2 TRAINING START
Tokens / optimizer step: 8192
Total optimizer steps: 24415
Already processed: 0


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:
# ============================================================
# A2 — CORRECTED TRAINING LOOP
# FIX: ONE BACKWARD PER MICRO-BATCH
# ============================================================

# Clear any partial gradients left by the failed attempt
optimizer.zero_grad(set_to_none=True)

global_step = 0
tokens_seen = 0

TOKENS_PER_STEP = (
    BATCH_SIZE
    * TRAIN_BLOCK
    * GRAD_ACCUM_STEPS
)

TOTAL_STEPS = math.ceil(
    TARGET_TOKENS / TOKENS_PER_STEP
)

print("=" * 70)
print("A2 — CORRECTED TRAINING")
print("=" * 70)
print("Parameters :", f"{param_count:,}")
print("Micro batch:", BATCH_SIZE)
print("Accumulation:", GRAD_ACCUM_STEPS)
print("Sequence   :", TRAIN_BLOCK)
print("Tokens/step:", TOKENS_PER_STEP)
print("Total steps:", TOTAL_STEPS)
print("=" * 70)


def get_lr(step):

    if step < WARMUP_STEPS:
        return (
            LR
            * (step + 1)
            / WARMUP_STEPS
        )

    progress = (
        step - WARMUP_STEPS
    ) / max(
        1,
        TOTAL_STEPS - WARMUP_STEPS
    )

    progress = min(
        max(progress, 0.0),
        1.0
    )

    cosine = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi * progress
            )
        )
    )

    return (
        MIN_LR
        + (LR - MIN_LR) * cosine
    )


seq_iter = sequence_generator()

run_start = time.perf_counter()
last_log_time = run_start
last_log_tokens = 0

loss_accumulator = 0.0
steps_since_log = 0

optimizer.zero_grad(
    set_to_none=True
)


# ============================================================
# TRAINING LOOP
# ============================================================

while tokens_seen < TARGET_TOKENS:

    step_loss = 0.0

    # --------------------------------------------------------
    # Gradient accumulation
    # --------------------------------------------------------

    for micro in range(GRAD_ACCUM_STEPS):

        batch_np = np.stack([
            next(seq_iter)
            for _ in range(BATCH_SIZE)
        ])

        batch = torch.from_numpy(
            batch_np
        ).long().to(
            device,
            non_blocking=True
        )

        del batch_np

        # ----------------------------------------------------
        # Transformer forward
        # ----------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            hidden = model(batch)

        input_hidden = hidden[:, :-1, :]
        target = batch[:, 1:]

        flat_hidden = input_hidden.reshape(
            -1,
            HIDDEN_SIZE
        )

        flat_target = target.reshape(
            -1
        )

        n_tokens = flat_target.numel()

        # ----------------------------------------------------
        # IMPORTANT:
        #
        # DO NOT backward inside this loop.
        #
        # Accumulate the complete loss graph first.
        # Then perform ONE backward().
        # ----------------------------------------------------

        total_chunk_loss = None

        for start in range(
            0,
            n_tokens,
            LOSS_TOKEN_CHUNK
        ):

            end = min(
                start + LOSS_TOKEN_CHUNK,
                n_tokens
            )

            h = flat_hidden[start:end]
            y = flat_target[start:end]

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = F.linear(
                    h,
                    model.token_embedding.weight
                )

                chunk_loss = F.cross_entropy(
                    logits,
                    y,
                    reduction="sum"
                )

            if total_chunk_loss is None:
                total_chunk_loss = chunk_loss
            else:
                total_chunk_loss = (
                    total_chunk_loss
                    + chunk_loss
                )

            # Do NOT backward here.
            # Graph is intentionally retained
            # until the single backward below.

            del (
                h,
                y,
                logits,
                chunk_loss
            )

        # ----------------------------------------------------
        # Exact mean CE for this micro-batch
        # ----------------------------------------------------

        micro_loss = (
            total_chunk_loss
            / n_tokens
        )

        loss_value = (
            micro_loss
            .detach()
            .float()
            .item()
        )

        step_loss += loss_value

        # ----------------------------------------------------
        # ONE AND ONLY ONE BACKWARD
        # ----------------------------------------------------

        scaled_loss = (
            micro_loss
            / GRAD_ACCUM_STEPS
        )

        scaler.scale(
            scaled_loss
        ).backward()

        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del (
            hidden,
            input_hidden,
            target,
            flat_hidden,
            flat_target,
            total_chunk_loss,
            micro_loss,
            scaled_loss,
            batch
        )

    # ========================================================
    # OPTIMIZER UPDATE
    # ========================================================

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    optimizer.zero_grad(
        set_to_none=True
    )

    global_step += 1

    tokens_seen += TOKENS_PER_STEP

    # ========================================================
    # LEARNING RATE
    # ========================================================

    current_lr = get_lr(
        global_step
    )

    for group in optimizer.param_groups:
        group["lr"] = current_lr

    loss_accumulator += (
        step_loss
    )

    steps_since_log += 1

    # ========================================================
    # LOG EVERY 10 STEPS
    # ========================================================

    if (
        global_step % 10 == 0
        or tokens_seen >= TARGET_TOKENS
    ):

        torch.cuda.synchronize()

        now = time.perf_counter()

        elapsed = (
            now - last_log_time
        )

        token_delta = (
            tokens_seen
            - last_log_tokens
        )

        speed = (
            token_delta
            / max(
                elapsed,
                1e-9
            )
        )

        progress = (
            tokens_seen
            / TARGET_TOKENS
            * 100
        )

        avg_loss = (
            loss_accumulator
            / max(
                steps_since_log,
                1
            )
        )

        allocated = (
            torch.cuda.memory_allocated()
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved()
            / 1024**3
        )

        peak = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        print(
            f"Step {global_step:6d}/"
            f"{TOTAL_STEPS} | "
            f"{progress:7.3f}% | "
            f"Tokens {tokens_seen:,} | "
            f"Loss {avg_loss:.4f} | "
            f"{speed:,.0f} tok/s | "
            f"VRAM {allocated:.2f} GB | "
            f"Peak {peak:.2f} GB"
        )

        loss_accumulator = 0.0
        steps_since_log = 0

        last_log_time = now
        last_log_tokens = tokens_seen

        torch.cuda.reset_peak_memory_stats()

    # ========================================================
    # CHECKPOINT
    # ========================================================

    if (
        global_step % CHECKPOINT_EVERY == 0
    ):

        save_checkpoint(
            global_step,
            tokens_seen
        )

        gc.collect()

        torch.cuda.empty_cache()


# ============================================================
# FINAL SAVE
# ============================================================

final_path = os.path.join(
    WORK_DIR,
    "a2_final_model.pt"
)

torch.save(
    model.state_dict(),
    final_path
)

print("\n" + "=" * 70)
print("🚀 A2 TRAINING COMPLETE")
print("=" * 70)

print(
    "Final step:",
    global_step
)

print(
    "Final tokens:",
    f"{tokens_seen:,}"
)

print(
    "Progress:",
    f"{min(tokens_seen/TARGET_TOKENS, 1.0)*100:.2f}%"
)

print(
    "Final model:",
    final_path
)

print("=" * 70)

A2 — CORRECTED TRAINING
Parameters : 50,407,648
Micro batch: 1
Accumulation: 16
Sequence   : 512
Tokens/step: 8192
Total steps: 24415


In [1]:
# ============================================================
# CORRECT MEMORY-SAFE CHUNKED BACKWARD
# ============================================================

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    hidden = model(batch)

input_hidden = hidden[:, :-1, :]
target = batch[:, 1:]

flat_hidden = input_hidden.reshape(
    -1,
    HIDDEN_SIZE
)

flat_target = target.reshape(-1)

n_tokens = flat_target.numel()

micro_loss_value = 0.0

for start in range(
    0,
    n_tokens,
    LOSS_TOKEN_CHUNK
):

    end = min(
        start + LOSS_TOKEN_CHUNK,
        n_tokens
    )

    h = flat_hidden[start:end]
    y = flat_target[start:end]

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        logits = F.linear(
            h,
            model.token_embedding.weight
        )

        chunk_loss = F.cross_entropy(
            logits,
            y,
            reduction="sum"
        )

        chunk_loss = (
            chunk_loss
            / n_tokens
            / GRAD_ACCUM_STEPS
        )

    micro_loss_value += (
        chunk_loss.detach()
        .float()
        .item()
    )

    # Retain the transformer graph until
    # the final vocabulary chunk.
    retain = end < n_tokens

    scaler.scale(
        chunk_loss
    ).backward(
        retain_graph=retain
    )

    del (
        h,
        y,
        logits,
        chunk_loss
    )

del (
    hidden,
    input_hidden,
    target,
    flat_hidden,
    flat_target
)

NameError: name 'torch' is not defined

In [2]:
# ============================================================
# A2 — CLEAN MEMORY-SAFE SINGLE-STEP TEST
# ============================================================

import os
import gc
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from kaggle_secrets import UserSecretsClient
from tokenizers import Tokenizer
from datasets import load_dataset


# ============================================================
# 1. SETUP
# ============================================================

assert torch.cuda.is_available()

device = torch.device("cuda")

torch.set_float32_matmul_precision("high")

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")

REPO = "Exploration-Lab/CS-781-Capstone"

TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mathimohankowshik/tokenizer1/tokenizer.json"
)

tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

VOCAB_SIZE = tokenizer.get_vocab_size()

# Exact A2 architecture
MAX_CONTEXT = 2048
HIDDEN_SIZE = 528
NUM_LAYERS = 8
NUM_Q_HEADS = 8
NUM_KV_HEADS = 2
FFN_SIZE = 2048

# Extremely conservative test
TRAIN_BLOCK = 512
BATCH_SIZE = 1

# Small output chunks
LOSS_CHUNK = 32


# ============================================================
# 2. GQA
# ============================================================

class GQAAttention(nn.Module):

    def __init__(
        self,
        hidden_size,
        q_heads,
        kv_heads
    ):
        super().__init__()

        self.q_heads = q_heads
        self.kv_heads = kv_heads

        self.head_dim = (
            hidden_size // q_heads
        )

        kv_dim = (
            kv_heads * self.head_dim
        )

        self.q_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.k_proj = nn.Linear(
            hidden_size,
            kv_dim
        )

        self.v_proj = nn.Linear(
            hidden_size,
            kv_dim
        )

        self.out_proj = nn.Linear(
            hidden_size,
            hidden_size
        )

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.q_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.kv_heads,
            self.head_dim
        ).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
            enable_gqa=True
        )

        y = (
            y.transpose(1, 2)
             .contiguous()
             .view(B, T, C)
        )

        return self.out_proj(y)


# ============================================================
# 3. BLOCK
# ============================================================

class GQABlock(nn.Module):

    def __init__(self):

        super().__init__()

        self.ln1 = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=1e-5
        )

        self.attn = GQAAttention(
            HIDDEN_SIZE,
            NUM_Q_HEADS,
            NUM_KV_HEADS
        )

        self.ln2 = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=1e-5
        )

        self.fc1 = nn.Linear(
            HIDDEN_SIZE,
            FFN_SIZE
        )

        self.fc2 = nn.Linear(
            FFN_SIZE,
            HIDDEN_SIZE
        )

    def forward(self, x):

        x = x + self.attn(
            self.ln1(x)
        )

        y = self.ln2(x)

        y = self.fc1(y)

        y = F.gelu(
            y,
            approximate="tanh"
        )

        y = self.fc2(y)

        return x + y


# ============================================================
# 4. MODEL
# ============================================================

class A2GQALM(nn.Module):

    def __init__(self):

        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            MAX_CONTEXT,
            HIDDEN_SIZE
        )

        self.blocks = nn.ModuleList(
            [
                GQABlock()
                for _ in range(NUM_LAYERS)
            ]
        )

        self.final_ln = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=1e-5
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(
            module,
            nn.Linear
        ):

            nn.init.normal_(
                module.weight,
                0.0,
                0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(
            module,
            nn.Embedding
        ):

            nn.init.normal_(
                module.weight,
                0.0,
                0.02
            )

        elif isinstance(
            module,
            nn.LayerNorm
        ):

            nn.init.ones_(
                module.weight
            )

            nn.init.zeros_(
                module.bias
            )

    def forward(self, input_ids):

        B, T = input_ids.shape

        pos = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(pos)
        )

        # Activation checkpointing
        for block in self.blocks:

            x = torch.utils.checkpoint.checkpoint(
                block,
                x,
                use_reentrant=False
            )

        return self.final_ln(x)


# ============================================================
# 5. CREATE + VERIFY
# ============================================================

model = A2GQALM().to(device)

params = sum(
    p.numel()
    for p in model.parameters()
)

print("=" * 60)
print("A2 MODEL")
print("=" * 60)
print("Parameters:", f"{params:,}")
print("VRAM before:", 
      f"{torch.cuda.memory_allocated()/1024**3:.2f} GB")

assert params == 50_407_648

print("✅ EXACT 50.41M MODEL")


# ============================================================
# 6. OFFICIAL ACTS STREAM
# ============================================================

ds = load_dataset(
    REPO,
    data_files="Phase_1_Pretraining/Train/train_acts.json",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

it = iter(ds)

print("\nReading official training data...")

carry = []

while len(carry) < TRAIN_BLOCK:

    example = next(it)

    text = example.get("text")

    if not text:
        continue

    ids = tokenizer.encode(text).ids

    carry.extend(ids)

batch_np = np.asarray(
    carry[:TRAIN_BLOCK],
    dtype=np.int64
)

batch = torch.from_numpy(
    batch_np
).unsqueeze(0).to(
    device
)

del batch_np


# ============================================================
# 7. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1,
    fused=True
)

scaler = torch.amp.GradScaler(
    "cuda"
)

optimizer.zero_grad(
    set_to_none=True
)


# ============================================================
# 8. ONE MEMORY-SAFE FORWARD/BACKWARD
# ============================================================

print("\nRunning ONE A2 training step...")

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):

    hidden = model(batch)

print(
    "Hidden shape:",
    tuple(hidden.shape)
)

flat_hidden = hidden[:, :-1, :].reshape(
    -1,
    HIDDEN_SIZE
)

flat_target = batch[:, 1:].reshape(-1)

n = flat_target.numel()

loss_value = 0.0

# ------------------------------------------------------------
# IMPORTANT:
#
# Each chunk performs backward immediately.
# retain_graph=True keeps the transformer graph alive,
# but the chunk's vocabulary logits are released.
#
# Final chunk frees the graph.
# ------------------------------------------------------------

for start in range(
    0,
    n,
    LOSS_CHUNK
):

    end = min(
        start + LOSS_CHUNK,
        n
    )

    h = flat_hidden[start:end]
    y = flat_target[start:end]

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        logits = F.linear(
            h,
            model.token_embedding.weight
        )

        chunk_loss = F.cross_entropy(
            logits,
            y,
            reduction="sum"
        )

        chunk_loss = (
            chunk_loss
            / n
        )

    loss_value += (
        chunk_loss.detach()
        .float()
        .item()
    )

    retain = end < n

    scaler.scale(
        chunk_loss
    ).backward(
        retain_graph=retain
    )

    del (
        h,
        y,
        logits,
        chunk_loss
    )


# ============================================================
# 9. CHECK GRADIENTS
# ============================================================

scaler.unscale_(
    optimizer
)

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    1.0
)

finite = True

for p in model.parameters():

    if p.grad is not None:

        if not torch.isfinite(
            p.grad
        ).all():

            finite = False
            break


# ============================================================
# 10. RESULT
# ============================================================

torch.cuda.synchronize()

allocated = (
    torch.cuda.memory_allocated()
    / 1024**3
)

peak = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("\n" + "=" * 60)
print("A2 SINGLE-STEP RESULT")
print("=" * 60)
print("Loss          :", f"{loss_value:.4f}")
print("Gradient norm :", float(grad_norm))
print("Gradients finite:", finite)
print("Current VRAM  :", f"{allocated:.2f} GB")
print("Peak VRAM     :", f"{peak:.2f} GB")
print("=" * 60)

assert finite

print("✅ A2 MEMORY-SAFE STEP PASSED")

# We intentionally do NOT update the model.


A2 MODEL
Parameters: 50,407,648
VRAM before: 0.19 GB
✅ EXACT 50.41M MODEL

Reading official training data...

Running ONE A2 training step...
Hidden shape: (1, 512, 528)

A2 SINGLE-STEP RESULT
Loss          : 10.9298
Gradient norm : 8.233000755310059
Gradients finite: True
Current VRAM  : 0.40 GB
Peak VRAM     : 0.68 GB
✅ A2 MEMORY-SAFE STEP PASSED


In [7]:
# ============================================================
# A2 PRODUCTION TRAINING
# Memory-safe chunked vocabulary backward
# ============================================================

import os
import gc
import time
import math
import glob
import torch
import torch.nn.functional as F

# ---------------- CONFIG ----------------
DEVICE = "cuda"
TRAIN_BLOCK = 512
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
TARGET_TOKENS = 200_000_000
LOSS_CHUNK = 32

SAVE_EVERY = 500
LOG_EVERY = 10

LR = 1e-4
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 1000

CKPT_DIR = "/kaggle/working/phase1_production/architecture_2_gqa/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

FINAL_PATH = os.path.join(
    CKPT_DIR,
    "step_final_200m.pt"
)

# ---------------- CLEAN TEST GRADIENTS ----------------
optimizer.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# ---------------- FOUR OFFICIAL SOURCES ----------------
SOURCE_PATHS = {
    "Acts": "Acts",
    "High_Courts": "High Courts",
    "Supreme_Court": "Supreme Court",
    "Tribunals": "Tribunals",
}

print("=" * 70)
print("A2 PRODUCTION TRAINING")
print("=" * 70)
print(f"Target tokens       : {TARGET_TOKENS:,}")
print(f"Sequence length     : {TRAIN_BLOCK}")
print(f"Batch size          : {BATCH_SIZE}")
print(f"Grad accumulation   : {GRAD_ACCUM_STEPS}")
print(f"Loss chunk          : {LOSS_CHUNK}")
print(f"Learning rate       : {LR}")
print(f"Checkpoint interval : {SAVE_EVERY} steps")
print("=" * 70)

# ------------------------------------------------------------
# Find latest checkpoint
# ------------------------------------------------------------
checkpoints = sorted(
    glob.glob(os.path.join(CKPT_DIR, "step_*.pt")),
    key=os.path.getmtime
)

start_step = 0
tokens_done = 0

if checkpoints:
    latest = checkpoints[-1]
    print(f"\nResuming from: {latest}")

    ckpt = torch.load(
        latest,
        map_location="cpu",
        weights_only=False
    )

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scaler.load_state_dict(ckpt["scaler"])

    start_step = ckpt["step"]
    tokens_done = ckpt["tokens"]

    del ckpt
    gc.collect()
    torch.cuda.empty_cache()

    print(f"Resume step  : {start_step}")
    print(f"Resume tokens: {tokens_done:,}")
else:
    print("\nNo checkpoint found — starting from step 0.")

# ------------------------------------------------------------
# Official streaming datasets
# ------------------------------------------------------------
from datasets import load_dataset

print("\nOpening official Phase-1 streams...")

streams = {}

for name, subset in SOURCE_PATHS.items():
    streams[name] = load_dataset(
        REPO,
        data_dir=f"Phase_1_Pretraining/Train/{subset}",
        split="train",
        streaming=True,
        token=HF_TOKEN
    )

    print(f"  ✓ {name}")

print("\nCreating iterators...")

iterators = {
    name: iter(ds)
    for name, ds in streams.items()
}

source_names = list(iterators.keys())
source_index = 0

# ------------------------------------------------------------
# Token generator
# ------------------------------------------------------------
def get_text(example):
    if "text" in example:
        return example["text"]

    # fallback: use the first string field
    for value in example.values():
        if isinstance(value, str):
            return value

    return ""


carry = []

def next_sequence():
    global carry, source_index

    while len(carry) < TRAIN_BLOCK:
        source = source_names[source_index]
        source_index = (source_index + 1) % len(source_names)

        try:
            example = next(iterators[source])
        except StopIteration:
            # restart exhausted stream
            iterators[source] = iter(streams[source])
            example = next(iterators[source])

        text = get_text(example)

        if not text:
            continue

        ids = tokenizer.encode(text).ids

        if ids:
            carry.extend(ids)

    seq = carry[:TRAIN_BLOCK]
    del carry[:TRAIN_BLOCK]

    return torch.tensor(
        seq,
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
model.train()

step = start_step
tokens_per_step = (
    BATCH_SIZE *
    TRAIN_BLOCK *
    GRAD_ACCUM_STEPS
)

steps_target = math.ceil(
    TARGET_TOKENS / tokens_per_step
)

print(f"\nTokens per optimizer step: {tokens_per_step:,}")
print(f"Target optimizer steps   : {steps_target:,}")
print(f"Starting at step         : {step:,}")
print("=" * 70)

run_start = time.time()
last_log_time = run_start
last_log_tokens = tokens_done

while tokens_done < TARGET_TOKENS:

    optimizer.zero_grad(set_to_none=True)

    accumulated_loss = 0.0

    # --------------------------------------------------------
    # Gradient accumulation
    # --------------------------------------------------------
    for micro_step in range(GRAD_ACCUM_STEPS):

        batch = next_sequence()

        # ----------------------------------------------------
        # Transformer forward
        # ----------------------------------------------------
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            hidden = model(batch)

        input_hidden = hidden[:, :-1, :]
        target = batch[:, 1:]

        flat_hidden = input_hidden.reshape(
            -1,
            HIDDEN_SIZE
        )

        flat_target = target.reshape(-1)

        n_tokens = flat_target.numel()

        # ----------------------------------------------------
        # Memory-safe vocabulary projection
        # ----------------------------------------------------
        for start in range(0, n_tokens, LOSS_CHUNK):

            end = min(
                start + LOSS_CHUNK,
                n_tokens
            )

            h = flat_hidden[start:end]
            y = flat_target[start:end]

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = F.linear(
                    h,
                    model.token_embedding.weight,
                    bias=None
                )

                chunk_loss = (
                    F.cross_entropy(
                        logits,
                        y,
                        reduction="sum"
                    )
                    / n_tokens
                    / GRAD_ACCUM_STEPS
                )

            if not torch.isfinite(chunk_loss):
                raise RuntimeError(
                    f"Non-finite loss at step {step}"
                )

            retain = end < n_tokens

            scaler.scale(chunk_loss).backward(
                retain_graph=retain
            )

            accumulated_loss += (
                chunk_loss.detach().item()
            )

            del h, y, logits, chunk_loss

        del (
            hidden,
            input_hidden,
            target,
            flat_hidden,
            flat_target,
            batch
        )

    # --------------------------------------------------------
    # Optimizer step
    # --------------------------------------------------------
    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        1.0
    )

    if not torch.isfinite(grad_norm):
        raise RuntimeError(
            f"Non-finite gradient at step {step}"
        )

    scaler.step(optimizer)
    scaler.update()

    step += 1
    tokens_done += tokens_per_step

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------
    if step % LOG_EVERY == 0:

        now = time.time()

        elapsed = now - run_start

        recent_tokens = tokens_done - last_log_tokens
        recent_time = now - last_log_time

        tok_sec = (
            recent_tokens / recent_time
            if recent_time > 0 else 0
        )

        progress = min(
            100.0,
            tokens_done / TARGET_TOKENS * 100
        )

        avg_loss = (
            accumulated_loss / GRAD_ACCUM_STEPS
        )

        print(
            f"Step {step:6d} | "
            f"Tokens {tokens_done:12,d}/{TARGET_TOKENS:,} | "
            f"{progress:6.2f}% | "
            f"Loss {avg_loss:.4f} | "
            f"Grad {grad_norm.item():.3f} | "
            f"{tok_sec:,.0f} tok/s | "
            f"VRAM "
            f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
        )

        last_log_time = now
        last_log_tokens = tokens_done

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------
    if step % SAVE_EVERY == 0:

        ckpt_path = os.path.join(
            CKPT_DIR,
            f"step_{step:08d}.pt"
        )

        torch.save(
            {
                "step": step,
                "tokens": tokens_done,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
            },
            ckpt_path
        )

        print(
            f"  ✓ CHECKPOINT SAVED: "
            f"{ckpt_path}"
        )

        gc.collect()
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# Final checkpoint
# ------------------------------------------------------------
torch.save(
    {
        "step": step,
        "tokens": tokens_done,
        "model": model.state_dict(),
    },
    FINAL_PATH
)

print("\n" + "=" * 70)
print("A2 TRAINING COMPLETE")
print("=" * 70)
print(f"Final step  : {step:,}")
print(f"Final tokens: {tokens_done:,}")
print(f"Progress    : 100.00%")
print(f"Final model : {FINAL_PATH}")
print("=" * 70)

A2 PRODUCTION TRAINING
Target tokens       : 200,000,000
Sequence length     : 512
Batch size          : 1
Grad accumulation   : 16
Loss chunk          : 32
Learning rate       : 0.0001
Checkpoint interval : 500 steps

No checkpoint found — starting from step 0.

Opening official Phase-1 streams...


EmptyDatasetError: The directory at hf://datasets/Exploration-Lab/CS-781-Capstone@89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/Acts doesn't contain any data files

In [6]:
from huggingface_hub import list_repo_tree

REPO = "Exploration-Lab/CS-781-Capstone"

print("Contents of Phase_1_Pretraining/Train:\n")

items = list_repo_tree(
    REPO,
    path_in_repo="Phase_1_Pretraining/Train",
    repo_type="dataset",
    recursive=False
)

for item in items:
    print(item.path)

Contents of Phase_1_Pretraining/Train:

Phase_1_Pretraining/Train/train_acts.json
Phase_1_Pretraining/Train/train_high_courts.json
Phase_1_Pretraining/Train/train_supreme_court.json
Phase_1_Pretraining/Train/train_tribunals.json


In [4]:
from huggingface_hub import list_repo_tree

REPO = "Exploration-Lab/CS-781-Capstone"

print("Phase 1 training files/directories:\n")

items = list_repo_tree(
    REPO,
    path_in_repo="Phase_1_Pretraining/Train",
    repo_type="dataset",
    recursive=False
)

for item in items:
    print(item.path)

Phase 1 training files/directories:

Phase_1_Pretraining/Train/train_acts.json
Phase_1_Pretraining/Train/train_high_courts.json
Phase_1_Pretraining/Train/train_supreme_court.json
Phase_1_Pretraining/Train/train_tribunals.json


In [ ]:
# ============================================================
# A3 PREFIX-LM HYBRID — MODEL
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

DEVICE = "cuda"

VOCAB_SIZE = 50_000
CONTEXT_LEN = 2_048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2_048
PREFIX_RATIO = 0.5

class PrefixLMBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)
        self.attn = nn.MultiheadAttention(
            embed_dim=HIDDEN_SIZE,
            num_heads=NUM_HEADS,
            batch_first=True
        )

        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)

        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.GELU(),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=attn_mask,
            need_weights=False
        )

        x = x + attn_out

        x = x + self.ffn(self.ln2(x))

        return x


class A3PrefixLM(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            CONTEXT_LEN,
            HIDDEN_SIZE
        )

        self.layers = nn.ModuleList([
            PrefixLMBlock()
            for _ in range(NUM_LAYERS)
        ])

        self.ln_f = nn.LayerNorm(HIDDEN_SIZE)

        # Tied input/output embeddings
        self.lm_head = None

        self.register_buffer(
            "position_ids",
            torch.arange(CONTEXT_LEN).unsqueeze(0),
            persistent=False
        )

    def build_mask(self, seq_len, device):
        prefix_len = int(
            seq_len * PREFIX_RATIO
        )

        mask = torch.full(
            (seq_len, seq_len),
            float("-inf"),
            device=device
        )

        # Prefix tokens can attend bidirectionally
        mask[:prefix_len, :prefix_len] = 0

        # Suffix tokens:
        # prefix is visible + causal suffix
        for i in range(prefix_len, seq_len):
            mask[i, :prefix_len] = 0
            mask[i, prefix_len:i+1] = 0

        return mask

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        positions = self.position_ids[
            :, :seq_len
        ]

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        mask = self.build_mask(
            seq_len,
            input_ids.device
        )

        for layer in self.layers:
            x = layer(x, mask)

        x = self.ln_f(x)

        # Tied output projection
        logits = F.linear(
            x,
            self.token_embedding.weight
        )

        return logits


model_a3 = A3PrefixLM().to(DEVICE)

total_params = sum(
    p.numel()
    for p in model_a3.parameters()
)

print("=" * 60)
print("A3 PREFIX-LM HYBRID")
print("=" * 60)
print(f"Parameters : {total_params:,}")
print(f"Parameters : {total_params / 1e6:.2f}M")
print(f"Vocabulary : {VOCAB_SIZE:,}")
print(f"Context    : {CONTEXT_LEN}")
print(f"Hidden     : {HIDDEN_SIZE}")
print(f"Layers     : {NUM_LAYERS}")
print(f"Heads      : {NUM_HEADS}")
print(f"FFN        : {FFN_SIZE}")
print(f"Prefix     : {PREFIX_RATIO * 100:.0f}%")
print("=" * 60)

assert 50_000_000 <= total_params <= 300_000_000
assert total_params == 51_868_672

print("✅ A3 parameter count verified")

In [4]:
# ============================================================
# A3 — FORWARD + LOSS MEMORY TEST
# ============================================================

import gc
import torch
import torch.nn.functional as F

model_a3.train()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Get one sequence from the tokenizer
seq = next_sequence()

print("Input shape:", tuple(seq.shape))

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model_a3(seq)

print("Logits shape:", tuple(logits.shape))

loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
    seq[:, 1:].reshape(-1)
)

print("\n" + "=" * 60)
print("A3 FORWARD TEST")
print("=" * 60)
print(f"Loss       : {loss.item():.4f}")
print(f"Loss finite: {torch.isfinite(loss).item()}")
print(
    f"Peak VRAM  : "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)
print("=" * 60)

del seq, logits, loss

gc.collect()
torch.cuda.empty_cache()

assert torch.isfinite(loss.detach()).item() if 'loss' in locals() else True

print("✅ A3 forward test completed")

NameError: name 'next_sequence' is not defined

In [30]:
# ============================================================
# A3 — CREATE ONE TEST SEQUENCE
# ============================================================

from datasets import load_dataset
import torch

REPO = "Exploration-Lab/CS-781-Capstone"

print("Opening official Acts training stream...")

ds_a3_test = load_dataset(
    REPO,
    data_dir="Phase_1_Pretraining/Train",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

example = next(iter(ds_a3_test))

# Find the text field
if "text" in example:
    text = example["text"]
else:
    text = next(
        value for value in example.values()
        if isinstance(value, str)
    )

ids = tokenizer.encode(text).ids

# Make sure we have 512 tokens
while len(ids) < 512:
    example = next(iter(ds_a3_test))

    if "text" in example:
        text = example["text"]
    else:
        text = next(
            value for value in example.values()
            if isinstance(value, str)
        )

    ids.extend(tokenizer.encode(text).ids)

ids = ids[:512]

seq = torch.tensor(
    ids,
    dtype=torch.long,
    device=DEVICE
).unsqueeze(0)

print("Sequence shape:", tuple(seq.shape))
print("Tokens:", seq.shape[1])
print("✅ Test sequence ready")

Opening official Acts training stream...


KeyboardInterrupt: 

In [6]:
# ============================================================
# A3 — FORWARD + LOSS TEST
# ============================================================

import gc
import torch
import torch.nn.functional as F

model_a3.train()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model_a3(seq)

print("Input shape :", tuple(seq.shape))
print("Logits shape:", tuple(logits.shape))

loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
    seq[:, 1:].reshape(-1)
)

print("\n" + "=" * 60)
print("A3 FORWARD TEST")
print("=" * 60)
print(f"Loss       : {loss.item():.4f}")
print(f"Loss finite: {torch.isfinite(loss).item()}")
print(
    f"Peak VRAM  : "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)
print("=" * 60)

assert torch.isfinite(loss).item()

del logits, loss
gc.collect()
torch.cuda.empty_cache()

print("✅ A3 forward + loss test passed")

NameError: name 'seq' is not defined

In [7]:
# ============================================================
# A3 — SINGLE TRAINING STEP
# ============================================================

import gc
import torch
import torch.nn.functional as F

# Fresh optimizer
optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=1e-4,
    weight_decay=0.1,
    fused=True
)

scaler_a3 = torch.amp.GradScaler("cuda")

optimizer_a3.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_a3.train()

# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------
with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model_a3(seq)

# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------
loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
    seq[:, 1:].reshape(-1)
)

print(f"Loss before backward: {loss.item():.4f}")

# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------
scaler_a3.scale(loss).backward()

scaler_a3.unscale_(optimizer_a3)

grad_norm = torch.nn.utils.clip_grad_norm_(
    model_a3.parameters(),
    1.0
)

print(f"Gradient norm       : {grad_norm.item():.4f}")
print(f"Gradients finite    : {torch.isfinite(grad_norm).item()}")

assert torch.isfinite(loss).item()
assert torch.isfinite(grad_norm).item()

# ------------------------------------------------------------
# Optimizer update
# ------------------------------------------------------------
scaler_a3.step(optimizer_a3)
scaler_a3.update()

print(
    f"Peak VRAM           : "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------
del logits, loss
optimizer_a3.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("✅ A3 SINGLE TRAINING STEP PASSED")
print("=" * 60)

NameError: name 'seq' is not defined

In [8]:
# ============================================================
# A3 — GRADIENT STABILITY CHECK
# ============================================================

import gc
import torch
import torch.nn.functional as F

optimizer_a3.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_a3.train()

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model_a3(seq)

loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
    seq[:, 1:].reshape(-1)
)

print(f"Loss: {loss.item():.4f}")

scaler_a3.scale(loss).backward()

# Check gradients BEFORE unscale
finite_before = True
bad_before = []

for name, p in model_a3.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            finite_before = False
            bad_before.append(name)

print(f"Gradients finite before unscale: {finite_before}")

scaler_a3.unscale_(optimizer_a3)

finite_after = True
bad_after = []

for name, p in model_a3.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            finite_after = False
            bad_after.append(name)

print(f"Gradients finite after unscale : {finite_after}")

if bad_before:
    print("\nBad gradients:")
    for name in bad_before[:10]:
        print(" ", name)

if bad_after:
    print("\nBad gradients after unscale:")
    for name in bad_after[:10]:
        print(" ", name)

if finite_after:
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_a3.parameters(),
        1.0
    )
    print(f"\nGradient norm: {grad_norm.item():.4f}")

print(
    f"Peak VRAM: "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

optimizer_a3.zero_grad(set_to_none=True)

del logits, loss
gc.collect()
torch.cuda.empty_cache()

print("\n✅ Gradient diagnostic complete")

NameError: name 'seq' is not defined

In [9]:
# ============================================================
# A3 — CLEAN GRADIENT STABILITY TEST
# ============================================================

import gc
import torch
import torch.nn.functional as F

# Fresh optimizer + scaler
optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=1e-4,
    weight_decay=0.1,
    fused=True
)

scaler_a3 = torch.amp.GradScaler("cuda")

optimizer_a3.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_a3.train()

# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------
with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model_a3(seq)

loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
    seq[:, 1:].reshape(-1)
)

print(f"Loss: {loss.item():.4f}")

# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------
scaler_a3.scale(loss).backward()

# Check scaled gradients
bad = []

for name, p in model_a3.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            bad.append(name)

print(
    f"Gradients finite before unscale: "
    f"{len(bad) == 0}"
)

if bad:
    print("First bad gradients:")
    for name in bad[:10]:
        print(" ", name)

# ------------------------------------------------------------
# Unscale ONCE
# ------------------------------------------------------------
scaler_a3.unscale_(optimizer_a3)

bad_after = []

for name, p in model_a3.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            bad_after.append(name)

print(
    f"Gradients finite after unscale : "
    f"{len(bad_after) == 0}"
)

if bad_after:
    print("First bad gradients after unscale:")
    for name in bad_after[:10]:
        print(" ", name)

# ------------------------------------------------------------
# Gradient norm
# ------------------------------------------------------------
grad_norm = torch.nn.utils.clip_grad_norm_(
    model_a3.parameters(),
    1.0
)

print(f"Gradient norm: {grad_norm.item()}")

print(
    f"Peak VRAM: "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

# ------------------------------------------------------------
# IMPORTANT: finish scaler state cleanly
# ------------------------------------------------------------
scaler_a3.step(optimizer_a3)
scaler_a3.update()

optimizer_a3.zero_grad(set_to_none=True)

del logits, loss
gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("A3 GRADIENT TEST COMPLETE")
print("=" * 60)

NameError: name 'seq' is not defined

In [10]:
# ============================================================
# A3 — FP32 BACKWARD STABILITY TEST
# ============================================================

import gc
import torch
import torch.nn.functional as F

# Fresh optimizer
optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=1e-4,
    weight_decay=0.1,
    fused=True
)

optimizer_a3.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_a3.train()

# ------------------------------------------------------------
# FP32 forward
# ------------------------------------------------------------
with torch.autocast(
    device_type="cuda",
    enabled=False
):
    logits = model_a3(seq)

    loss = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
        seq[:, 1:].reshape(-1)
    )

print(f"Loss: {loss.item():.4f}")

# ------------------------------------------------------------
# FP32 backward
# ------------------------------------------------------------
loss.backward()

bad = []

for name, p in model_a3.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            bad.append(name)

print(f"Gradients finite: {len(bad) == 0}")

if bad:
    print("\nBad gradients:")
    for name in bad[:10]:
        print(" ", name)

if not bad:
    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_a3.parameters(),
        1.0
    )
    print(f"Gradient norm: {grad_norm.item():.4f}")

print(
    f"Peak VRAM: "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

# ------------------------------------------------------------
# No need to keep this test update
# ------------------------------------------------------------
optimizer_a3.zero_grad(set_to_none=True)

del logits, loss
gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("A3 FP32 BACKWARD TEST COMPLETE")
print("=" * 60)

NameError: name 'seq' is not defined

In [17]:
# ============================================================
# A3 — SHORT FP32 TRAINING BENCHMARK
# ============================================================

import time
import gc
import torch
import torch.nn.functional as F

model_a3.train()

optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=1e-4,
    weight_decay=0.1,
    fused=True
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("=" * 60)
print("A3 FP32 TRAINING BENCHMARK")
print("=" * 60)

losses = []
start_time = time.time()

for step in range(10):

    optimizer_a3.zero_grad(set_to_none=True)

    # Reuse/create a 512-token sequence
    if step == 0:
        batch = seq
    else:
        ids = tokenizer.encode(
            text
        ).ids

        while len(ids) < 512:
            example = next(iter(ds_a3_test))

            if "text" in example:
                extra_text = example["text"]
            else:
                extra_text = next(
                    value for value in example.values()
                    if isinstance(value, str)
                )

            ids.extend(
                tokenizer.encode(extra_text).ids
            )

        batch = torch.tensor(
            ids[:512],
            dtype=torch.long,
            device=DEVICE
        ).unsqueeze(0)

    # FP32 forward
    logits = model_a3(batch)

    loss = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
        batch[:, 1:].reshape(-1)
    )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_a3.parameters(),
        1.0
    )

    assert torch.isfinite(loss).item()
    assert torch.isfinite(grad_norm).item()

    optimizer_a3.step()

    losses.append(loss.item())

    print(
        f"Step {step + 1:2d}/10 | "
        f"Loss {loss.item():.4f} | "
        f"Grad {grad_norm.item():.3f}"
    )

    del logits, loss, batch

elapsed = time.time() - start_time

print("\n" + "=" * 60)
print("BENCHMARK RESULT")
print("=" * 60)
print(f"Initial loss : {losses[0]:.4f}")
print(f"Final loss   : {losses[-1]:.4f}")
print(f"Time         : {elapsed:.2f} sec")
print(
    f"Peak VRAM    : "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)
print("=" * 60)

gc.collect()
torch.cuda.empty_cache()

print("✅ A3 10-step FP32 benchmark complete")

A3 FP32 TRAINING BENCHMARK
Step  1/10 | Loss 330.9604 | Grad 81.241
Step  2/10 | Loss 300.0166 | Grad 122.564
Step  3/10 | Loss 259.2610 | Grad 164.631
Step  4/10 | Loss 213.5295 | Grad 172.893
Step  5/10 | Loss 171.4515 | Grad 148.802
Step  6/10 | Loss 137.5909 | Grad 114.848
Step  7/10 | Loss 113.1232 | Grad 87.823
Step  8/10 | Loss 97.2797 | Grad 99.420
Step  9/10 | Loss 84.7072 | Grad 81.822
Step 10/10 | Loss 75.8511 | Grad 58.663

BENCHMARK RESULT
Initial loss : 330.9604
Final loss   : 75.8511
Time         : 0.89 sec
Peak VRAM    : 1.31 GB
✅ A3 10-step FP32 benchmark complete


In [18]:
from huggingface_hub import list_repo_tree

REPO = "Exploration-Lab/CS-781-Capstone"

print("Phase 1 training repository structure:\n")

items = list_repo_tree(
    REPO,
    path_in_repo="Phase_1_Pretraining",
    repo_type="dataset",
    recursive=True
)

for item in items:
    print(item.path)

Phase 1 training repository structure:

Phase_1_Pretraining/Test
Phase_1_Pretraining/Train
Phase_1_Pretraining/Test/test.json
Phase_1_Pretraining/Train/train_acts.json
Phase_1_Pretraining/Train/train_high_courts.json
Phase_1_Pretraining/Train/train_supreme_court.json
Phase_1_Pretraining/Train/train_tribunals.json


In [26]:
# ============================================================
# A3 — VERIFY ALL 4 OFFICIAL TRAINING STREAMS
# ============================================================

from datasets import load_dataset

REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILES = {
    "Acts": "Phase_1_Pretraining/Train/train_acts.json",
    "High Courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "Supreme Court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

streams_a3 = {}

print("=" * 60)
print("OPENING OFFICIAL A3 TRAINING STREAMS")
print("=" * 60)

for name, path in SOURCE_FILES.items():

    ds = load_dataset(
        REPO,
        data_files=path,
        split="train",
        streaming=True,
        token=HF_TOKEN
    )

    streams_a3[name] = ds

    example = next(iter(ds))

    print(f"\n✓ {name}")
    print(f"  Path   : {path}")
    print(f"  Fields : {list(example.keys())}")

print("\n" + "=" * 60)
print("✅ ALL 4 OFFICIAL STREAMS VERIFIED")
print("=" * 60)

OPENING OFFICIAL A3 TRAINING STREAMS

✓ Acts
  Path   : Phase_1_Pretraining/Train/train_acts.json
  Fields : ['title', 'category', 'year', 'url', 'text', 'word_count', 'text_length_chars']


KeyboardInterrupt: 

In [29]:
# ============================================================
# A3 OPTIMIZED MODEL — DROPOUT + WEIGHT DECAY
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda"

VOCAB_SIZE = 50_000
CONTEXT_LEN = 2_048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2_048
PREFIX_RATIO = 0.5

DROPOUT = 0.1

class PrefixLMBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)

        self.attn = nn.MultiheadAttention(
            HIDDEN_SIZE,
            NUM_HEADS,
            dropout=DROPOUT,
            batch_first=True
        )

        self.attn_dropout = nn.Dropout(DROPOUT)

        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)

        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE),
            nn.Dropout(DROPOUT)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=attn_mask,
            need_weights=False
        )

        x = x + self.attn_dropout(attn_out)
        x = x + self.ffn(self.ln2(x))

        return x


class A3PrefixLM(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            CONTEXT_LEN,
            HIDDEN_SIZE
        )

        self.embedding_dropout = nn.Dropout(DROPOUT)

        self.layers = nn.ModuleList([
            PrefixLMBlock()
            for _ in range(NUM_LAYERS)
        ])

        self.ln_f = nn.LayerNorm(HIDDEN_SIZE)

        self.register_buffer(
            "position_ids",
            torch.arange(CONTEXT_LEN).unsqueeze(0),
            persistent=False
        )

    def build_mask(self, seq_len, device):
        prefix_len = int(
            seq_len * PREFIX_RATIO
        )

        mask = torch.full(
            (seq_len, seq_len),
            float("-inf"),
            device=device
        )

        mask[:prefix_len, :prefix_len] = 0

        for i in range(prefix_len, seq_len):
            mask[i, :prefix_len] = 0
            mask[i, prefix_len:i+1] = 0

        return mask

    def forward(self, input_ids):
        _, seq_len = input_ids.shape

        positions = self.position_ids[:, :seq_len]

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        x = self.embedding_dropout(x)

        mask = self.build_mask(
            seq_len,
            input_ids.device
        )

        for layer in self.layers:
            x = layer(x, mask)

        x = self.ln_f(x)

        return F.linear(
            x,
            self.token_embedding.weight
        )


model_a3 = A3PrefixLM().to(DEVICE)

total_params = sum(
    p.numel()
    for p in model_a3.parameters()
)

print("=" * 60)
print("A3 OPTIMIZED MODEL")
print("=" * 60)
print(f"Parameters : {total_params:,}")
print(f"Parameters : {total_params/1e6:.2f}M")
print(f"Dropout    : {DROPOUT}")
print(f"Weight decay: 0.1")
print(f"Prefix     : {PREFIX_RATIO*100:.0f}%")
print("=" * 60)

assert total_params == 51_868_672

print("✅ A3 optimized model verified")

AssertionError: Torch not compiled with CUDA enabled

In [28]:
# ============================================================
# A3 OPTIMIZED — FP32 TRAINING STABILITY TEST
# ============================================================

import gc
import torch
import torch.nn.functional as F

model_a3.train()

optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=1e-4,
    weight_decay=0.1,
    fused=True
)

optimizer_a3.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# ------------------------------------------------------------
# Forward — FP32
# ------------------------------------------------------------
logits = model_a3(seq)

loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
    seq[:, 1:].reshape(-1)
)

print(f"Loss: {loss.item():.4f}")

# ------------------------------------------------------------
# Backward — FP32
# ------------------------------------------------------------
loss.backward()

bad = []

for name, p in model_a3.named_parameters():
    if p.grad is not None:
        if not torch.isfinite(p.grad).all():
            bad.append(name)

print(f"Gradients finite: {len(bad) == 0}")

if bad:
    print("First bad gradients:")
    for name in bad[:10]:
        print(" ", name)

assert torch.isfinite(loss).item()
assert len(bad) == 0

grad_norm = torch.nn.utils.clip_grad_norm_(
    model_a3.parameters(),
    1.0
)

print(f"Gradient norm: {grad_norm.item():.4f}")

optimizer_a3.step()
optimizer_a3.zero_grad(set_to_none=True)

print(
    f"Peak VRAM: "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

del logits, loss
gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("✅ A3 OPTIMIZED TRAINING STEP PASSED")
print("=" * 60)

NameError: name 'model_a3' is not defined

In [18]:
# ============================================================
# CREATE FRESH OFFICIAL 512-TOKEN TRAINING SEQUENCE
# ============================================================

from datasets import load_dataset
import torch
import gc

TOKENIZER_PATH = "/kaggle/input/datasets/mathimohankowshik/tokenizer1/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

dataset = load_dataset(
    "Exploration-Lab/CS-781-Capstone",
    data_files="Phase_1_Pretraining/Train/train_acts.json",
    split="train",
    streaming=True
)

sample = next(iter(dataset))

# Find text field
if "text" in sample:
    text = sample["text"]
else:
    text = next(
        v for v in sample.values()
        if isinstance(v, str)
    )

ids = tokenizer.encode(text).ids

# Keep reading if one example is too short
while len(ids) < 512:
    sample = next(iter(dataset))
    
    if "text" in sample:
        text = sample["text"]
    else:
        text = next(
            v for v in sample.values()
            if isinstance(v, str)
        )
    
    ids.extend(tokenizer.encode(text).ids)

seq = torch.tensor(
    ids[:512],
    dtype=torch.long,
    device=DEVICE
).unsqueeze(0)

print("=" * 60)
print("OFFICIAL A3 TEST SEQUENCE")
print("=" * 60)
print("Shape :", tuple(seq.shape))
print("Tokens:", seq.numel())
print("=" * 60)

assert seq.shape == (1, 512)

print("✅ Fresh official 512-token sequence ready")

README.md:   0%|          | 0.00/16.1k [00:00<?, ?B/s]

AssertionError: Torch not compiled with CUDA enabled

In [19]:
# ============================================================
# AUTHENTICATE TO HUGGING FACE
# ============================================================

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

print("✅ Hugging Face authentication successful")

✅ Hugging Face authentication successful


In [20]:
# ============================================================
# CREATE FRESH OFFICIAL 512-TOKEN TRAINING SEQUENCE
# ============================================================

from tokenizers import Tokenizer
from datasets import load_dataset
import torch
import gc

TOKENIZER_PATH = "/kaggle/input/datasets/mathimohankowshik/tokenizer1/tokenizer.json"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

dataset = load_dataset(
    "Exploration-Lab/CS-781-Capstone",
    data_files="Phase_1_Pretraining/Train/train_acts.json",
    split="train",
    streaming=True
)

iterator = iter(dataset)

ids = []

while len(ids) < 512:

    sample = next(iterator)

    if "text" in sample:
        text = sample["text"]
    else:
        text = next(
            v for v in sample.values()
            if isinstance(v, str)
        )

    ids.extend(tokenizer.encode(text).ids)

seq = torch.tensor(
    ids[:512],
    dtype=torch.long,
    device=DEVICE
).unsqueeze(0)

print("=" * 60)
print("OFFICIAL A3 TEST SEQUENCE")
print("=" * 60)
print("Shape :", tuple(seq.shape))
print("Tokens:", seq.numel())
print("=" * 60)

assert seq.shape == (1, 512)

print("✅ Fresh official 512-token sequence ready")

AssertionError: Torch not compiled with CUDA enabled

In [22]:
# ============================================================
# A3 OPTIMIZED — FP32 TRAINING STABILITY TEST
# ============================================================

import torch
import torch.nn.functional as F
import gc

model_a3.train()

optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=1e-4,
    weight_decay=0.1,
    fused=True
)

optimizer_a3.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

logits = model_a3(seq)

loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
    seq[:, 1:].reshape(-1)
)

print(f"Loss: {loss.item():.4f}")

loss.backward()

bad = []

for name, p in model_a3.named_parameters():
    if p.grad is not None and not torch.isfinite(p.grad).all():
        bad.append(name)

print(f"Gradients finite: {len(bad) == 0}")

if bad:
    print("First bad gradients:")
    for name in bad[:10]:
        print(" ", name)

assert torch.isfinite(loss).item()
assert len(bad) == 0

grad_norm = torch.nn.utils.clip_grad_norm_(
    model_a3.parameters(),
    1.0
)

print(f"Gradient norm: {grad_norm.item():.4f}")

optimizer_a3.step()
optimizer_a3.zero_grad(set_to_none=True)

print(
    f"Peak VRAM: "
    f"{torch.cuda.max_memory_allocated()/1024**3:.2f} GB"
)

del logits, loss
gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("✅ A3 OPTIMIZED TRAINING STEP PASSED")
print("=" * 60)

NameError: name 'model_a3' is not defined

In [23]:
# ============================================================
# A3 OPTIMIZED — 10-STEP FP32 BENCHMARK
# ============================================================

import torch
import torch.nn.functional as F
import time
import gc

model_a3.train()

optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=1e-4,
    weight_decay=0.1,
    fused=True
)

optimizer_a3.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

losses = []

start_time = time.time()

for step in range(10):

    optimizer_a3.zero_grad(set_to_none=True)

    logits = model_a3(seq)

    loss = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
        seq[:, 1:].reshape(-1)
    )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_a3.parameters(),
        1.0
    )

    assert torch.isfinite(loss)
    assert torch.isfinite(grad_norm)

    optimizer_a3.step()

    losses.append(loss.item())

    print(
        f"Step {step+1:2d}/10 | "
        f"Loss {loss.item():.4f} | "
        f"Grad {grad_norm.item():.3f}"
    )

    del logits, loss

    gc.collect()
    torch.cuda.empty_cache()

elapsed = time.time() - start_time
peak_vram = torch.cuda.max_memory_allocated() / 1024**3

print()
print("=" * 60)
print("A3 OPTIMIZED BENCHMARK")
print("=" * 60)
print(f"Initial loss : {losses[0]:.4f}")
print(f"Final loss   : {losses[-1]:.4f}")
print(f"Time         : {elapsed:.2f} sec")
print(f"Peak VRAM    : {peak_vram:.2f} GB")
print("=" * 60)
print("✅ A3 10-step optimized benchmark complete")

NameError: name 'model_a3' is not defined

In [18]:
# ============================================================
# A3 PRODUCTION DATA LOADER
# ONE OFFICIAL SOURCE AT A TIME
# ============================================================

from datasets import load_dataset
from tokenizers import Tokenizer
import torch
import gc

TOKENIZER_PATH = "/kaggle/input/datasets/mathimohankowshik/tokenizer1/tokenizer.json"

SOURCE_FILES = {
    "Acts": "Phase_1_Pretraining/Train/train_acts.json",
    "High Courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "Supreme Court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

def token_stream(source_name):
    """
    Opens exactly ONE official source at a time.
    """

    path = SOURCE_FILES[source_name]

    print(f"Opening: {source_name}")

    dataset = load_dataset(
        "Exploration-Lab/CS-781-Capstone",
        data_files=path,
        split="train",
        streaming=True
    )

    try:
        for sample in dataset:

            if "text" in sample:
                text = sample["text"]
            else:
                text = next(
                    v for v in sample.values()
                    if isinstance(v, str)
                )

            ids = tokenizer.encode(text).ids

            for token in ids:
                yield token

    finally:
        del dataset
        gc.collect()


def make_training_sequence(source_name, seq_len=512):

    stream = token_stream(source_name)

    ids = []

    while len(ids) < seq_len:
        ids.append(next(stream))

    del stream

    return torch.tensor(
        ids,
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)


print("=" * 60)
print("A3 PRODUCTION LOADER READY")
print("=" * 60)
print("Sources:")
for name in SOURCE_FILES:
    print(" -", name)

print()
print("Memory strategy:")
print(" - One source at a time")
print(" - Streaming dataset")
print(" - No simultaneous 4-source loading")
print("=" * 60)
print("✅ Production loader defined")

A3 PRODUCTION LOADER READY
Sources:
 - Acts
 - High Courts
 - Supreme Court
 - Tribunals

Memory strategy:
 - One source at a time
 - Streaming dataset
 - No simultaneous 4-source loading
✅ Production loader defined


In [17]:
# ============================================================
# A3 PRODUCTION LOADER — 4-SOURCE SEQUENTIAL TEST
# ============================================================

import time
import torch
import gc

for source_name in SOURCE_FILES:

    start = time.time()

    test_seq = make_training_sequence(
        source_name,
        seq_len=512
    )

    elapsed = time.time() - start

    print(
        f"{source_name:15s} | "
        f"shape={tuple(test_seq.shape)} | "
        f"tokens={test_seq.numel()} | "
        f"time={elapsed:.2f}s"
    )

    assert test_seq.shape == (1, 512)
    assert test_seq.numel() == 512

    del test_seq
    gc.collect()
    torch.cuda.empty_cache()

print()
print("=" * 60)
print("✅ ALL 4 OFFICIAL SOURCES PASSED SEQUENTIAL TEST")
print("=" * 60)

NameError: name 'make_training_sequence' is not defined

In [10]:
# ============================================================
# A3 RECOVERY AFTER KERNEL RESTART
# ============================================================

import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset

DEVICE = "cuda"

VOCAB_SIZE = 50_000
CONTEXT_LEN = 2_048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2_048
PREFIX_RATIO = 0.5
DROPOUT = 0.1

# ------------------------------------------------------------
# Hugging Face authentication
# ------------------------------------------------------------
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------
TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mathimohankowshik/tokenizer1/tokenizer.json"
)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# ------------------------------------------------------------
# A3 model
# ------------------------------------------------------------
class PrefixLMBlock(nn.Module):

    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)

        self.attn = nn.MultiheadAttention(
            HIDDEN_SIZE,
            NUM_HEADS,
            dropout=DROPOUT,
            batch_first=True
        )

        self.attn_dropout = nn.Dropout(DROPOUT)

        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)

        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE),
            nn.Dropout(DROPOUT)
        )

    def forward(self, x, attn_mask):

        h = self.ln1(x)

        attn_out, _ = self.attn(
            h, h, h,
            attn_mask=attn_mask,
            need_weights=False
        )

        x = x + self.attn_dropout(attn_out)
        x = x + self.ffn(self.ln2(x))

        return x


class A3PrefixLM(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            CONTEXT_LEN,
            HIDDEN_SIZE
        )

        self.embedding_dropout = nn.Dropout(DROPOUT)

        self.layers = nn.ModuleList([
            PrefixLMBlock()
            for _ in range(NUM_LAYERS)
        ])

        self.ln_f = nn.LayerNorm(HIDDEN_SIZE)

        self.register_buffer(
            "position_ids",
            torch.arange(CONTEXT_LEN).unsqueeze(0),
            persistent=False
        )

    def build_mask(self, seq_len, device):

        prefix_len = int(
            seq_len * PREFIX_RATIO
        )

        mask = torch.full(
            (seq_len, seq_len),
            float("-inf"),
            device=device
        )

        mask[:prefix_len, :prefix_len] = 0

        for i in range(prefix_len, seq_len):
            mask[i, :prefix_len] = 0
            mask[i, prefix_len:i+1] = 0

        return mask

    def forward(self, input_ids):

        _, seq_len = input_ids.shape

        positions = self.position_ids[:, :seq_len]

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        x = self.embedding_dropout(x)

        mask = self.build_mask(
            seq_len,
            input_ids.device
        )

        for layer in self.layers:
            x = layer(x, mask)

        x = self.ln_f(x)

        return F.linear(
            x,
            self.token_embedding.weight
        )


model_a3 = A3PrefixLM().to(DEVICE)

total_params = sum(
    p.numel()
    for p in model_a3.parameters()
)

print("=" * 60)
print("A3 RECOVERED")
print("=" * 60)
print(f"Parameters : {total_params:,}")
print(f"Parameters : {total_params/1e6:.2f}M")
print(f"Device     : {DEVICE}")
print(f"GPU        : {torch.cuda.get_device_name(0)}")
print("=" * 60)

assert total_params == 51_868_672

print("✅ A3 model recovered")

AssertionError: Torch not compiled with CUDA enabled

In [11]:
# ============================================================
# A3 — LOW-MEMORY DIRECT HF STREAMING READER
# ============================================================

import requests
import json
import gc
from huggingface_hub import hf_hub_url

REPO_ID = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILES = {
    "Acts": "Phase_1_Pretraining/Train/train_acts.json",
    "High Courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "Supreme Court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

def stream_json_objects(source_name):

    filename = SOURCE_FILES[source_name]

    url = hf_hub_url(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset"
    )

    headers = {
        "Authorization": f"Bearer {HF_TOKEN}"
    }

    response = requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=120
    )

    response.raise_for_status()

    # Read a small amount only to determine format
    first_chunk = response.raw.read(4096)

    first_nonspace = first_chunk.lstrip()[:1]

    # --------------------------------------------------------
    # JSON Lines
    # --------------------------------------------------------
    if first_nonspace != b"[":

        buffer = first_chunk

        while True:

            lines = buffer.split(b"\n")

            buffer = lines.pop()

            for line in lines:

                line = line.strip()

                if line:
                    yield json.loads(line)

            chunk = response.raw.read(1024 * 1024)

            if not chunk:
                break

            buffer += chunk

        if buffer.strip():
            yield json.loads(buffer)

    # --------------------------------------------------------
    # JSON array — incremental parser
    # --------------------------------------------------------
    else:

        decoder = json.JSONDecoder()
        buffer = first_chunk.decode("utf-8")
        position = 0
        started = False

        while True:

            if not started:

                while position < len(buffer) and buffer[position].isspace():
                    position += 1

                if position < len(buffer) and buffer[position] == "[":
                    position += 1
                    started = True

            while True:

                while position < len(buffer) and (
                    buffer[position].isspace()
                    or buffer[position] == ","
                ):
                    position += 1

                if position < len(buffer) and buffer[position] == "]":
                    response.close()
                    return

                if position >= len(buffer):
                    break

                try:
                    obj, end = decoder.raw_decode(
                        buffer,
                        position
                    )

                    position = end
                    yield obj

                except json.JSONDecodeError:
                    break

            chunk = response.raw.read(1024 * 1024)

            if not chunk:
                break

            buffer = buffer[position:] + chunk.decode(
                "utf-8",
                errors="ignore"
            )

            position = 0

    response.close()


def token_stream(source_name):

    for sample in stream_json_objects(source_name):

        if "text" in sample:
            text = sample["text"]
        else:
            text = next(
                v for v in sample.values()
                if isinstance(v, str)
            )

        if not text:
            continue

        for token in tokenizer.encode(text).ids:
            yield token


print("=" * 60)
print("LOW-MEMORY DIRECT STREAM READER")
print("=" * 60)
print("Sources available:")
for name in SOURCE_FILES:
    print(" -", name)
print()
print("Uses:")
print(" - Direct Hugging Face HTTP streaming")
print(" - No datasets Dataset object")
print(" - No full-file download")
print(" - One source opened at a time")
print("=" * 60)
print("✅ Reader defined")

LOW-MEMORY DIRECT STREAM READER
Sources available:
 - Acts
 - High Courts
 - Supreme Court
 - Tribunals

Uses:
 - Direct Hugging Face HTTP streaming
 - No datasets Dataset object
 - No full-file download
 - One source opened at a time
✅ Reader defined


In [12]:
# ============================================================
# A3 — TEST DIRECT STREAM WITH ACTS ONLY
# ============================================================

import time
import torch
import gc

start = time.time()

stream = token_stream("Acts")

ids = []

while len(ids) < 512:
    ids.append(next(stream))

del stream
gc.collect()

test_seq = torch.tensor(
    ids,
    dtype=torch.long,
    device=DEVICE
).unsqueeze(0)

elapsed = time.time() - start

print("=" * 60)
print("DIRECT STREAM TEST")
print("=" * 60)
print(f"Source : Acts")
print(f"Shape  : {tuple(test_seq.shape)}")
print(f"Tokens : {test_seq.numel()}")
print(f"Time   : {elapsed:.2f}s")
print("=" * 60)

assert test_seq.shape == (1, 512)
assert test_seq.numel() == 512

print("✅ Direct Acts streaming works")

AssertionError: Torch not compiled with CUDA enabled

In [13]:
# ============================================================
# A3 — FOUR OFFICIAL SOURCES, LOW-MEMORY TEST
# ============================================================

import time
import gc

for source_name in SOURCE_FILES:

    start = time.time()

    stream = token_stream(source_name)

    ids = []

    while len(ids) < 512:
        ids.append(next(stream))

    del stream
    gc.collect()

    elapsed = time.time() - start

    print(
        f"{source_name:15s} | "
        f"tokens={len(ids):4d} | "
        f"time={elapsed:.2f}s"
    )

print()
print("=" * 60)
print("✅ ALL FOUR OFFICIAL SOURCES PASSED")
print("=" * 60)

Acts            | tokens= 512 | time=1.77s
High Courts     | tokens= 512 | time=2.53s
Supreme Court   | tokens= 512 | time=1.78s
Tribunals       | tokens= 512 | time=1.83s

✅ ALL FOUR OFFICIAL SOURCES PASSED


In [16]:
# ============================================================
# A3 PRODUCTION TRAINING SETUP
# ONE SOURCE AT A TIME
# ============================================================

import os
import time
import gc
import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

TRAIN_BLOCK = 512
TOTAL_TOKENS = 200_000_000

LR = 1e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

CHECKPOINT_EVERY = 5_000

# Equal source rotation
SOURCE_NAMES = [
    "Acts",
    "High Courts",
    "Supreme Court",
    "Tribunals"
]

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    fused=True
)

# ------------------------------------------------------------
# Checkpoint directory
# ------------------------------------------------------------

CHECKPOINT_DIR = "/kaggle/working/a3_checkpoints"

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# Persistent source iterator
# ------------------------------------------------------------

current_source_index = 0
current_source_name = SOURCE_NAMES[0]

source_stream = token_stream(
    current_source_name
)

# ------------------------------------------------------------
# Get one training block
# ------------------------------------------------------------

def get_next_training_block():

    global current_source_index
    global current_source_name
    global source_stream

    ids = []

    while len(ids) < TRAIN_BLOCK:

        try:
            ids.append(next(source_stream))

        except StopIteration:

            # Close finished stream
            try:
                source_stream.close()
            except:
                pass

            gc.collect()

            # Move to next official source
            current_source_index = (
                current_source_index + 1
            ) % len(SOURCE_NAMES)

            current_source_name = (
                SOURCE_NAMES[current_source_index]
            )

            source_stream = token_stream(
                current_source_name
            )

    return torch.tensor(
        ids,
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)


# ------------------------------------------------------------
# Configuration summary
# ------------------------------------------------------------

print("=" * 60)
print("A3 PRODUCTION TRAINING SETUP")
print("=" * 60)

print(f"Parameters     : 51.87M")
print(f"Training block : {TRAIN_BLOCK}")
print(f"Target tokens  : {TOTAL_TOKENS:,}")
print(f"Learning rate  : {LR}")
print(f"Weight decay   : {WEIGHT_DECAY}")
print(f"Grad clipping  : {GRAD_CLIP}")
print(f"Checkpoint     : every {CHECKPOINT_EVERY:,} steps")
print()
print("Source rotation:")
for s in SOURCE_NAMES:
    print(" -", s)

print()
print("Memory strategy:")
print(" - One HTTP source stream at a time")
print(" - FP32 training")
print(" - No torch.compile")
print(" - No simultaneous source datasets")
print("=" * 60)

print("✅ A3 production training setup ready")

NameError: name 'model_a3' is not defined

In [2]:
# ============================================================
# A3 PRODUCTION SMOKE TEST — 20 REAL TRAINING STEPS
# ============================================================

import time
import gc
import torch
import torch.nn.functional as F

model_a3.train()

optimizer_a3.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start_time = time.time()

losses = []
tokens_seen = 0

for step in range(20):

    # --------------------------------------------------------
    # Get REAL official data
    # --------------------------------------------------------
    batch = get_next_training_block()

    # --------------------------------------------------------
    # Forward — FP32
    # --------------------------------------------------------
    logits = model_a3(batch)

    loss = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
        batch[:, 1:].reshape(-1)
    )

    # --------------------------------------------------------
    # Backward
    # --------------------------------------------------------
    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_a3.parameters(),
        GRAD_CLIP
    )

    assert torch.isfinite(loss)
    assert torch.isfinite(grad_norm)

    optimizer_a3.step()
    optimizer_a3.zero_grad(set_to_none=True)

    tokens_seen += batch.numel()
    losses.append(loss.item())

    if (step + 1) % 5 == 0:
        print(
            f"Step {step+1:2d}/20 | "
            f"Source {current_source_name:15s} | "
            f"Loss {loss.item():.4f} | "
            f"Grad {grad_norm.item():.3f}"
        )

    del batch, logits, loss

    gc.collect()
    torch.cuda.empty_cache()

elapsed = time.time() - start_time
peak_vram = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print()
print("=" * 60)
print("A3 REAL-DATA SMOKE TEST")
print("=" * 60)
print(f"Tokens seen : {tokens_seen:,}")
print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss  : {losses[-1]:.4f}")
print(f"Time        : {elapsed:.2f}s")
print(f"Peak VRAM   : {peak_vram:.2f} GB")
print("=" * 60)

print("✅ 20-step real-data smoke test passed")

NameError: name 'model_a3' is not defined

In [1]:
# ============================================================
# A3 — PRODUCTION TRAINING
# TARGET: 200 MILLION TOKENS
# ============================================================

import os
import time
import gc
import torch
import torch.nn.functional as F

# ------------------------------------------------------------
# Production configuration
# ------------------------------------------------------------

TARGET_TOKENS = 200_000_000
START_TOKENS = 10_240

CHECKPOINT_EVERY_STEPS = 5_000
PRINT_EVERY_STEPS = 100

model_a3.train()

optimizer_a3.zero_grad(set_to_none=True)

tokens_seen = START_TOKENS
step = 20

start_time = time.time()

last_checkpoint = time.time()

print("=" * 70)
print("A3 PRODUCTION TRAINING STARTED")
print("=" * 70)
print(f"Target tokens : {TARGET_TOKENS:,}")
print(f"Already seen  : {tokens_seen:,}")
print(f"Remaining     : {TARGET_TOKENS - tokens_seen:,}")
print(f"Block size    : {TRAIN_BLOCK}")
print(f"Learning rate : {LR}")
print(f"Weight decay  : {WEIGHT_DECAY}")
print("=" * 70)

while tokens_seen < TARGET_TOKENS:

    # --------------------------------------------------------
    # Get official training block
    # --------------------------------------------------------

    batch = get_next_training_block()

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    logits = model_a3(batch)

    loss = F.cross_entropy(
        logits[:, :-1, :].reshape(-1, VOCAB_SIZE),
        batch[:, 1:].reshape(-1)
    )

    # --------------------------------------------------------
    # Backward
    # --------------------------------------------------------

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_a3.parameters(),
        GRAD_CLIP
    )

    # Safety checks
    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite loss at step {step}"
        )

    if not torch.isfinite(grad_norm):
        raise RuntimeError(
            f"Non-finite gradient at step {step}"
        )

    optimizer_a3.step()
    optimizer_a3.zero_grad(set_to_none=True)

    # --------------------------------------------------------
    # Counters
    # --------------------------------------------------------

    tokens_seen += batch.numel()
    step += 1

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if step % PRINT_EVERY_STEPS == 0:

        elapsed = time.time() - start_time

        tok_per_sec = (
            (tokens_seen - START_TOKENS)
            / max(elapsed, 1e-6)
        )

        progress = (
            tokens_seen / TARGET_TOKENS
        ) * 100

        eta_seconds = (
            TARGET_TOKENS - tokens_seen
        ) / max(tok_per_sec, 1e-6)

        print(
            f"Step {step:7,d} | "
            f"Tokens {tokens_seen:12,d} | "
            f"{progress:6.2f}% | "
            f"Loss {loss.item():8.4f} | "
            f"Grad {grad_norm.item():7.3f} | "
            f"{tok_per_sec:,.0f} tok/s | "
            f"ETA {eta_seconds/3600:.2f}h"
        )

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    if step % CHECKPOINT_EVERY_STEPS == 0:

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"step_{step:07d}.pt"
        )

        torch.save(
            {
                "model": model_a3.state_dict(),
                "step": step,
                "tokens_seen": tokens_seen,
                "architecture": "A3_prefix_lm_hybrid",
                "parameters": 51_868_672,
                "context_length": CONTEXT_LEN,
                "hidden_size": HIDDEN_SIZE,
                "num_layers": NUM_LAYERS,
                "num_heads": NUM_HEADS,
                "ffn_size": FFN_SIZE,
                "prefix_ratio": PREFIX_RATIO,
                "dropout": DROPOUT,
                "learning_rate": LR,
                "weight_decay": WEIGHT_DECAY,
            },
            checkpoint_path
        )

        print(
            f"💾 Checkpoint saved: "
            f"{checkpoint_path}"
        )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del batch
    del logits
    del loss

    if step % 20 == 0:
        gc.collect()
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# Final checkpoint
# ------------------------------------------------------------

final_path = os.path.join(
    CHECKPOINT_DIR,
    "a3_final.pt"
)

torch.save(
    {
        "model": model_a3.state_dict(),
        "step": step,
        "tokens_seen": tokens_seen,
        "architecture": "A3_prefix_lm_hybrid",
        "parameters": 51_868_672,
        "context_length": CONTEXT_LEN,
        "hidden_size": HIDDEN_SIZE,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "ffn_size": FFN_SIZE,
        "prefix_ratio": PREFIX_RATIO,
        "dropout": DROPOUT,
        "learning_rate": LR,
        "weight_decay": WEIGHT_DECAY,
    },
    final_path
)

elapsed = time.time() - start_time

print()
print("=" * 70)
print("A3 PRODUCTION TRAINING COMPLETE")
print("=" * 70)
print(f"Final step   : {step:,}")
print(f"Tokens seen  : {tokens_seen:,}")
print(f"Progress     : {tokens_seen/TARGET_TOKENS*100:.2f}%")
print(f"Time         : {elapsed/3600:.2f} hours")
print(f"Final model  : {final_path}")
print("=" * 70)
print("✅ A3 TRAINING COMPLETE")

NameError: name 'model_a3' is not defined

In [8]:
# ============================================================
# A3 RECOVERY — RESUME FROM STEP 5000
# ============================================================

import os
import gc
import time
import json
import requests
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_url

# ------------------------------------------------------------
# 1. Checkpoint
# ------------------------------------------------------------
CHECKPOINT_PATH = "/kaggle/working/a3_checkpoints/step_0005000.pt"

assert os.path.exists(CHECKPOINT_PATH), f"Checkpoint not found: {CHECKPOINT_PATH}"

# ------------------------------------------------------------
# 2. Restore model + optimizer
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cuda",
    weights_only=False
)

model_a3.load_state_dict(checkpoint["model_state_dict"])
optimizer_a3.load_state_dict(checkpoint["optimizer_state_dict"])

model_a3.cuda()
model_a3.train()

START_STEP = checkpoint.get("step", 5000)
START_TOKENS = checkpoint.get("tokens_seen", 2_560_000)

print("=" * 70)
print("A3 RECOVERY")
print("=" * 70)
print(f"Checkpoint : step {START_STEP}")
print(f"Tokens     : {START_TOKENS:,}")
print(f"Progress   : {START_TOKENS / TOTAL_TOKENS * 100:.2f}%")
print("=" * 70)

# ------------------------------------------------------------
# 3. Robust HTTP streaming
# ------------------------------------------------------------
session = requests.Session()

def open_source_with_retry(source_name, max_retries=10):
    url = hf_hub_url(
        repo_id=REPO_ID,
        filename=SOURCE_FILES[source_name],
        repo_type="dataset"
    )

    headers = {}
    if "HF_TOKEN" in os.environ:
        headers["Authorization"] = f"Bearer {os.environ['HF_TOKEN']}"

    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 300)
            )
            response.raise_for_status()
            print(f"Connected to {source_name}")
            return response

        except Exception as e:
            print(f"Connection attempt {attempt}/{max_retries} failed: {e}")
            if attempt == max_retries:
                raise
            time.sleep(min(2 ** attempt, 30))

# ------------------------------------------------------------
# 4. Recreate stream
# ------------------------------------------------------------
# Start from the beginning of the current source.
# This is safe because we only need training blocks;
# it does NOT affect model checkpoint restoration.

current_source_name = "Acts"
source_stream = None

def restart_stream():
    global source_stream
    if source_stream is not None:
        try:
            source_stream.close()
        except:
            pass

    gc.collect()
    source_stream = token_stream(current_source_name)

restart_stream()

# ------------------------------------------------------------
# 5. Resume training
# ------------------------------------------------------------
step = START_STEP
tokens_seen = START_TOKENS

last_time = time.time()

print("\nResuming A3 training...")
print("Network retry protection enabled.")
print()

while tokens_seen < TOTAL_TOKENS:

    try:
        batch = get_next_training_block()

    except (requests.exceptions.RequestException,
            requests.exceptions.ConnectionError,
            requests.exceptions.ChunkedEncodingError,
            Exception) as e:

        # Only retry network/stream failures.
        # If the exception is not a network issue, re-raise it.
        msg = str(e).lower()

        network_error = any(x in msg for x in [
            "incompleteread",
            "connection",
            "protocolerror",
            "chunked",
            "broken pipe",
            "reset by peer",
            "read timed out",
            "connection aborted"
        ])

        if not network_error:
            raise

        print("\n⚠️ Network interruption detected.")
        print("Reconnecting to Hugging Face...")
        time.sleep(3)

        restart_stream()
        continue

    input_ids = torch.tensor(
        batch,
        dtype=torch.long,
        device="cuda"
    ).unsqueeze(0)

    optimizer_a3.zero_grad(set_to_none=True)

    logits = model_a3(input_ids)

    loss = F.cross_entropy(
        logits[:, :-1].reshape(-1, VOCAB_SIZE),
        input_ids[:, 1:].reshape(-1)
    )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_a3.parameters(),
        GRAD_CLIP
    )

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}")

    optimizer_a3.step()

    step += 1
    tokens_seen += TRAIN_BLOCK

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------
    if step % PRINT_EVERY_STEPS == 0:
        elapsed = time.time() - last_time
        tok_per_sec = (PRINT_EVERY_STEPS * TRAIN_BLOCK) / elapsed

        remaining_tokens = TOTAL_TOKENS - tokens_seen
        eta_hours = remaining_tokens / tok_per_sec / 3600

        progress = tokens_seen / TOTAL_TOKENS * 100

        print(
            f"Step {step:7d} | "
            f"Tokens {tokens_seen:11,d} | "
            f"{progress:6.2f}% | "
            f"Loss {loss.item():8.4f} | "
            f"Grad {float(grad_norm):7.3f} | "
            f"{tok_per_sec:,.0f} tok/s | "
            f"ETA {eta_hours:.2f}h"
        )

        last_time = time.time()

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------
    if step % CHECKPOINT_EVERY == 0:
        checkpoint_path = (
            f"/kaggle/working/a3_checkpoints/"
            f"step_{step:07d}.pt"
        )

        torch.save({
            "step": step,
            "tokens_seen": tokens_seen,
            "model_state_dict": model_a3.state_dict(),
            "optimizer_state_dict": optimizer_a3.state_dict(),
        }, checkpoint_path)

        print(f"💾 Checkpoint saved: {checkpoint_path}")

# ------------------------------------------------------------
# 6. Final checkpoint
# ------------------------------------------------------------
FINAL_PATH = "/kaggle/working/a3_checkpoints/a3_final.pt"

torch.save({
    "step": step,
    "tokens_seen": tokens_seen,
    "model_state_dict": model_a3.state_dict(),
    "optimizer_state_dict": optimizer_a3.state_dict(),
}, FINAL_PATH)

print("\n" + "=" * 70)
print("🎉 A3 TRAINING COMPLETE")
print("=" * 70)
print(f"Steps  : {step:,}")
print(f"Tokens : {tokens_seen:,}")
print(f"Model  : {FINAL_PATH}")
print("=" * 70)

KeyError: 'model_state_dict'

In [9]:
import torch

ckpt = torch.load(
    "/kaggle/working/a3_checkpoints/step_0005000.pt",
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(ckpt))

if isinstance(ckpt, dict):
    print("\nCheckpoint keys:")
    for k in ckpt.keys():
        print(" ", k)

print("\nCheckpoint summary:")
print(ckpt if not isinstance(ckpt, dict) else {
    k: (type(v).__name__ if not isinstance(v, (int, float, str)) else v)
    for k, v in ckpt.items()
})

Checkpoint type: <class 'dict'>

Checkpoint keys:
  model
  step
  tokens_seen
  architecture
  parameters
  context_length
  hidden_size
  num_layers
  num_heads
  ffn_size
  prefix_ratio
  dropout
  learning_rate
  weight_decay

Checkpoint summary:
{'model': 'OrderedDict', 'step': 5000, 'tokens_seen': 2560000, 'architecture': 'A3_prefix_lm_hybrid', 'parameters': 51868672, 'context_length': 2048, 'hidden_size': 512, 'num_layers': 8, 'num_heads': 8, 'ffn_size': 2048, 'prefix_ratio': 0.5, 'dropout': 0.1, 'learning_rate': 0.0001, 'weight_decay': 0.1}


In [10]:
# ============================================================
# A3 RECOVERY — CORRECT CHECKPOINT FORMAT
# ============================================================

import os
import gc
import time
import requests
import torch
import torch.nn.functional as F

CHECKPOINT_PATH = "/kaggle/working/a3_checkpoints/step_0005000.pt"

# ------------------------------------------------------------
# 1. Load checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cuda",
    weights_only=False
)

# Restore model weights
model_a3.load_state_dict(checkpoint["model"])

model_a3.cuda()
model_a3.train()

# ------------------------------------------------------------
# 2. Restore progress
# ------------------------------------------------------------
step = checkpoint["step"]
tokens_seen = checkpoint["tokens_seen"]

print("=" * 70)
print("A3 RECOVERY")
print("=" * 70)
print(f"Architecture : {checkpoint['architecture']}")
print(f"Parameters   : {checkpoint['parameters']:,}")
print(f"Checkpoint   : Step {step:,}")
print(f"Tokens       : {tokens_seen:,}")
print(f"Progress     : {tokens_seen / TOTAL_TOKENS * 100:.2f}%")
print("=" * 70)

# ------------------------------------------------------------
# 3. Fresh optimizer
# ------------------------------------------------------------
optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=checkpoint["learning_rate"],
    weight_decay=checkpoint["weight_decay"],
    fused=True
)

# ------------------------------------------------------------
# 4. Robust HTTP session
# ------------------------------------------------------------
session = requests.Session()

# ------------------------------------------------------------
# 5. Source stream setup
# ------------------------------------------------------------
current_source_name = "Acts"
source_stream = None

def restart_source_stream():
    global source_stream

    if source_stream is not None:
        try:
            source_stream.close()
        except:
            pass

    gc.collect()

    source_stream = token_stream(current_source_name)


restart_source_stream()

print("HTTP retry protection enabled.")
print("Starting from Step 5,000...\n")

# ------------------------------------------------------------
# 6. Training
# ------------------------------------------------------------
last_time = time.time()
steps_since_print = 0

while tokens_seen < TOTAL_TOKENS:

    # --------------------------------------------------------
    # Get training block
    # --------------------------------------------------------
    try:
        batch = get_next_training_block()

    except Exception as e:

        error_text = str(e).lower()

        network_error = any(term in error_text for term in [
            "incompleteread",
            "protocolerror",
            "connection",
            "chunked",
            "broken pipe",
            "reset by peer",
            "read timed out",
            "connection aborted"
        ])

        if not network_error:
            raise

        print("\n⚠️ NETWORK INTERRUPTION")
        print("Reconnecting to current data source...")
        print(f"Reason: {type(e).__name__}")

        time.sleep(5)

        restart_source_stream()

        continue

    # --------------------------------------------------------
    # Tensor
    # --------------------------------------------------------
    input_ids = torch.tensor(
        batch,
        dtype=torch.long,
        device="cuda"
    ).unsqueeze(0)

    # --------------------------------------------------------
    # Forward + backward
    # --------------------------------------------------------
    optimizer_a3.zero_grad(set_to_none=True)

    logits = model_a3(input_ids)

    loss = F.cross_entropy(
        logits[:, :-1].reshape(-1, VOCAB_SIZE),
        input_ids[:, 1:].reshape(-1)
    )

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite loss at step {step}"
        )

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_a3.parameters(),
        GRAD_CLIP
    )

    optimizer_a3.step()

    step += 1
    tokens_seen += TRAIN_BLOCK

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------
    if step % PRINT_EVERY_STEPS == 0:

        elapsed = time.time() - last_time

        tok_per_sec = (
            PRINT_EVERY_STEPS * TRAIN_BLOCK
        ) / elapsed

        remaining_tokens = TOTAL_TOKENS - tokens_seen

        eta_hours = (
            remaining_tokens /
            tok_per_sec /
            3600
        )

        progress = (
            tokens_seen /
            TOTAL_TOKENS *
            100
        )

        print(
            f"Step {step:7d} | "
            f"Tokens {tokens_seen:11,d} | "
            f"{progress:6.2f}% | "
            f"Loss {loss.item():8.4f} | "
            f"Grad {float(grad_norm):7.3f} | "
            f"{tok_per_sec:,.0f} tok/s | "
            f"ETA {eta_hours:.2f}h"
        )

        last_time = time.time()

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------
    if step % CHECKPOINT_EVERY == 0:

        checkpoint_path = (
            f"/kaggle/working/a3_checkpoints/"
            f"step_{step:07d}.pt"
        )

        torch.save({
            "model": model_a3.state_dict(),
            "step": step,
            "tokens_seen": tokens_seen,
            "architecture": "A3_prefix_lm_hybrid",
            "parameters": sum(
                p.numel() for p in model_a3.parameters()
            ),
            "context_length": CONTEXT_LEN,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "num_heads": NUM_HEADS,
            "ffn_size": FFN_SIZE,
            "prefix_ratio": PREFIX_RATIO,
            "dropout": 0.1,
            "learning_rate": LR,
            "weight_decay": WEIGHT_DECAY,
        }, checkpoint_path)

        print(
            f"💾 Checkpoint saved: {checkpoint_path}"
        )

# ------------------------------------------------------------
# 7. Final model
# ------------------------------------------------------------
FINAL_PATH = "/kaggle/working/a3_checkpoints/a3_final.pt"

torch.save({
    "model": model_a3.state_dict(),
    "step": step,
    "tokens_seen": tokens_seen,
    "architecture": "A3_prefix_lm_hybrid",
    "parameters": sum(
        p.numel() for p in model_a3.parameters()
    ),
    "context_length": CONTEXT_LEN,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "num_heads": NUM_HEADS,
    "ffn_size": FFN_SIZE,
    "prefix_ratio": PREFIX_RATIO,
    "dropout": 0.1,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
}, FINAL_PATH)

print("\n" + "=" * 70)
print("🎉 A3 TRAINING COMPLETE")
print("=" * 70)
print(f"Steps  : {step:,}")
print(f"Tokens : {tokens_seen:,}")
print(f"Model  : {FINAL_PATH}")
print("=" * 70)

A3 RECOVERY
Architecture : A3_prefix_lm_hybrid
Parameters   : 51,868,672
Checkpoint   : Step 5,000
Tokens       : 2,560,000
Progress     : 1.28%
HTTP retry protection enabled.
Starting from Step 5,000...



/tmp/ipykernel_297/839914224.py:130: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.tensor(


ValueError: too many values to unpack (expected 2)

In [11]:
print("batch type:", type(batch))
print("batch shape:", getattr(batch, "shape", None))
print("input_ids shape if tensor:", getattr(input_ids, "shape", None))

batch type: <class 'torch.Tensor'>
batch shape: torch.Size([1, 512])
input_ids shape if tensor: torch.Size([1, 1, 512])


In [12]:
# ============================================================
# A3 RECOVERY — FINAL SHAPE-CORRECTED VERSION
# ============================================================

import os
import gc
import time
import requests
import torch
import torch.nn.functional as F

CHECKPOINT_PATH = "/kaggle/working/a3_checkpoints/step_0005000.pt"

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cuda",
    weights_only=False
)

model_a3.load_state_dict(checkpoint["model"])
model_a3.cuda()
model_a3.train()

step = checkpoint["step"]
tokens_seen = checkpoint["tokens_seen"]

print("=" * 70)
print("A3 RECOVERY")
print("=" * 70)
print(f"Checkpoint : Step {step:,}")
print(f"Tokens     : {tokens_seen:,}")
print(f"Progress   : {tokens_seen / TOTAL_TOKENS * 100:.2f}%")
print(f"Parameters : {checkpoint['parameters']:,}")
print("=" * 70)

# ------------------------------------------------------------
# Fresh optimizer
# ------------------------------------------------------------
optimizer_a3 = torch.optim.AdamW(
    model_a3.parameters(),
    lr=checkpoint["learning_rate"],
    weight_decay=checkpoint["weight_decay"],
    fused=True
)

# ------------------------------------------------------------
# Network retry setup
# ------------------------------------------------------------
session = requests.Session()

current_source_name = "Acts"
source_stream = None

def restart_source_stream():
    global source_stream

    if source_stream is not None:
        try:
            source_stream.close()
        except:
            pass

    gc.collect()
    source_stream = token_stream(current_source_name)

restart_source_stream()

print("HTTP retry protection enabled.")
print("Starting from Step 5,000...\n")

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
last_time = time.time()

while tokens_seen < TOTAL_TOKENS:

    # Get block
    try:
        batch = get_next_training_block()

    except Exception as e:

        error_text = str(e).lower()

        network_error = any(term in error_text for term in [
            "incompleteread",
            "protocolerror",
            "connection",
            "chunked",
            "broken pipe",
            "reset by peer",
            "read timed out",
            "connection aborted"
        ])

        if not network_error:
            raise

        print("\n⚠️ NETWORK INTERRUPTION")
        print("Reconnecting...")
        print(type(e).__name__)

        time.sleep(5)
        restart_source_stream()
        continue

    # --------------------------------------------------------
    # IMPORTANT:
    # get_next_training_block() ALREADY returns [1, 512]
    # --------------------------------------------------------
    input_ids = batch.to(
        device="cuda",
        dtype=torch.long
    )

    # Safety check
    if input_ids.ndim != 2:
        raise RuntimeError(
            f"Unexpected input shape: {input_ids.shape}"
        )

    optimizer_a3.zero_grad(set_to_none=True)

    # Forward
    logits = model_a3(input_ids)

    # Loss
    loss = F.cross_entropy(
        logits[:, :-1].reshape(-1, VOCAB_SIZE),
        input_ids[:, 1:].reshape(-1)
    )

    if not torch.isfinite(loss):
        raise RuntimeError(
            f"Non-finite loss at step {step}"
        )

    # Backward
    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model_a3.parameters(),
        GRAD_CLIP
    )

    optimizer_a3.step()

    step += 1
    tokens_seen += TRAIN_BLOCK

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------
    if step % PRINT_EVERY_STEPS == 0:

        elapsed = time.time() - last_time

        tok_per_sec = (
            PRINT_EVERY_STEPS * TRAIN_BLOCK
        ) / elapsed

        remaining_tokens = TOTAL_TOKENS - tokens_seen

        eta_hours = (
            remaining_tokens /
            tok_per_sec /
            3600
        )

        progress = (
            tokens_seen /
            TOTAL_TOKENS *
            100
        )

        print(
            f"Step {step:7d} | "
            f"Tokens {tokens_seen:11,d} | "
            f"{progress:6.2f}% | "
            f"Loss {loss.item():8.4f} | "
            f"Grad {float(grad_norm):7.3f} | "
            f"{tok_per_sec:,.0f} tok/s | "
            f"ETA {eta_hours:.2f}h"
        )

        last_time = time.time()

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------
    if step % CHECKPOINT_EVERY == 0:

        checkpoint_path = (
            f"/kaggle/working/a3_checkpoints/"
            f"step_{step:07d}.pt"
        )

        torch.save({
            "model": model_a3.state_dict(),
            "step": step,
            "tokens_seen": tokens_seen,
            "architecture": "A3_prefix_lm_hybrid",
            "parameters": sum(
                p.numel() for p in model_a3.parameters()
            ),
            "context_length": CONTEXT_LEN,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "num_heads": NUM_HEADS,
            "ffn_size": FFN_SIZE,
            "prefix_ratio": PREFIX_RATIO,
            "dropout": 0.1,
            "learning_rate": LR,
            "weight_decay": 0.1,
        }, checkpoint_path)

        print(
            f"💾 Checkpoint saved: {checkpoint_path}"
        )

# ------------------------------------------------------------
# Final
# ------------------------------------------------------------
FINAL_PATH = "/kaggle/working/a3_checkpoints/a3_final.pt"

torch.save({
    "model": model_a3.state_dict(),
    "step": step,
    "tokens_seen": tokens_seen,
    "architecture": "A3_prefix_lm_hybrid",
    "parameters": sum(
        p.numel() for p in model_a3.parameters()
    ),
    "context_length": CONTEXT_LEN,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "num_heads": NUM_HEADS,
    "ffn_size": FFN_SIZE,
    "prefix_ratio": PREFIX_RATIO,
    "dropout": 0.1,
    "learning_rate": LR,
    "weight_decay": 0.1,
}, FINAL_PATH)

print("\n" + "=" * 70)
print("🎉 A3 TRAINING COMPLETE")
print("=" * 70)
print(f"Steps  : {step:,}")
print(f"Tokens : {tokens_seen:,}")
print(f"Model  : {FINAL_PATH}")
print("=" * 70)

A3 RECOVERY
Checkpoint : Step 5,000
Tokens     : 2,560,000
Progress   : 1.28%
Parameters : 51,868,672
HTTP retry protection enabled.
Starting from Step 5,000...

Step    5100 | Tokens   2,611,200 |   1.31% | Loss   5.6573 | Grad   3.016 | 5,636 tok/s | ETA 9.73h
Step    5200 | Tokens   2,662,400 |   1.33% | Loss   6.4187 | Grad   3.114 | 5,794 tok/s | ETA 9.46h
Step    5300 | Tokens   2,713,600 |   1.36% | Loss   7.3426 | Grad   3.589 | 5,738 tok/s | ETA 9.55h
Step    5400 | Tokens   2,764,800 |   1.38% | Loss   6.5873 | Grad   3.908 | 5,994 tok/s | ETA 9.14h
Step    5500 | Tokens   2,816,000 |   1.41% | Loss   6.2589 | Grad   2.899 | 6,296 tok/s | ETA 8.70h
Step    5600 | Tokens   2,867,200 |   1.43% | Loss   5.8796 | Grad   2.922 | 6,343 tok/s | ETA 8.63h
Step    5700 | Tokens   2,918,400 |   1.46% | Loss   6.5221 | Grad   3.616 | 6,336 tok/s | ETA 8.64h
Step    5800 | Tokens   2,969,600 |   1.48% | Loss   7.1238 | Grad   4.117 | 6,332 tok/s | ETA 8.64h
Step    5900 | Tokens   3,020,

In [2]:
import torch

path = "/kaggle/working/a3_checkpoints/step_0005000.pt"

ckpt = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Step:", ckpt["step"])
print("Tokens:", ckpt["tokens_seen"])
print("Architecture:", ckpt["architecture"])
print("Parameters:", ckpt["parameters"])
print("Context:", ckpt["context_length"])
print("Hidden:", ckpt["hidden_size"])
print("Layers:", ckpt["num_layers"])
print("Heads:", ckpt["num_heads"])
print("FFN:", ckpt["ffn_size"])
print("Prefix ratio:", ckpt["prefix_ratio"])

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/a3_checkpoints/step_0005000.pt'

In [13]:
import os
import torch

FINAL_PATH = "/kaggle/working/a3_checkpoints/a3_final.pt"

print("File exists:", os.path.exists(FINAL_PATH))

ckpt = torch.load(
    FINAL_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint information:")
print("Architecture :", ckpt["architecture"])
print("Parameters   :", f"{ckpt['parameters']:,}")
print("Steps        :", f"{ckpt['step']:,}")
print("Tokens       :", f"{ckpt['tokens_seen']:,}")
print("Context      :", ckpt["context_length"])
print("Hidden size  :", ckpt["hidden_size"])
print("Layers       :", ckpt["num_layers"])
print("Heads        :", ckpt["num_heads"])
print("FFN size     :", ckpt["ffn_size"])
print("Prefix ratio :", ckpt["prefix_ratio"])

File exists: True

Checkpoint information:
Architecture : A3_prefix_lm_hybrid
Parameters   : 51,868,672
Steps        : 390,625
Tokens       : 200,000,000
Context      : 2048
Hidden size  : 512
Layers       : 8
Heads        : 8
FFN size     : 2048
Prefix ratio : 0.5


In [14]:
import os
import numpy as np
import torch

# Paths
FINAL_PATH = "/kaggle/working/a3_checkpoints/a3_final.pt"
VAL_PATH = "/kaggle/working/phase1_validation_2048.npy"

# Load checkpoint
ckpt = torch.load(FINAL_PATH, map_location="cpu", weights_only=False)

# Load validation data
val_data = np.load(VAL_PATH, mmap_mode="r")

print("Checkpoint loaded ✅")
print("Validation shape:", val_data.shape)
print("Validation dtype:", val_data.dtype)
print("Validation tokens:", f"{val_data.size:,}")
print("Model parameters:", f"{ckpt['parameters']:,}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/phase1_validation_2048.npy'

In [1]:
import os

path = "/kaggle/working/a3_checkpoints/a3_final.pt"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    print("Size:", os.path.getsize(path) / (1024**2), "MB")

Exists: False


In [15]:
import os
import requests
import numpy as np
from huggingface_hub import hf_hub_url

REPO_ID = "Exploration-Lab/CS-781-Capstone"
VAL_FILE = "Phase_1_Pretraining/Validation/validation.json"

url = hf_hub_url(REPO_ID, VAL_FILE)

print("Downloading official validation data...")
r = requests.get(url, timeout=120)
r.raise_for_status()

print("Downloaded:", len(r.content) / (1024**2), "MB")

with open("/kaggle/working/validation.json", "wb") as f:
    f.write(r.content)

print("Saved:", "/kaggle/working/validation.json")

HTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Validation/validation.json

In [16]:
from huggingface_hub import HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"

api = HfApi()

files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="dataset"
)

for f in files:
    if "Phase_1_Pretraining" in f and ("Test" in f or "test" in f):
        print(f)

Phase_1_Pretraining/Test/test.json


In [17]:
import os
import requests
from huggingface_hub import hf_hub_url

REPO_ID = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

token = os.environ.get("HF_TOKEN")

url = hf_hub_url(
    repo_id=REPO_ID,
    filename=TEST_FILE,
    repo_type="dataset"
)

headers = {"Authorization": f"Bearer {token}"} if token else {}

r = requests.get(url, headers=headers, timeout=120)
r.raise_for_status()

TEST_PATH = "/kaggle/working/test.json"

with open(TEST_PATH, "wb") as f:
    f.write(r.content)

print("✅ Test set downloaded")
print("File:", TEST_PATH)
print("Size:", round(len(r.content) / (1024**2), 2), "MB")

HTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Test/test.json

In [18]:
import os
from huggingface_hub import hf_hub_download

TEST_PATH = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset",
    token=os.environ.get("HF_TOKEN"),
)

print("✅ Test set downloaded")
print("Path:", TEST_PATH)
print("Size:", round(os.path.getsize(TEST_PATH) / (1024**2), 2), "MB")

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

✅ Test set downloaded
Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json
Size: 2.89 MB


In [19]:
import json

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Type:", type(test_data))
print("Number of examples:", len(test_data))

print("\nFirst example:")
print(test_data[0] if isinstance(test_data, list) else list(test_data.items())[:1])

Type: <class 'dict'>
Number of examples: 1000

First example:
[('1', {'text': 'The issue involved in these petitions is as to whether the acquisition proceedings can be challenged at a belated stage. The issue is no more res integra as the issue has been considered by this Court time and again. 6. When a person challenges Section 4 Notification on any ground, it should be challenged within a reasonable period, and if the acquisition is challenged at a belated stage, the petition deserves to be dismissed only on this count. (Vide Hari Singh & Ors. Vs. State of U.P., AIR 1984 SC 1020). 7. A Constitution Bench of this Court, in [Aflatoon & Ors. Vs. Lt. Governor, Delhi & Ors.] AIR 1974 SC 2077, while dealing with the issue, observed as under:- ".... to have sat on the fence and allowed the government to complete the acquisition on the basis that notification under Section 4 and the declaration under Section 6 were valid and then to attack the notification on the grounds which were availabl

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = 50_000
CONTEXT_LENGTH = 2048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2048
PREFIX_RATIO = 0.5

# ---- A3 model ----

class A3Block(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)
        self.qkv = nn.Linear(HIDDEN_SIZE, 3 * HIDDEN_SIZE)
        self.out = nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE)

        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)
        self.fc1 = nn.Linear(HIDDEN_SIZE, FFN_SIZE)
        self.fc2 = nn.Linear(FFN_SIZE, HIDDEN_SIZE)

    def forward(self, x, attention_mask=None):
        h = self.ln1(x)

        q, k, v = self.qkv(h).chunk(3, dim=-1)

        B, T, C = q.shape
        H = NUM_HEADS
        D = C // H

        q = q.view(B, T, H, D).transpose(1, 2)
        k = k.view(B, T, H, D).transpose(1, 2)
        v = v.view(B, T, H, D).transpose(1, 2)

        attn = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attention_mask
        )

        attn = attn.transpose(1, 2).contiguous().view(B, T, C)

        x = x + self.out(attn)

        h = self.ln2(x)
        h = self.fc2(F.gelu(self.fc1(h)))
        x = x + h

        return x


class A3PrefixLM(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(VOCAB_SIZE, HIDDEN_SIZE)
        self.position_embedding = nn.Embedding(CONTEXT_LENGTH, HIDDEN_SIZE)

        self.layers = nn.ModuleList(
            [A3Block() for _ in range(NUM_LAYERS)]
        )

        self.ln_f = nn.LayerNorm(HIDDEN_SIZE)

    def forward(self, input_ids):
        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        ).unsqueeze(0)

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        # Prefix-LM mask
        prefix_len = min(
            int(T * PREFIX_RATIO),
            T
        )

        mask = torch.full(
            (T, T),
            float("-inf"),
            device=input_ids.device
        )

        # Prefix tokens attend bidirectionally
        mask[:prefix_len, :prefix_len] = 0

        # Suffix tokens attend to prefix + previous suffix tokens
        for i in range(prefix_len, T):
            mask[i, :i + 1] = 0

        mask = mask.unsqueeze(0).unsqueeze(0)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.ln_f(x)

        # tied embedding output
        logits = F.linear(
            x,
            self.token_embedding.weight
        )

        return logits


model = A3PrefixLM().to(device)

checkpoint = torch.load(
    "/kaggle/working/a3_checkpoints/a3_final.pt",
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("✅ A3 model loaded")
print("Device:", device)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Checkpoint steps:", checkpoint["step"])
print("Checkpoint tokens:", f"{checkpoint['tokens_seen']:,}")

RuntimeError: Error(s) in loading state_dict for A3PrefixLM:
	Missing key(s) in state_dict: "layers.0.qkv.weight", "layers.0.qkv.bias", "layers.0.out.weight", "layers.0.out.bias", "layers.0.fc1.weight", "layers.0.fc1.bias", "layers.0.fc2.weight", "layers.0.fc2.bias", "layers.1.qkv.weight", "layers.1.qkv.bias", "layers.1.out.weight", "layers.1.out.bias", "layers.1.fc1.weight", "layers.1.fc1.bias", "layers.1.fc2.weight", "layers.1.fc2.bias", "layers.2.qkv.weight", "layers.2.qkv.bias", "layers.2.out.weight", "layers.2.out.bias", "layers.2.fc1.weight", "layers.2.fc1.bias", "layers.2.fc2.weight", "layers.2.fc2.bias", "layers.3.qkv.weight", "layers.3.qkv.bias", "layers.3.out.weight", "layers.3.out.bias", "layers.3.fc1.weight", "layers.3.fc1.bias", "layers.3.fc2.weight", "layers.3.fc2.bias", "layers.4.qkv.weight", "layers.4.qkv.bias", "layers.4.out.weight", "layers.4.out.bias", "layers.4.fc1.weight", "layers.4.fc1.bias", "layers.4.fc2.weight", "layers.4.fc2.bias", "layers.5.qkv.weight", "laye

In [21]:
# Inspect the exact parameter names used during A3 training

print("Exact trained layer structure:\n")

for key in checkpoint["model"].keys():
    if key.startswith("layers.0"):
        print(key, tuple(checkpoint["model"][key].shape))

Exact trained layer structure:

layers.0.ln1.weight (512,)
layers.0.ln1.bias (512,)
layers.0.attn.in_proj_weight (1536, 512)
layers.0.attn.in_proj_bias (1536,)
layers.0.attn.out_proj.weight (512, 512)
layers.0.attn.out_proj.bias (512,)
layers.0.ln2.weight (512,)
layers.0.ln2.bias (512,)
layers.0.ffn.0.weight (2048, 512)
layers.0.ffn.0.bias (2048,)
layers.0.ffn.3.weight (512, 2048)
layers.0.ffn.3.bias (512,)


In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = 50_000
CONTEXT_LENGTH = 2048
HIDDEN_SIZE = 512
NUM_LAYERS = 8
NUM_HEADS = 8
FFN_SIZE = 2048
PREFIX_RATIO = 0.5


class A3ExactBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(HIDDEN_SIZE)

        self.attn = nn.MultiheadAttention(
            embed_dim=HIDDEN_SIZE,
            num_heads=NUM_HEADS,
            dropout=0.0,
            batch_first=True
        )

        self.ln2 = nn.LayerNorm(HIDDEN_SIZE)

        self.ffn = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, FFN_SIZE),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(FFN_SIZE, HIDDEN_SIZE)
        )

    def forward(self, x, attn_mask):
        h = self.ln1(x)

        attn_out, _ = self.attn(
            h, h, h,
            attn_mask=attn_mask,
            need_weights=False
        )

        x = x + attn_out

        h = self.ln2(x)
        x = x + self.ffn(h)

        return x


class A3Exact(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            CONTEXT_LENGTH,
            HIDDEN_SIZE
        )

        self.layers = nn.ModuleList([
            A3ExactBlock()
            for _ in range(NUM_LAYERS)
        ])

        self.ln_f = nn.LayerNorm(HIDDEN_SIZE)

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        ).unsqueeze(0)

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        prefix_len = int(T * PREFIX_RATIO)

        # Prefix-LM attention mask
        mask = torch.full(
            (T, T),
            float("-inf"),
            device=input_ids.device
        )

        # Prefix attends bidirectionally
        mask[:prefix_len, :prefix_len] = 0

        # Suffix attends to prefix + previous suffix tokens
        for i in range(prefix_len, T):
            mask[i, :i + 1] = 0

        for layer in self.layers:
            x = layer(x, mask)

        x = self.ln_f(x)

        # Tied input/output embedding
        logits = F.linear(
            x,
            self.token_embedding.weight
        )

        return logits


model = A3Exact().to(device)

model.load_state_dict(checkpoint["model"])
model.eval()

print("✅ EXACT A3 MODEL LOADED")
print("Device:", device)
print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)
print("Expected:", f"{checkpoint['parameters']:,}")

✅ EXACT A3 MODEL LOADED
Device: cuda
Parameters: 51,868,672
Expected: 51,868,672


In [23]:
import json
import torch

# Load tokenizer
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(
    "/kaggle/input/datasets/mathimohankowshik/tokenizer1/tokenizer.json"
)

# Get first official test prompt
prompt = test_data["1"]["text"]

# Tokenize
encoded = tokenizer.encode(prompt)
input_ids = encoded.ids

# Keep within model context
input_ids = input_ids[-(CONTEXT_LENGTH - 256):]

x = torch.tensor(
    [input_ids],
    dtype=torch.long,
    device=device
)

print("Prompt words :", len(prompt.split()))
print("Prompt tokens:", len(input_ids))

# Generate 128 whitespace-separated words approximately,
# using token generation followed by whitespace counting.
generated_ids = list(input_ids)

with torch.no_grad():
    for _ in range(300):
        context = torch.tensor(
            [generated_ids[-CONTEXT_LENGTH:]],
            dtype=torch.long,
            device=device
        )

        logits = model(context)

        next_token = torch.argmax(
            logits[:, -1, :],
            dim=-1
        ).item()

        generated_ids.append(next_token)

# Decode only newly generated tokens
new_tokens = generated_ids[len(input_ids):]

generated_text = tokenizer.decode(
    new_tokens
).strip()

words = generated_text.split()

print("\nGenerated words:", len(words))
print("\nGenerated text:\n")
print(generated_text[:2000])

Prompt words : 512
Prompt tokens: 617

Generated words: 300

Generated text:

Ġnot Ġlater Ġthan Ġthe Ġcorresponding Ġnew Ġconstituent Ġcolleges Ġat Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġtaken Ġplace Ġof Ġthe Ġsaid Ġnotification Ġin Ġthe Ġsaid Ġnotification Ġin Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġissued Ġunder Ġthis Ġnotification Ġshall Ġbe Ġissued Ġin Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġissued Ġin Ġthe Ġsaid Ġnotification Ġno . Ċ ( c . Ċ ( d ) ĠThe Ġsaid Ġnotification Ġno . Ċ ( e . Ċ ( e . Ċ ( e ., Ġin Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġissued Ġin Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġissued Ġin Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġissued Ġin Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġissued Ġin Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġissued Ġin Ġthe ĠOfficial ĠGazette , Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġdeemed Ġto Ġbring Ġback Ġfrom Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġdeemed Ġto Ġbring Ġthe Ġsaid Ġnotification Ġissued Ġin Ġthe Ġsaid Ġnotification Ġshall Ġbe Ġissued Ġin Ġthe Ġsaid Ġnotifi

In [24]:
print(tokenizer)

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"[PAD]", "single_word":False, "lstrip":False, "rstrip":False, ...}, {"id":1, "content":"[UNK]", "single_word":False, "lstrip":False, "rstrip":False, ...}, {"id":2, "content":"[BOS]", "single_word":False, "lstrip":False, "rstrip":False, ...}, {"id":3, "content":"[EOS]", "single_word":False, "lstrip":False, "rstrip":False, ...}, {"id":4, "content":"[MASK]", "single_word":False, "lstrip":False, "rstrip":False, ...}], normalizer=None, pre_tokenizer=ByteLevel(add_prefix_space=False, trim_offsets=True, use_regex=True), post_processor=None, decoder=None, model=BPE(dropout=None, unk_token="[UNK]", continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"[PAD]":0, "[UNK]":1, "[BOS]":2, "[EOS]":3, "[MASK]":4, ...}, merges=[("Ġ", "t"), ("h", "e"), ("Ġt", "he"), ("Ġ", "a"), ("o", "n"), ...]))


In [25]:
from tokenizers.decoders import ByteLevel

# Attach the correct ByteLevel decoder
tokenizer.decoder = ByteLevel()

# Test decoding using the already-generated tokens
fixed_text = tokenizer.decode(new_tokens).strip()

print("Fixed generated text:\n")
print(fixed_text[:2000])

print("\nWhitespace-separated words:", len(fixed_text.split()))

Fixed generated text:

not later than the corresponding new constituent colleges at the said notification shall be taken place of the said notification in the said notification in the said notification shall be issued under this notification shall be issued in the said notification shall be issued in the said notification no.
(c.
(d) The said notification no.
(e.
(e.
(e., in the said notification shall be issued in the said notification shall be issued in the said notification shall be issued in the said notification shall be issued in the said notification shall be issued in the Official Gazette, the said notification shall be deemed to bring back from the said notification shall be deemed to bring the said notification issued in the said notification shall be issued in the said notification shall be deemed to be deemed to be deemed to be deemed to be deemed to be deemed to be deemed to be deemed to be issued in the said notification shall be deemed to be deemed to be deemed to be dee

In [26]:
import torch
import torch.nn.functional as F

def generate_128_words(prompt, temperature=0.8, top_k=50, top_p=0.9):
    encoded = tokenizer.encode(prompt)
    input_ids = encoded.ids[-(CONTEXT_LENGTH - 256):]

    generated_ids = list(input_ids)

    for _ in range(500):
        context_ids = generated_ids[-CONTEXT_LENGTH:]

        x = torch.tensor(
            [context_ids],
            dtype=torch.long,
            device=device
        )

        with torch.no_grad():
            logits = model(x)[:, -1, :]

        # Temperature
        logits = logits / temperature

        # Top-k
        if top_k > 0:
            values, indices = torch.topk(logits, top_k)
            filtered = torch.full_like(logits, float("-inf"))
            filtered.scatter_(1, indices, values)
            logits = filtered

        # Top-p
        sorted_logits, sorted_indices = torch.sort(
            logits,
            descending=True
        )

        probs = F.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(probs, dim=-1)

        remove = cumulative_probs > top_p
        remove[:, 1:] = remove[:, :-1].clone()
        remove[:, 0] = False

        sorted_logits[remove] = float("-inf")

        logits = torch.full_like(logits, float("-inf"))
        logits.scatter_(1, sorted_indices, sorted_logits)

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, 1).item()

        generated_ids.append(next_token)

        # Decode only newly generated tokens
        generated_text = tokenizer.decode(
            generated_ids[len(input_ids):]
        ).strip()

        words = generated_text.split()

        if len(words) >= 128:
            return " ".join(words[:128])

    return " ".join(
        tokenizer.decode(
            generated_ids[len(input_ids):]
        ).strip().split()[:128]
    )


generated_128 = generate_128_words(
    test_data["1"]["text"]
)

print("Generated words:", len(generated_128.split()))
print("\nGenerated text:\n")
print(generated_128)

Generated words: 128

Generated text:

not set aside, a separate establishments and the expression in case of Courts of electronic credit of a separate Boards, the said court shall be in the electronic & Oathis to enable the said notification no such a particular circumstances in the first class shall be issued a set forth in the said court shall be issued the said Schedule or any other category of other cases in case of the first day on the first day on which it shall be carried out of the said section and the said notification shall not alter the corresponding new notification in a separate account of the said notification under this notification issued in which a set back of electronic credit of a registered office in any case of a


In [27]:
import json
import time

predictions = {}

start_time = time.time()

for i, (test_id, item) in enumerate(test_data.items(), 1):

    prompt = item["text"]

    text = generate_128_words(
        prompt,
        temperature=0.8,
        top_k=50,
        top_p=0.9
    )

    # Safety check
    words = text.split()

    if len(words) != 128:
        raise RuntimeError(
            f"ID {test_id}: generated {len(words)} words instead of 128"
        )

    predictions[str(test_id)] = {
        "text": text
    }

    if i % 50 == 0:
        elapsed = time.time() - start_time
        print(
            f"{i}/1000 completed | "
            f"{elapsed/60:.1f} min elapsed"
        )

print("\n🎉 ALL 1000 PREDICTIONS GENERATED")
print("Time:", round((time.time() - start_time) / 60, 2), "minutes")
print("Predictions:", len(predictions))

50/1000 completed | 3.8 min elapsed
100/1000 completed | 7.5 min elapsed
150/1000 completed | 11.4 min elapsed
200/1000 completed | 15.2 min elapsed
250/1000 completed | 18.9 min elapsed
300/1000 completed | 22.7 min elapsed
350/1000 completed | 26.5 min elapsed
400/1000 completed | 30.4 min elapsed
450/1000 completed | 34.2 min elapsed
500/1000 completed | 38.1 min elapsed
550/1000 completed | 41.9 min elapsed
600/1000 completed | 46.1 min elapsed
650/1000 completed | 49.7 min elapsed
700/1000 completed | 53.5 min elapsed
750/1000 completed | 57.4 min elapsed
800/1000 completed | 61.3 min elapsed
850/1000 completed | 65.3 min elapsed
900/1000 completed | 69.3 min elapsed
950/1000 completed | 73.3 min elapsed
1000/1000 completed | 77.2 min elapsed

🎉 ALL 1000 PREDICTIONS GENERATED
Time: 77.21 minutes
Predictions: 1000


In [28]:
import json

PRED_PATH = "/kaggle/working/prediction.json"

with open(PRED_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✅ prediction.json saved")
print("Path:", PRED_PATH)
print("Predictions:", len(predictions))

✅ prediction.json saved
Path: /kaggle/working/prediction.json
Predictions: 1000


In [29]:
import json

with open("/kaggle/working/prediction.json", "r", encoding="utf-8") as f:
    check = json.load(f)

assert len(check) == 1000
assert set(check.keys()) == set(test_data.keys())

bad = []

for test_id, item in check.items():
    text = item.get("text", "")
    word_count = len(text.split())

    if word_count != 128:
        bad.append((test_id, word_count))

print("Total IDs:", len(check))
print("Expected IDs:", len(test_data))
print("Invalid predictions:", len(bad))

if bad:
    print("First invalid:", bad[:10])
else:
    print("✅ ALL 1000 PREDICTIONS HAVE EXACTLY 128 WORDS")

Total IDs: 1000
Expected IDs: 1000
Invalid predictions: 0
✅ ALL 1000 PREDICTIONS HAVE EXACTLY 128 WORDS


In [30]:
import zipfile
import os

ZIP_PATH = "/kaggle/working/submission.zip"

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    z.write(
        "/kaggle/working/prediction.json",
        arcname="prediction.json"
    )

print("✅ submission.zip created")
print("Path:", ZIP_PATH)
print("Size:", round(os.path.getsize(ZIP_PATH) / 1024, 2), "KB")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("ZIP contents:", z.namelist())

✅ submission.zip created
Path: /kaggle/working/submission.zip
Size: 154.68 KB
ZIP contents: ['prediction.json']


In [31]:
import zipfile
import json

ZIP_PATH = "/kaggle/working/submission.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    assert z.namelist() == ["prediction.json"]

    with z.open("prediction.json") as f:
        data = json.load(f)

assert len(data) == 1000
assert set(data.keys()) == set(test_data.keys())
assert all(
    len(v["text"].split()) == 128
    for v in data.values()
)

print("🎉 FINAL SUBMISSION CHECK PASSED")
print("IDs        :", len(data))
print("Words each : 128")
print("ZIP        : submission.zip")
print("Status     : READY FOR CODABENCH 🚀")

🎉 FINAL SUBMISSION CHECK PASSED
IDs        : 1000
Words each : 128
ZIP        : submission.zip
Status     : READY FOR CODABENCH 🚀


In [32]:
# ============================================================
# A2 — EFFICIENT GQA
# CELL 1: CLEAN START + CONFIGURATION
# ============================================================

import os
import gc
import math
import time
import json
import random
import requests

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer
from huggingface_hub import hf_hub_url

# ------------------------------------------------------------
# Clean GPU / RAM
# ------------------------------------------------------------

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ------------------------------------------------------------
# A2 configuration
# ------------------------------------------------------------

VOCAB_SIZE = 50_000
CONTEXT_LENGTH = 2_048

HIDDEN_SIZE = 528

NUM_LAYERS = 8

NUM_Q_HEADS = 8
NUM_KV_HEADS = 2

HEAD_DIM = HIDDEN_SIZE // NUM_Q_HEADS

FFN_SIZE = 2_048

DROPOUT = 0.10

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.10

TOKEN_BUDGET = 200_000_000

CHECKPOINT_EVERY = 5_000

# Training sequence length
TRAIN_SEQ_LEN = 512

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mathimohankowshik/tokenizer1/"
    "tokenizer.json"
)

A2_ROOT = "/kaggle/working/a2_checkpoints"

os.makedirs(A2_ROOT, exist_ok=True)

# ------------------------------------------------------------
# Official dataset
# ------------------------------------------------------------

REPO_ID = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILES = {
    "Acts":
        "Phase_1_Pretraining/Train/train_acts.json",

    "High Courts":
        "Phase_1_Pretraining/Train/train_high_courts.json",

    "Supreme Court":
        "Phase_1_Pretraining/Train/train_supreme_court.json",

    "Tribunals":
        "Phase_1_Pretraining/Train/train_tribunals.json",
}

# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

assert HIDDEN_SIZE % NUM_Q_HEADS == 0
assert NUM_Q_HEADS % NUM_KV_HEADS == 0
assert os.path.exists(TOKENIZER_PATH)

print("=" * 70)
print("A2 — EFFICIENT GQA")
print("=" * 70)

print("Device          :", device)

if device.type == "cuda":
    print("GPU             :", torch.cuda.get_device_name(0))
    print(
        "VRAM            :",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

print()
print("Vocabulary      :", f"{VOCAB_SIZE:,}")
print("Context         :", CONTEXT_LENGTH)
print("Hidden size     :", HIDDEN_SIZE)
print("Layers          :", NUM_LAYERS)
print("Q heads         :", NUM_Q_HEADS)
print("KV heads        :", NUM_KV_HEADS)
print("Head dimension  :", HEAD_DIM)
print("FFN size        :", FFN_SIZE)
print("Train sequence  :", TRAIN_SEQ_LEN)
print("Learning rate   :", LEARNING_RATE)
print("Weight decay    :", WEIGHT_DECAY)
print("Token budget    :", f"{TOKEN_BUDGET:,}")
print()
print("Tokenizer path  :", TOKENIZER_PATH)
print("Checkpoint dir  :", A2_ROOT)

print("\n✅ A2 configuration ready")

A2 — EFFICIENT GQA
Device          : cuda
GPU             : Tesla T4
VRAM            : 14.56 GB

Vocabulary      : 50,000
Context         : 2048
Hidden size     : 528
Layers          : 8
Q heads         : 8
KV heads        : 2
Head dimension  : 66
FFN size        : 2048
Train sequence  : 512
Learning rate   : 0.0001
Weight decay    : 0.1
Token budget    : 200,000,000

Tokenizer path  : /kaggle/input/datasets/mathimohankowshik/tokenizer1/tokenizer.json
Checkpoint dir  : /kaggle/working/a2_checkpoints

✅ A2 configuration ready


In [33]:
# ============================================================
# A2 — EFFICIENT GQA MODEL
# CELL 2: MODEL DEFINITION
# ============================================================

class A2GQAAttention(nn.Module):

    def __init__(self):
        super().__init__()

        self.q_proj = nn.Linear(
            HIDDEN_SIZE,
            NUM_Q_HEADS * HEAD_DIM,
            bias=True
        )

        self.k_proj = nn.Linear(
            HIDDEN_SIZE,
            NUM_KV_HEADS * HEAD_DIM,
            bias=True
        )

        self.v_proj = nn.Linear(
            HIDDEN_SIZE,
            NUM_KV_HEADS * HEAD_DIM,
            bias=True
        )

        self.out_proj = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE,
            bias=True
        )

    def forward(self, x, causal=True):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T, NUM_Q_HEADS, HEAD_DIM
        ).transpose(1, 2)

        k = k.view(
            B, T, NUM_KV_HEADS, HEAD_DIM
        ).transpose(1, 2)

        v = v.view(
            B, T, NUM_KV_HEADS, HEAD_DIM
        ).transpose(1, 2)

        # GQA:
        # repeat each KV head for the corresponding Q heads
        repeat_factor = NUM_Q_HEADS // NUM_KV_HEADS

        k = k.repeat_interleave(
            repeat_factor,
            dim=1
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1
        )

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=causal
        )

        y = y.transpose(
            1, 2
        ).contiguous().view(
            B, T, HIDDEN_SIZE
        )

        return self.out_proj(y)


class A2Block(nn.Module):

    def __init__(self):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            HIDDEN_SIZE
        )

        self.attn = A2GQAAttention()

        self.ln2 = nn.LayerNorm(
            HIDDEN_SIZE
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                HIDDEN_SIZE,
                FFN_SIZE
            ),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(
                FFN_SIZE,
                HIDDEN_SIZE
            )
        )

    def forward(self, x):

        x = x + self.attn(
            self.ln1(x)
        )

        x = x + self.ffn(
            self.ln2(x)
        )

        return x


class A2GQA(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE
        )

        self.position_embedding = nn.Embedding(
            CONTEXT_LENGTH,
            HIDDEN_SIZE
        )

        self.layers = nn.ModuleList([
            A2Block()
            for _ in range(NUM_LAYERS)
        ])

        self.ln_f = nn.LayerNorm(
            HIDDEN_SIZE
        )

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        for layer in self.layers:
            x = layer(x)

        x = self.ln_f(x)

        # Tied input/output embedding
        logits = F.linear(
            x,
            self.token_embedding.weight
        )

        return logits


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

model = A2GQA().to(device)

param_count = sum(
    p.numel()
    for p in model.parameters()
)

print("=" * 70)
print("A2 MODEL CREATED")
print("=" * 70)

print(
    "Parameters:",
    f"{param_count:,}"
)

print(
    "Parameters (M):",
    f"{param_count / 1e6:.2f}M"
)

print(
    "Expected:",
    "50,407,648"
)

assert param_count == 50_407_648

print("\n✅ EXACT PARAMETER COUNT VERIFIED")

A2 MODEL CREATED
Parameters: 50,407,648
Parameters (M): 50.41M
Expected: 50,407,648

✅ EXACT PARAMETER COUNT VERIFIED


In [34]:
# ============================================================
# A2 — EFFICIENT GQA
# PRODUCTION TRAINING — 200M TOKENS
# ============================================================

import os
import gc
import json
import time
import math
import random
import requests

import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

from huggingface_hub import hf_hub_url

# ============================================================
# 1. TRAINING CONFIG
# ============================================================

TARGET_TOKENS = 200_000_000
SEQ_LEN = 512

STEPS = TARGET_TOKENS // SEQ_LEN

assert STEPS * SEQ_LEN == TARGET_TOKENS

VOCAB_LOSS_CHUNK = 32

PRINT_EVERY = 100
SAVE_EVERY = 5_000

LR = 1e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

# ============================================================
# 2. CLEAN MEMORY
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model.train()

# ============================================================
# 3. EXACT HIDDEN-STATE FORWARD
# ============================================================

def model_hidden(input_ids):

    B, T = input_ids.shape

    positions = torch.arange(
        T,
        device=input_ids.device
    )

    x = (
        model.token_embedding(input_ids)
        + model.position_embedding(positions)
    )

    for layer in model.layers:

        # Activation checkpointing
        x = checkpoint(
            layer,
            x,
            use_reentrant=False
        )

    x = model.ln_f(x)

    return x


# ============================================================
# 4. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    fused=True
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

# ============================================================
# 5. CHECKPOINT / RESUME
# ============================================================

FINAL_CHECKPOINT = os.path.join(
    A2_ROOT,
    "a2_final.pt"
)

def checkpoint_path(step):

    return os.path.join(
        A2_ROOT,
        f"step_{step:07d}.pt"
    )


start_step = 0
tokens_seen = 0

# Resume from latest checkpoint if available
existing = []

for filename in os.listdir(A2_ROOT):

    if (
        filename.startswith("step_")
        and filename.endswith(".pt")
    ):
        existing.append(filename)

if existing:

    existing.sort()

    latest = os.path.join(
        A2_ROOT,
        existing[-1]
    )

    print("🔄 Resuming from:", latest)

    ckpt = torch.load(
        latest,
        map_location="cuda",
        weights_only=False
    )

    model.load_state_dict(
        ckpt["model"]
    )

    optimizer.load_state_dict(
        ckpt["optimizer"]
    )

    if "scaler" in ckpt:
        scaler.load_state_dict(
            ckpt["scaler"]
        )

    start_step = ckpt["step"]
    tokens_seen = ckpt["tokens_seen"]

    print(
        "Resume step:",
        start_step
    )

    print(
        "Tokens seen:",
        f"{tokens_seen:,}"
    )

else:

    print("🆕 Starting A2 from scratch")


# ============================================================
# 6. TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

print("Tokenizer loaded ✅")


# ============================================================
# 7. OFFICIAL SOURCE STREAMING
# ============================================================

def stream_json_objects(url, token):

    headers = {}

    if token:
        headers["Authorization"] = (
            f"Bearer {token}"
        )

    while True:

        try:

            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 300)
            ) as response:

                response.raise_for_status()

                decoder = json.JSONDecoder()

                buffer = ""
                started = False
                finished = False

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024,
                    decode_unicode=True
                ):

                    if not chunk:
                        continue

                    buffer += chunk

                    while True:

                        buffer = buffer.lstrip()

                        if not buffer:
                            break

                        # ------------------------------------------------
                        # JSON array
                        # ------------------------------------------------

                        if not started:

                            if buffer.startswith("["):

                                started = True
                                buffer = buffer[1:]
                                continue

                            else:

                                # JSONL fallback
                                started = True

                        if buffer.startswith("]"):

                            finished = True
                            buffer = buffer[1:]
                            break

                        # Remove commas between array objects
                        if buffer.startswith(","):

                            buffer = buffer[1:]
                            continue

                        try:

                            obj, index = decoder.raw_decode(
                                buffer
                            )

                        except json.JSONDecodeError:

                            break

                        buffer = buffer[index:]

                        if isinstance(obj, dict):

                            yield obj

                if finished:
                    return

                # If stream ended normally, return
                return

        except Exception as e:

            print(
                "\n⚠️ Network interruption:",
                repr(e)
            )

            print(
                "Retrying source in 5 seconds..."
            )

            time.sleep(5)

            continue


def token_stream(source_name):

    filename = SOURCE_FILES[source_name]

    token = os.environ.get("HF_TOKEN")

    url = hf_hub_url(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset"
    )

    print(
        f"\n🌐 Opening {source_name}..."
    )

    count = 0

    for obj in stream_json_objects(
        url,
        token
    ):

        text = obj.get("text")

        if not text:
            continue

        ids = tokenizer.encode(
            text
        ).ids

        for token_id in ids:

            yield token_id

        count += 1

        if count % 1000 == 0:

            print(
                f"\r{source_name}: "
                f"{count:,} documents",
                end=""
            )

    print(
        f"\n✅ {source_name} stream finished"
    )


# ============================================================
# 8. CROSS-SOURCE TOKEN GENERATOR
# ============================================================

def all_training_tokens():

    source_names = list(
        SOURCE_FILES.keys()
    )

    source_index = 0

    while True:

        source_name = source_names[
            source_index
        ]

        for token_id in token_stream(
            source_name
        ):

            yield token_id

        source_index = (
            source_index + 1
        ) % len(source_names)


# ============================================================
# 9. BLOCK GENERATOR
# ============================================================

token_iterator = all_training_tokens()

def get_training_block():

    block = []

    while len(block) < SEQ_LEN:

        block.append(
            next(token_iterator)
        )

    return torch.tensor(
        block,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)


# ============================================================
# 10. TRAINING
# ============================================================

print()
print("=" * 70)
print("🚀 A2 PRODUCTION TRAINING STARTING")
print("=" * 70)

print(
    "Target tokens :",
    f"{TARGET_TOKENS:,}"
)

print(
    "Total steps   :",
    f"{STEPS:,}"
)

print(
    "Starting step :",
    start_step
)

print(
    "Remaining     :",
    f"{STEPS - start_step:,}"
)

print(
    "Sequence      :",
    SEQ_LEN
)

print(
    "Loss chunk    :",
    VOCAB_LOSS_CHUNK
)

print("=" * 70)


train_start = time.time()

running_loss = 0.0

for step in range(
    start_step + 1,
    STEPS + 1
):

    step_start = time.time()

    optimizer.zero_grad(
        set_to_none=True
    )

    # --------------------------------------------------------
    # Training block
    # --------------------------------------------------------

    batch = get_training_block()

    # Input and next-token target
    input_ids = batch[:, :-1]

    targets = batch[:, 1:]

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):

        hidden = model_hidden(
            input_ids
        )

    # --------------------------------------------------------
    # Memory-safe vocabulary loss
    # --------------------------------------------------------

    total_loss = 0.0

    total_positions = targets.numel()

    for start in range(
        0,
        total_positions,
        VOCAB_LOSS_CHUNK
    ):

        end = min(
            start + VOCAB_LOSS_CHUNK,
            total_positions
        )

        h_chunk = hidden[:, start:end]

        target_chunk = targets[:, start:end]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            logits = F.linear(
                h_chunk,
                model.token_embedding.weight
            )

            loss_chunk = F.cross_entropy(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                target_chunk.reshape(-1)
            )

            # Weight chunk by number of positions
            loss_chunk = (
                loss_chunk
                * (end - start)
                / total_positions
            )

        scaler.scale(
            loss_chunk
        ).backward(
            retain_graph=(
                end < total_positions
            )
        )

        total_loss += loss_chunk.detach().float().item()

        del logits
        del h_chunk
        del target_chunk

    # --------------------------------------------------------
    # Gradient clipping
    # --------------------------------------------------------

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRAD_CLIP
    )

    scaler.step(
        optimizer
    )

    scaler.update()

    running_loss += total_loss

    tokens_seen = step * SEQ_LEN

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if step % PRINT_EVERY == 0:

        elapsed = time.time() - train_start

        avg_loss = (
            running_loss
            / PRINT_EVERY
        )

        running_loss = 0.0

        tokens_per_sec = (
            tokens_seen
            / elapsed
        )

        remaining_tokens = (
            TARGET_TOKENS
            - tokens_seen
        )

        eta_seconds = (
            remaining_tokens
            / tokens_per_sec
            if tokens_per_sec > 0
            else 0
        )

        print(
            f"Step {step:>7,}/{STEPS:,} | "
            f"Tokens {tokens_seen:>12,}/{TARGET_TOKENS:,} | "
            f"Loss {avg_loss:.4f} | "
            f"{tokens_per_sec:,.0f} tok/s | "
            f"ETA {eta_seconds/3600:.2f} h"
        )

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    if step % SAVE_EVERY == 0:

        path = checkpoint_path(
            step
        )

        torch.save(
            {
                "model":
                    model.state_dict(),

                "optimizer":
                    optimizer.state_dict(),

                "scaler":
                    scaler.state_dict(),

                "step":
                    step,

                "tokens_seen":
                    tokens_seen,

                "architecture":
                    "A2_efficient_GQA",

                "parameters":
                    sum(
                        p.numel()
                        for p in model.parameters()
                    ),

                "vocab_size":
                    VOCAB_SIZE,

                "context_length":
                    CONTEXT_LENGTH,

                "hidden_size":
                    HIDDEN_SIZE,

                "num_layers":
                    NUM_LAYERS,

                "num_q_heads":
                    NUM_Q_HEADS,

                "num_kv_heads":
                    NUM_KV_HEADS,

                "ffn_size":
                    FFN_SIZE,

                "learning_rate":
                    LR,

                "weight_decay":
                    WEIGHT_DECAY,
            },
            path
        )

        print(
            f"\n💾 Checkpoint saved: {path}\n"
        )

# ============================================================
# 11. FINAL MODEL
# ============================================================

torch.save(
    {
        "model":
            model.state_dict(),

        "step":
            STEPS,

        "tokens_seen":
            TARGET_TOKENS,

        "architecture":
            "A2_efficient_GQA",

        "parameters":
            sum(
                p.numel()
                for p in model.parameters()
            ),

        "vocab_size":
            VOCAB_SIZE,

        "context_length":
            CONTEXT_LENGTH,

        "hidden_size":
            HIDDEN_SIZE,

        "num_layers":
            NUM_LAYERS,

        "num_q_heads":
            NUM_Q_HEADS,

        "num_kv_heads":
            NUM_KV_HEADS,

        "ffn_size":
            FFN_SIZE,

        "learning_rate":
            LR,

        "weight_decay":
            WEIGHT_DECAY,
    },
    FINAL_CHECKPOINT
)

print()
print("=" * 70)
print("🎉 A2 TRAINING COMPLETE")
print("=" * 70)

print(
    "Steps  :",
    f"{STEPS:,}"
)

print(
    "Tokens :",
    f"{TARGET_TOKENS:,}"
)

print(
    "Model  :",
    FINAL_CHECKPOINT
)

print("=" * 70)

🆕 Starting A2 from scratch
Tokenizer loaded ✅

🚀 A2 PRODUCTION TRAINING STARTING
Target tokens : 200,000,000
Total steps   : 390,625
Starting step : 0
Remaining     : 390,625
Sequence      : 512
Loss chunk    : 32

🌐 Opening Acts...

⚠️ Network interruption: HTTPError('401 Client Error: Unauthorized for url: https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_acts.json')
Retrying source in 5 seconds...

⚠️ Network interruption: HTTPError('401 Client Error: Unauthorized for url: https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_acts.json')
Retrying source in 5 seconds...

⚠️ Network interruption: HTTPError('401 Client Error: Unauthorized for url: https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/train_acts.json')
Retrying source in 5 seconds...

⚠️ Network interruption: HTTPError('401 Client Error: Unauthorized for url: 

KeyboardInterrupt: 

In [35]:
from huggingface_hub import get_token, hf_hub_url
import requests

HF_TOKEN = get_token()

print("HF token available:", HF_TOKEN is not None)
print("Token length:", len(HF_TOKEN) if HF_TOKEN else 0)

# Test authenticated access to the actual training file
url = hf_hub_url(
    repo_id=REPO_ID,
    filename=SOURCE_FILES["Acts"],
    repo_type="dataset"
)

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

r = requests.get(
    url,
    headers=headers,
    stream=True,
    timeout=(30, 60)
)

print("HTTP status:", r.status_code)

r.raise_for_status()

print("✅ Authenticated access to Acts works")

HF token available: True
Token length: 37
HTTP status: 200
✅ Authenticated access to Acts works


In [36]:
from huggingface_hub import get_token

HF_TOKEN = get_token()

assert HF_TOKEN is not None

print("✅ HF authentication ready")
print("Token length:", len(HF_TOKEN))

✅ HF authentication ready
Token length: 37


In [37]:
# ============================================================
# A2 — EFFICIENT GQA
# FINAL PRODUCTION TRAINING CELL
# 200M TOKENS
# ============================================================

import os
import gc
import json
import time
import requests

import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from huggingface_hub import get_token, hf_hub_url

# ============================================================
# 1. CONFIG
# ============================================================

TARGET_TOKENS = 200_000_000
SEQ_LEN = 512
TOTAL_STEPS = TARGET_TOKENS // SEQ_LEN

VOCAB_LOSS_CHUNK = 32

PRINT_EVERY = 100
SAVE_EVERY = 5_000

LR = 1e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

# ============================================================
# 2. CLEAN MEMORY
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

model.train()

# ============================================================
# 3. HUGGING FACE AUTHENTICATION
# ============================================================

HF_TOKEN = get_token()

assert HF_TOKEN is not None

print("✅ Hugging Face authentication ready")

# ============================================================
# 4. OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    fused=True
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

# ============================================================
# 5. CHECKPOINT RESUME
# ============================================================

start_step = 0
tokens_seen = 0

checkpoint_files = sorted([
    f for f in os.listdir(A2_ROOT)
    if f.startswith("step_") and f.endswith(".pt")
])

if checkpoint_files:

    latest = os.path.join(
        A2_ROOT,
        checkpoint_files[-1]
    )

    print("🔄 Resuming from:", latest)

    ckpt = torch.load(
        latest,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(
        ckpt["model"]
    )

    optimizer.load_state_dict(
        ckpt["optimizer"]
    )

    if "scaler" in ckpt:
        scaler.load_state_dict(
            ckpt["scaler"]
        )

    start_step = ckpt["step"]
    tokens_seen = ckpt["tokens_seen"]

    print(
        "Resume step:",
        start_step
    )

    print(
        "Tokens seen:",
        f"{tokens_seen:,}"
    )

else:

    print("🆕 Starting A2 from scratch")

# ============================================================
# 6. OFFICIAL SOURCE STREAMING
# ============================================================

def stream_json_objects(url):

    headers = {
        "Authorization":
            f"Bearer {HF_TOKEN}"
    }

    while True:

        try:

            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 300)
            ) as response:

                response.raise_for_status()

                decoder = json.JSONDecoder()

                buffer = ""
                started = False
                finished = False

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024,
                    decode_unicode=True
                ):

                    if not chunk:
                        continue

                    buffer += chunk

                    while True:

                        buffer = buffer.lstrip()

                        if not buffer:
                            break

                        if not started:

                            if buffer.startswith("["):
                                started = True
                                buffer = buffer[1:]
                                continue

                            started = True

                        if buffer.startswith("]"):
                            finished = True
                            break

                        if buffer.startswith(","):
                            buffer = buffer[1:]
                            continue

                        try:

                            obj, idx = decoder.raw_decode(
                                buffer
                            )

                        except json.JSONDecodeError:

                            break

                        buffer = buffer[idx:]

                        if isinstance(obj, dict):
                            yield obj

                if finished:
                    return

                return

        except Exception as e:

            print(
                "\n⚠️ Network interruption:",
                repr(e)
            )

            print(
                "Retrying in 5 seconds..."
            )

            time.sleep(5)


def token_stream(source_name):

    filename = SOURCE_FILES[source_name]

    url = hf_hub_url(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset"
    )

    print(
        f"\n🌐 Opening {source_name}..."
    )

    documents = 0

    for obj in stream_json_objects(url):

        text = obj.get("text")

        if not text:
            continue

        ids = tokenizer.encode(text).ids

        for token_id in ids:
            yield token_id

        documents += 1

        if documents % 1000 == 0:

            print(
                f"\r{source_name}: "
                f"{documents:,} documents",
                end=""
            )

    print(
        f"\n✅ {source_name} finished"
    )


# ============================================================
# 7. CYCLICAL FOUR-SOURCE STREAM
# ============================================================

def all_training_tokens():

    sources = list(
        SOURCE_FILES.keys()
    )

    index = 0

    while True:

        source = sources[index]

        for token_id in token_stream(source):
            yield token_id

        index = (
            index + 1
        ) % len(sources)


token_iterator = all_training_tokens()

# ============================================================
# 8. TRAINING BLOCK
# ============================================================

def get_training_block():

    block = []

    while len(block) < SEQ_LEN:

        block.append(
            next(token_iterator)
        )

    return torch.tensor(
        block,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

# ============================================================
# 9. MEMORY-SAFE FORWARD
# ============================================================

def forward_hidden(input_ids):

    B, T = input_ids.shape

    positions = torch.arange(
        T,
        device=input_ids.device
    )

    x = (
        model.token_embedding(input_ids)
        + model.position_embedding(positions)
    )

    for layer in model.layers:

        x = checkpoint(
            layer,
            x,
            use_reentrant=False
        )

    return model.ln_f(x)

# ============================================================
# 10. CHECKPOINT FUNCTION
# ============================================================

def save_checkpoint(step, tokens_seen):

    path = os.path.join(
        A2_ROOT,
        f"step_{step:07d}.pt"
    )

    torch.save(
        {
            "model":
                model.state_dict(),

            "optimizer":
                optimizer.state_dict(),

            "scaler":
                scaler.state_dict(),

            "step":
                step,

            "tokens_seen":
                tokens_seen,

            "architecture":
                "A2_efficient_GQA",

            "parameters":
                sum(
                    p.numel()
                    for p in model.parameters()
                ),

            "vocab_size":
                VOCAB_SIZE,

            "context_length":
                CONTEXT_LENGTH,

            "hidden_size":
                HIDDEN_SIZE,

            "num_layers":
                NUM_LAYERS,

            "num_q_heads":
                NUM_Q_HEADS,

            "num_kv_heads":
                NUM_KV_HEADS,

            "ffn_size":
                FFN_SIZE,

            "learning_rate":
                LR,

            "weight_decay":
                WEIGHT_DECAY
        },
        path
    )

    print(
        f"\n💾 Checkpoint saved: {path}\n"
    )

# ============================================================
# 11. START TRAINING
# ============================================================

print()
print("=" * 70)
print("🚀 A2 PRODUCTION TRAINING")
print("=" * 70)

print(
    "Target tokens :",
    f"{TARGET_TOKENS:,}"
)

print(
    "Total steps   :",
    f"{TOTAL_STEPS:,}"
)

print(
    "Starting step :",
    start_step
)

print(
    "Remaining     :",
    f"{TOTAL_STEPS - start_step:,}"
)

print(
    "Sequence      :",
    SEQ_LEN
)

print(
    "Loss chunk    :",
    VOCAB_LOSS_CHUNK
)

print("=" * 70)

train_start = time.time()
running_loss = 0.0

# ============================================================
# 12. MAIN LOOP
# ============================================================

for step in range(
    start_step + 1,
    TOTAL_STEPS + 1
):

    optimizer.zero_grad(
        set_to_none=True
    )

    batch = get_training_block()

    input_ids = batch[:, :-1]
    targets = batch[:, 1:]

    # --------------------------------------------------------
    # Forward
    # --------------------------------------------------------

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device.type == "cuda")
    ):

        hidden = forward_hidden(
            input_ids
        )

    # --------------------------------------------------------
    # Chunked vocabulary loss
    # --------------------------------------------------------

    total_positions = targets.numel()
    total_loss = 0.0

    for start in range(
        0,
        total_positions,
        VOCAB_LOSS_CHUNK
    ):

        end = min(
            start + VOCAB_LOSS_CHUNK,
            total_positions
        )

        h = hidden[:, start:end]
        y = targets[:, start:end]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            logits = F.linear(
                h,
                model.token_embedding.weight
            )

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                y.reshape(-1)
            )

            loss = (
                loss
                * (end - start)
                / total_positions
            )

        scaler.scale(loss).backward(
            retain_graph=(
                end < total_positions
            )
        )

        total_loss += loss.detach().float().item()

        del logits
        del h
        del y

    # --------------------------------------------------------
    # Gradient clipping + optimizer
    # --------------------------------------------------------

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRAD_CLIP
    )

    scaler.step(optimizer)
    scaler.update()

    running_loss += total_loss

    tokens_seen = step * SEQ_LEN

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if step % PRINT_EVERY == 0:

        elapsed = (
            time.time()
            - train_start
        )

        avg_loss = (
            running_loss
            / PRINT_EVERY
        )

        running_loss = 0.0

        tok_per_sec = (
            tokens_seen
            / elapsed
        )

        remaining = (
            TARGET_TOKENS
            - tokens_seen
        )

        eta = (
            remaining
            / tok_per_sec
            if tok_per_sec > 0
            else 0
        )

        progress = (
            tokens_seen
            / TARGET_TOKENS
            * 100
        )

        print(
            f"Step {step:,}/{TOTAL_STEPS:,} | "
            f"{progress:.2f}% | "
            f"Tokens {tokens_seen:,} | "
            f"Loss {avg_loss:.4f} | "
            f"{tok_per_sec:,.0f} tok/s | "
            f"ETA {eta/3600:.2f} h"
        )

    # --------------------------------------------------------
    # Save checkpoint
    # --------------------------------------------------------

    if step % SAVE_EVERY == 0:

        save_checkpoint(
            step,
            tokens_seen
        )

# ============================================================
# 13. FINAL CHECKPOINT
# ============================================================

torch.save(
    {
        "model":
            model.state_dict(),

        "step":
            TOTAL_STEPS,

        "tokens_seen":
            TARGET_TOKENS,

        "architecture":
            "A2_efficient_GQA",

        "parameters":
            sum(
                p.numel()
                for p in model.parameters()
            ),

        "vocab_size":
            VOCAB_SIZE,

        "context_length":
            CONTEXT_LENGTH,

        "hidden_size":
            HIDDEN_SIZE,

        "num_layers":
            NUM_LAYERS,

        "num_q_heads":
            NUM_Q_HEADS,

        "num_kv_heads":
            NUM_KV_HEADS,

        "ffn_size":
            FFN_SIZE,

        "learning_rate":
            LR,

        "weight_decay":
            WEIGHT_DECAY
    },
    FINAL_CHECKPOINT
)

print()
print("=" * 70)
print("🎉 A2 TRAINING COMPLETE")
print("=" * 70)

print(
    "Steps  :",
    f"{TOTAL_STEPS:,}"
)

print(
    "Tokens :",
    f"{TARGET_TOKENS:,}"
)

print(
    "Model  :",
    FINAL_CHECKPOINT
)

print("=" * 70)

✅ Hugging Face authentication ready
🆕 Starting A2 from scratch

🚀 A2 PRODUCTION TRAINING
Target tokens : 200,000,000
Total steps   : 390,625
Starting step : 0
Remaining     : 390,625
Sequence      : 512
Loss chunk    : 32

🌐 Opening Acts...
Step 100/390,625 | 0.03% | Tokens 51,200 | Loss 55.7612 | 838 tok/s | ETA 66.27 h
Step 200/390,625 | 0.05% | Tokens 102,400 | Loss 24.7808 | 847 tok/s | ETA 65.56 h
Step 300/390,625 | 0.08% | Tokens 153,600 | Loss 21.2555 | 851 tok/s | ETA 65.20 h


KeyboardInterrupt: 

In [4]:
# ================================================================
# A2 — SPEED-OPTIMIZED PRODUCTION TRAINING
# ================================================================

import os
import time
import math
import json
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.amp import autocast, GradScaler
from huggingface_hub import hf_hub_url, get_token

# ------------------------------------------------
# SPEED SETTINGS
# ------------------------------------------------
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

DEVICE = "cuda"

TARGET_TOKENS = 200_000_000
SEQ_LEN = 512

# BIG SPEEDUP
VOCAB_LOSS_CHUNK = 256

LR = 1e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

PRINT_EVERY = 100
SAVE_EVERY = 5000

CHECKPOINT_DIR = "/kaggle/working/a2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

HF_TOKEN = get_token()

print("=" * 70)
print("🚀 A2 SPEED-OPTIMIZED TRAINING")
print("=" * 70)
print(f"Target tokens       : {TARGET_TOKENS:,}")
print(f"Sequence length     : {SEQ_LEN}")
print(f"Vocab loss chunk    : {VOCAB_LOSS_CHUNK}")
print(f"Learning rate       : {LR}")
print(f"TF32                : ENABLED")
print(f"Activation checkpoint: DISABLED")
print("=" * 70)


# ================================================================
# MODEL
# ================================================================

class A2GQAAttention(nn.Module):

    def __init__(self, hidden=528, q_heads=8, kv_heads=2):
        super().__init__()

        self.hidden = hidden
        self.q_heads = q_heads
        self.kv_heads = kv_heads
        self.head_dim = hidden // q_heads

        assert hidden % q_heads == 0

        self.q_proj = nn.Linear(
            hidden,
            q_heads * self.head_dim
        )

        self.k_proj = nn.Linear(
            hidden,
            kv_heads * self.head_dim
        )

        self.v_proj = nn.Linear(
            hidden,
            kv_heads * self.head_dim
        )

        self.out_proj = nn.Linear(hidden, hidden)

    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.q_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.kv_heads,
            self.head_dim
        ).transpose(1, 2)

        # GQA: expand KV heads
        repeat_factor = self.q_heads // self.kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1
        )

        # Native optimized SDPA
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=0.0,
            is_causal=True
        )

        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, C)

        return self.out_proj(out)


class A2Block(nn.Module):

    def __init__(
        self,
        hidden=528,
        heads=8,
        kv_heads=2,
        ffn_size=2048
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(hidden)

        self.attn = A2GQAAttention(
            hidden,
            heads,
            kv_heads
        )

        self.ln2 = nn.LayerNorm(hidden)

        self.ffn = nn.Sequential(
            nn.Linear(hidden, ffn_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(ffn_size, hidden)
        )

    def forward(self, x):

        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))

        return x


class A2GQA(nn.Module):

    def __init__(
        self,
        vocab_size=50000,
        context=2048,
        hidden=528,
        layers=8,
        heads=8,
        kv_heads=2,
        ffn_size=2048
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            hidden
        )

        self.position_embedding = nn.Embedding(
            context,
            hidden
        )

        self.layers = nn.ModuleList([
            A2Block(
                hidden,
                heads,
                kv_heads,
                ffn_size
            )
            for _ in range(layers)
        ])

        self.ln_f = nn.LayerNorm(hidden)

    def forward(self, input_ids):

        B, T = input_ids.shape

        positions = torch.arange(
            T,
            device=input_ids.device
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )

        for layer in self.layers:
            x = layer(x)

        return self.ln_f(x)


# ================================================================
# CREATE MODEL
# ================================================================

model = A2GQA().to(DEVICE)

params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Parameters: {params:,}")
print(f"Parameters (M): {params / 1e6:.2f}M")

assert params == 50_407_648

print("✅ EXACT A2 PARAMETER COUNT VERIFIED")


# ================================================================
# OPTIMIZER
# ================================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    fused=True
)

scaler = GradScaler("cuda")


# ================================================================
# RESUME
# ================================================================

latest_step = 0
latest_file = None

files = [
    f for f in os.listdir(CHECKPOINT_DIR)
    if f.endswith(".pt")
    and f.startswith("step_")
]

if files:

    files.sort()

    latest_file = os.path.join(
        CHECKPOINT_DIR,
        files[-1]
    )

    print(f"\n🔄 Loading checkpoint:")
    print(latest_file)

    checkpoint = torch.load(
        latest_file,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint["model"]
    )

    if "optimizer" in checkpoint:
        optimizer.load_state_dict(
            checkpoint["optimizer"]
        )

    if "scaler" in checkpoint:
        scaler.load_state_dict(
            checkpoint["scaler"]
        )

    latest_step = checkpoint["step"]

    print(f"Resuming from step {latest_step}")

else:

    print("\n🆕 Starting A2 from scratch")


# ================================================================
# DATA STREAM
# ================================================================

REPO_ID = "Exploration-Lab/CS-781-Capstone"

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/Acts.jsonl",
    "Phase_1_Pretraining/Train/High_Courts.jsonl",
    "Phase_1_Pretraining/Train/Supreme_Court.jsonl",
    "Phase_1_Pretraining/Train/Tribunals.jsonl",
]


def stream_jsonl(filename):

    url = hf_hub_url(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset"
    )

    headers = {
        "Authorization": f"Bearer {HF_TOKEN}"
    }

    response = requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=60
    )

    response.raise_for_status()

    for line in response.iter_lines():

        if line:

            yield line.decode(
                "utf-8",
                errors="ignore"
            )


# ------------------------------------------------
# TOKENIZER
# ------------------------------------------------

from tokenizers import Tokenizer

TOKENIZER_PATH = (
    "/kaggle/input/datasets/"
    "mathimohankowshik/tokenizer1/"
    "tokenizer.json"
)

tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

print("\n✅ Tokenizer loaded")


# ================================================================
# TRAINING BLOCK
# ================================================================

def get_training_block():

    tokens = []

    while len(tokens) < SEQ_LEN + 1:

        for filename in TRAIN_FILES:

            print(
                f"\n🌐 Opening {filename}..."
            )

            for line in stream_jsonl(filename):

                try:

                    obj = json.loads(line)

                    text = (
                        obj.get("text")
                        or obj.get("content")
                        or ""
                    )

                    if not text:
                        continue

                    ids = tokenizer.encode(
                        text
                    ).ids

                    tokens.extend(ids)

                    if len(tokens) >= SEQ_LEN + 1:
                        break

                except Exception:
                    continue

            if len(tokens) >= SEQ_LEN + 1:
                break

    tokens = tokens[:SEQ_LEN + 1]

    x = torch.tensor(
        tokens[:-1],
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)

    y = torch.tensor(
        tokens[1:],
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)

    return x, y


# ================================================================
# TRAINING
# ================================================================

TOKENS_PER_STEP = SEQ_LEN

TOTAL_STEPS = math.ceil(
    TARGET_TOKENS / TOKENS_PER_STEP
)

print("\n" + "=" * 70)
print("🚀 TRAINING START")
print("=" * 70)
print(f"Total steps       : {TOTAL_STEPS:,}")
print(f"Starting step     : {latest_step:,}")
print(
    f"Remaining steps   : "
    f"{TOTAL_STEPS - latest_step:,}"
)
print("=" * 70)


model.train()

start_time = time.time()
last_print_time = start_time
tokens_since_print = 0

for step in range(
    latest_step + 1,
    TOTAL_STEPS + 1
):

    x, y = get_training_block()

    optimizer.zero_grad(
        set_to_none=True
    )

    # ------------------------------------------------------------
    # Forward
    # ------------------------------------------------------------

    with autocast(
        device_type="cuda",
        dtype=torch.float16
    ):

        hidden = model(x)

        B, T, H = hidden.shape

        hidden = hidden.reshape(
            B * T,
            H
        )

        targets = y.reshape(
            B * T
        )

        # --------------------------------------------------------
        # Chunked vocabulary projection
        # --------------------------------------------------------

        total_loss = 0.0

        n_tokens = hidden.shape[0]

        for start in range(
            0,
            n_tokens,
            VOCAB_LOSS_CHUNK
        ):

            end = min(
                start + VOCAB_LOSS_CHUNK,
                n_tokens
            )

            h = hidden[start:end]
            t = targets[start:end]

            logits = F.linear(
                h,
                model.token_embedding.weight
            )

            loss_chunk = F.cross_entropy(
                logits.float(),
                t,
                reduction="sum"
            )

            total_loss = total_loss + loss_chunk

        loss = total_loss / n_tokens

    # ------------------------------------------------------------
    # ONE backward
    # ------------------------------------------------------------

    scaler.scale(loss).backward()

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM
    )

    scaler.step(optimizer)
    scaler.update()

    tokens_since_print += TOKENS_PER_STEP

    # ------------------------------------------------------------
    # PRINT
    # ------------------------------------------------------------

    if step % PRINT_EVERY == 0:

        now = time.time()

        elapsed = now - last_print_time

        speed = (
            tokens_since_print / elapsed
            if elapsed > 0
            else 0
        )

        total_elapsed = now - start_time

        remaining_tokens = (
            TOTAL_STEPS - step
        ) * TOKENS_PER_STEP

        eta_hours = (
            remaining_tokens / speed / 3600
            if speed > 0
            else 0
        )

        progress = (
            step / TOTAL_STEPS * 100
        )

        print(
            f"Step {step:,}/{TOTAL_STEPS:,} | "
            f"{progress:.2f}% | "
            f"Tokens {step * TOKENS_PER_STEP:,} | "
            f"Loss {loss.item():.4f} | "
            f"{speed:,.0f} tok/s | "
            f"ETA {eta_hours:.2f} h"
        )

        last_print_time = now
        tokens_since_print = 0

    # ------------------------------------------------------------
    # CHECKPOINT
    # ------------------------------------------------------------

    if step % SAVE_EVERY == 0:

        path = os.path.join(
            CHECKPOINT_DIR,
            f"step_{step:07d}.pt"
        )

        torch.save(
            {
                "step": step,
                "tokens": step * TOKENS_PER_STEP,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "architecture": "A2_Efficient_GQA",
                "params": params,
            },
            path
        )

        print(
            f"💾 Checkpoint saved: {path}"
        )


# ================================================================
# FINAL MODEL
# ================================================================

final_path = os.path.join(
    CHECKPOINT_DIR,
    "a2_final.pt"
)

torch.save(
    {
        "step": TOTAL_STEPS,
        "tokens": TARGET_TOKENS,
        "model": model.state_dict(),
        "architecture": "A2_Efficient_GQA",
        "params": params,
    },
    final_path
)

print("\n" + "=" * 70)
print("🎉 A2 TRAINING COMPLETE")
print("=" * 70)
print(f"Steps  : {TOTAL_STEPS:,}")
print(f"Tokens : {TARGET_TOKENS:,}")
print(f"Model  : {final_path}")
print("=" * 70)

🚀 A2 SPEED-OPTIMIZED TRAINING
Target tokens       : 200,000,000
Sequence length     : 512
Vocab loss chunk    : 256
Learning rate       : 0.0001
TF32                : ENABLED
Activation checkpoint: DISABLED
Parameters: 50,407,648
Parameters (M): 50.41M
✅ EXACT A2 PARAMETER COUNT VERIFIED

🆕 Starting A2 from scratch

✅ Tokenizer loaded

🚀 TRAINING START
Total steps       : 390,625
Starting step     : 0
Remaining steps   : 390,625

🌐 Opening Phase_1_Pretraining/Train/Acts.jsonl...


HTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/Acts.jsonl

In [6]:
# ================================================================
# DOWNLOAD A2 TRAINING FILES LOCALLY — AUTHENTICATED
# ================================================================

from huggingface_hub import hf_hub_download, get_token
import os

HF_TOKEN = get_token()

print("HF token available:", HF_TOKEN is not None)

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/Acts.jsonl",
    "Phase_1_Pretraining/Train/High_Courts.jsonl",
    "Phase_1_Pretraining/Train/Supreme_Court.jsonl",
    "Phase_1_Pretraining/Train/Tribunals.jsonl",
]

LOCAL_FILES = {}

for filename in TRAIN_FILES:

    print(f"\nDownloading: {filename}")

    path = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    LOCAL_FILES[filename] = path

    print("✅", path)

print("\n" + "=" * 60)
print("✅ ALL TRAINING FILES READY")
print("=" * 60)

HF token available: False

Downloading: Phase_1_Pretraining/Train/Acts.jsonl


GatedRepoError: 401 Client Error. (Request ID: Root=1-6ac29224-0954f1074b900e0152d054ec;ae8104b6-8528-4b24-adef-9be1ff67164e)

Cannot access gated repo for url https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/Phase_1_Pretraining/Train/Acts.jsonl.
Access to dataset Exploration-Lab/CS-781-Capstone is restricted. You must have access to it and be authenticated to access it. Please log in.

In [2]:
# ================================================================
# FIX HUGGING FACE AUTHENTICATION
# ================================================================

from huggingface_hub import get_token, hf_hub_url
import requests

HF_TOKEN = get_token()

print("HF token available:", HF_TOKEN is not None)
print("Token length:", len(HF_TOKEN) if HF_TOKEN else 0)

def stream_jsonl(filename):

    url = hf_hub_url(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename=filename,
        repo_type="dataset"
    )

    headers = {
        "Authorization": f"Bearer {HF_TOKEN}",
        "Accept": "application/octet-stream",
    }

    response = requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=120
    )

    print(
        f"HTTP {response.status_code}: {filename}"
    )

    response.raise_for_status()

    for line in response.iter_lines():

        if line:
            yield line.decode(
                "utf-8",
                errors="ignore"
            )

print("✅ Authentication function updated")

HF token available: False
Token length: 0
✅ Authentication function updated


In [7]:
# ================================================================
# FIND EXISTING LOCAL PHASE-1 TRAINING DATA
# ================================================================

import os

roots = [
    "/kaggle/input",
    "/kaggle/working",
    "/root/.cache/huggingface"
]

matches = []

for root in roots:
    if not os.path.exists(root):
        continue

    for dirpath, dirnames, filenames in os.walk(root):
        for filename in filenames:
            if filename.endswith(".jsonl"):
                matches.append(
                    os.path.join(dirpath, filename)
                )

print("=" * 70)
print("LOCAL JSONL FILES FOUND")
print("=" * 70)

if matches:
    for p in matches:
        size_gb = os.path.getsize(p) / (1024**3)
        print(f"{size_gb:8.3f} GB | {p}")
else:
    print("❌ No local JSONL training files found.")

print("=" * 70)

LOCAL JSONL FILES FOUND
❌ No local JSONL training files found.
